# ENEM 2020–2024: distribuição das notas por UF da escola


André Marcelno Watanabe - 14558311  
Camila Aya Saito - 15635649  
Guilherme Vaz de Oliveira Taratá - 10817476  
Vinicius Moraes de Carvalho - 15642432



# Etapa 1 - Análise das notas do ENEM

**Pergunta desta etapa:** como as notas dos participantes com UF da escola informada se distribuem e variam entre estados, regiões, áreas e anos?

A pergunta maior do projeto é a relação entre condições socioeconômicas, condições escolares e desempenho. Aqui preparamos a base e descrevemos os padrões

**Base de dados utilizada nessa fase:** Microdados do ENEM (INEP)

### O que significa uma linha?

No arquivo bruto de resultados, cada linha corresponde a uma inscrição naquela edição, com as notas em colunas. Na amostra reorganizada, cada linha é uma nota de uma área; os sorteios são feitos separadamente por área. Já nas tabelas territoriais, cada linha é UF × ano × área. Uma inscrição pode ter notas ausentes. As escalas das provas diferem, por isso os painéis fixam área e ano, e as amostras por área não são unidas como se contivessem as mesmas pessoas.

### Critérios da análise

- Território: `SG_UF_ESC`, a UF escolar registrada na edição; não se preenche uma ausência usando UF da prova ou residência.
- Seleção: elegibilidade própria de cada área, seguida de 10% de cada ano × UF da escola × área, sem reposição.
- Medida principal: **mediana em todos os recortes**. Média e dispersão continuam disponíveis.
- Shapiro–Wilk: diagnóstico exploratório de normalidade; seu p-valor não escolhe a medida do mapa.
- Participação: contagens anteriores ao sorteio, sem multiplicar a amostra por dez.
- Rankings: ordem descritiva da amostra; diferenças pequenas não são automaticamente significativas.

Não há uma “nota verdadeira do estado” descoberta por um teste. O que estimamos aqui é um centro da distribuição dos participantes incluídos no recorte. A cobertura da UF escolar e a composição dos participantes limitam a generalização.

## 1. Preparar a execução

Execute em ordem a partir da pasta que contém este notebook e `dados/`. No Colab, monte o Drive e ajuste `PASTA_ESTUDO` se necessário. Dependências: pandas, NumPy, PyArrow, SciPy, Plotly e ipywidgets.

Por padrão, a Etapa 1 usa as notas por área do Parquet distribuído. As tabelas de validação são reconstruídas com as contagens completas e as contagens amostrais. Para refazer o sorteio dos CSVs brutos, configure `PASTA_DADOS` e ative `PROCESSAR_MICRODADOS`. A Etapa 2 continua lendo os recortes auditados em `dados/`.

Defina `MOSTRAR_FILTROS = False` para execução com os recortes iniciais em um ambiente sem widgets. As tabelas são gravadas em `resultados_etapa2/`. Passe o mouse sobre as marcas para ver os valores. Clique em **Baixar gráfico (PNG)** para salvar a vista atual após escolher os filtros.


In [ ]:
import gc
import json
import re
from pathlib import Path
from html import escape

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import HTML, IFrame, Markdown, display
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import probplot, shapiro

SEMENTE = 42
FRACAO_AMOSTRA = 0.10
TAMANHO_PARTE = 250_000
ANOS = [2020, 2021, 2022, 2023, 2024]

#ordem usada nos filtros e nos gráficos.
AREAS = [
    "Linguagens",
    "Ciências Humanas",
    "Ciências da Natureza",
    "Matemática",
    "Redação",
]


#o mesmo fundo nos graficos
px.defaults.template = "plotly_white"
pio.templates.default = "plotly_white"
PALETA_SERIES = ["#0056B3", "#C34109", "#007763", "#6D39B5", "#A41F66",
                  "#3D6F20", "#855000", "#006F91", "#4B466C", "#A5222B"]
ESCALA_SEQUENCIAL = [
    [0, "#EDF4FF"], [.25, "#A4C4FA"], [.5, "#5489E7"],
    [.75, "#2454AB"], [1, "#102650"],
]
ESCALA_DIVERGENTE = [
    [0, "#9F290D"], [.24, "#E56B25"], [.5, "#F8F6F3"],
    [.76, "#3F80E0"], [1, "#123B8D"],
]
px.defaults.color_discrete_sequence = PALETA_SERIES
pio.templates["plotly_white"].layout.update(
    font=dict(family="Arial, sans-serif", size=14, color="#243447"),
    title=dict(font=dict(size=20), x=0.02, xanchor="left"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    hoverlabel=dict(font_size=13, bgcolor="#FFFFFF", bordercolor="#B5C4D1",
                    font_color="#243447", namelength=-1),
    separators=",.",
    colorway=PALETA_SERIES,
)
REGIOES = ["Norte","Nordeste","Centro-Oeste","Sudeste", "Sul"]
cores_regioes = dict(
    zip(REGIOES, PALETA_SERIES[:len(REGIOES)])
)


In [ ]:
PASTA_ESTUDO = Path.cwd()
try:
    from google.colab import drive, output
except ImportError:
    pass
else:
    if not (PASTA_ESTUDO / 'dados').is_dir():
        drive.mount('/content/drive')
        PASTA_ESTUDO = Path('/content/drive/MyDrive/ENEM/estudo_enem_revisado')
    output.enable_custom_widget_manager()

PASTA_ENTRADA = PASTA_ESTUDO / 'dados'
PASTA_DADOS = PASTA_ESTUDO
PROCESSAR_MICRODADOS = False
EXPORTAR_TABELAS = False
MOSTRAR_FILTROS = True
print('Pasta do estudo:', PASTA_ESTUDO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta do estudo: /content/drive/MyDrive/ENEM/estudo_enem_revisado


In [ ]:
#relaciono cada ano ao arquivo que contem as notas e a UF da escola
arquivos_por_ano = {
    2020: PASTA_DADOS / "MICRODADOS_ENEM_2020.csv",
    2021: PASTA_DADOS / "MICRODADOS_ENEM_2021.csv",
    2022: PASTA_DADOS / "MICRODADOS_ENEM_2022.csv",
    2023: PASTA_DADOS / "MICRODADOS_ENEM_2023.csv",
    2024: PASTA_DADOS / "RESULTADOS_2024.csv",
}

#arquivos criados e utilizados aqui
ARQUIVO_AMOSTRA = PASTA_DADOS / "enem_amostra_uf_escola_2020_2024.csv"
ARQUIVO_VALIDACAO = PASTA_DADOS / "enem_validacao_amostra_uf_escola_2020_2024.csv"
ARQUIVO_COBERTURA = PASTA_DADOS / "enem_cobertura_uf_escola_2020_2024.csv"
ARQUIVO_MAPA = PASTA_ENTRADA / "malha_ufs_plotly.geojson"

conferencia = []

for ano,caminho in arquivos_por_ano.items():
    conferencia.append(
        {
            "ANO": ano,
            "ARQUIVO": caminho.name,
            "ENCONTRADO": caminho.exists(),
        }
    )

if PROCESSAR_MICRODADOS:
    display(pd.DataFrame(conferencia))
print("Parquet de notas:", (PASTA_ENTRADA / "amostra_notas_contexto.parquet").exists())
print("mapa encontrado:", ARQUIVO_MAPA.exists())


#
if PROCESSAR_MICRODADOS:
    entradas = list(arquivos_por_ano.values())
else:
    entradas = [PASTA_ENTRADA / nome for nome in (
        'amostra_notas_contexto.parquet', 'enem_uf_2020_2024.csv', 'cobertura_uf_escola.csv')]

faltantes = [caminho.name for caminho in entradas if not caminho.exists()]
if faltantes:
    raise FileNotFoundError("coloque na pasta ENEM: " + ", ".join(faltantes))


Parquet de notas: True
mapa encontrado: True


## 2. Selecionar as colunas e organizar as UFs

Usamos nota, presença e situação da redação para selecionar os registros. A UF da escola define o território e o ano vem do arquivo. Região, nome e código do estado são obtidos a partir da UF.

O primeiro dicionário liga cada área às colunas dos microdados. Redação usa a presença de Linguagens, pois as provas acontecem no mesmo dia. Renda, rede escolar e escolaridade familiar ficam para uma próxima extração, depois de conferir a disponibilidade em cada ano.


In [ ]:
#colunas  de nota e presença de cada área
colunas_por_area = {
    "Linguagens": ("NU_NOTA_LC", "TP_PRESENCA_LC"),
    "Ciências Humanas": ("NU_NOTA_CH", "TP_PRESENCA_CH"),
    "Ciências da Natureza": ("NU_NOTA_CN", "TP_PRESENCA_CN"),
    "Matemática": ("NU_NOTA_MT", "TP_PRESENCA_MT"),
    "Redação": ("NU_NOTA_REDACAO", "TP_PRESENCA_LC"),
}

#regiao de cada UF
regiao_por_uf = {
    "AC": "Norte",
    "AP": "Norte",
    "AM": "Norte",
    "PA": "Norte",
    "RO": "Norte",
    "RR": "Norte",
    "TO": "Norte",
    "AL": "Nordeste",
    "BA": "Nordeste",
    "CE": "Nordeste",
    "MA": "Nordeste",
    "PB": "Nordeste",
    "PE": "Nordeste",
    "PI": "Nordeste",
    "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "ES": "Sudeste",
    "MG": "Sudeste",
    "RJ": "Sudeste",
    "SP": "Sudeste",
    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul",
}

#codigo do IBGE usado no mapa
codigo_por_uf = {
    "RO": "11",
    "AC": "12",
    "AM": "13",
    "RR": "14",
    "PA": "15",
    "AP": "16",
    "TO": "17",
    "MA": "21",
    "PI": "22",
    "CE": "23",
    "RN": "24",
    "PB": "25",
    "PE": "26",
    "AL": "27",
    "SE": "28",
    "BA": "29",
    "MG": "31",
    "ES": "32",
    "RJ": "33",
    "SP": "35",
    "PR": "41",
    "SC": "42",
    "RS": "43",
    "MS": "50",
    "MT": "51",
    "GO": "52",
    "DF": "53",
}

#nome mostrados nos graficos
nome_por_uf = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins",
}

ufs_validas = set(regiao_por_uf)


## 3. Preparar a leitura dos microdados

Lemos apenas as colunas necessárias e mantemos seus nomes oficiais para facilitar a consulta ao dicionário do INEP. As siglas são texto, os códigos são inteiros e as notas são números decimais.

A leitura é feita em blocos, mas reunimos os blocos de um ano antes do sorteio. Por isso, ainda é preciso memória para um ano dessas colunas. Os cinco anos não ficam carregados juntos nesta etapa.


In [ ]:
#colunas para criar a nova amostra
colunas_necessarias = [
    "SG_UF_ESC",
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT",
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "TP_STATUS_REDACAO",
    "NU_NOTA_REDACAO",
]

#tipos menores diminuem o uso de memória durante a leitura
tipos_microdados = {
    "SG_UF_ESC": "string",
    "TP_PRESENCA_CN": "Int8",
    "TP_PRESENCA_CH": "Int8",
    "TP_PRESENCA_LC": "Int8",
    "TP_PRESENCA_MT": "Int8",
    "TP_STATUS_REDACAO": "Int8",
    "NU_NOTA_CN": "Float64",
    "NU_NOTA_CH": "Float64",
    "NU_NOTA_LC": "Float64",
    "NU_NOTA_MT": "Float64",
    "NU_NOTA_REDACAO": "Float64",
}


## 4. Selecionar os registros e sortear a amostra

A leitura usa blocos de 250 mil linhas, depois reúne as colunas selecionadas de um ano na memória. Esta célula só processa os microdados se `PROCESSAR_MICRODADOS = True`. Para cada ano e área:

1. Mantemos presença igual a 1 e nota preenchida. Em Redação, exigimos também `TP_STATUS_REDACAO = 1`; outras situações ficam de fora, mesmo quando têm nota.
2. Contamos os elegíveis com e sem UF da escola reconhecida.
3. Entre os que têm UF reconhecida, sorteamos 10% de cada UF, sem reposição. O tamanho é arredondado para um número inteiro: 2.000 elegíveis geram 200 notas, por exemplo. O sorteio é feito uma única vez por grupo.
4. Salvamos a amostra e as contagens usadas na validação e nos gráficos de participação e cobertura.

Notas iguais são mantidas, pois podem pertencer a pessoas diferentes. A amostra não guarda identificador individual e não permite uma futura junção pessoa a pessoa. Os CSVs anteriores só são substituídos depois que todos os anos terminam de ser processados.


In [ ]:
if PROCESSAR_MICRODADOS:
    #confere as colunas de todos os arquivos antes de começar
    for caminho in arquivos_por_ano.values():
        cabecalho = pd.read_csv(caminho, sep=";", encoding="latin1", nrows=0)
        ausentes = set(colunas_necessarias) - set(cabecalho.columns)
        if ausentes:
            raise ValueError(f"Colunas ausentes em {caminho.name}: {sorted(ausentes)}")

    #grava em arquivo temporário ate terminar todos os anos
    amostra_em_preparo = ARQUIVO_AMOSTRA.with_suffix(".em_preparo.csv")

    escrever_cabecalho = True
    linhas_validacao = []
    linhas_cobertura = []

    #le um ano por vez para poupar memoria
    for ano in ANOS:
        caminho = arquivos_por_ano[ano]

        if not caminho.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho.name}")

        print(f"Lendo {ano}: {caminho.name}")
        partes = []

        #le apenas as colunas necessárias, em blocos
        leitor = pd.read_csv(
            caminho,
            sep=";",
            encoding="latin1",
            usecols=colunas_necessarias,
            dtype=tipos_microdados,
            chunksize=TAMANHO_PARTE,
            low_memory=False,
        )

        for numero_parte, parte in enumerate(leitor, start=1):
            #remove espaços e deixa as siglas em maiúsculas
            parte["SG_UF_ESC"] = parte["SG_UF_ESC"].str.strip().str.upper()
            partes.append(parte)

            if numero_parte % 5 == 0:
                print(f"  {numero_parte * TAMANHO_PARTE:,} participantes lidos")

        dados_ano = pd.concat(partes, ignore_index=True)
        del partes, parte
        gc.collect()

        #so aceita siglas presentes na lista de UFs
        uf_escola_valida = dados_ano["SG_UF_ESC"].isin(ufs_validas)

        #a selecao e o sorteio são feitos separadamente para cada área
        for area, (coluna_nota, coluna_presenca) in colunas_por_area.items():
            nota_valida = (
                dados_ano[coluna_presenca].eq(1) & dados_ano[coluna_nota].notna()
            )

            #em Redação, mantemos apenas a situação de correção 1
            if area == "Redação":
                nota_valida &= dados_ano["TP_STATUS_REDACAO"].eq(1)

            total_notas_validas = int(nota_valida.sum())
            total_com_uf_escola = int((nota_valida & uf_escola_valida).sum())

            #conta também quem não informou a UF da escola, antes do sorteio
            linhas_cobertura.append(
                {
                    "ANO": ano,
                    "AREA": area,
                    "NOTAS_VALIDAS": total_notas_validas,
                    "NOTAS_COM_UF_ESCOLA": total_com_uf_escola,
                    "NOTAS_SEM_UF_ESCOLA": total_notas_validas - total_com_uf_escola,
                    "COBERTURA_UF_ESCOLA": total_com_uf_escola
                    / total_notas_validas
                    * 100,
                }
            )

            elegiveis = dados_ano.loc[
                nota_valida & uf_escola_valida,
                ["SG_UF_ESC", coluna_nota],
            ].copy()

            contagem_elegiveis = elegiveis["SG_UF_ESC"].value_counts()

            #sorteia 10% separadamente dentro de cada UF da escola
            amostra_area = (
                elegiveis.groupby("SG_UF_ESC", group_keys=False)
                .sample(frac=FRACAO_AMOSTRA, replace=False, random_state=SEMENTE)
                .reset_index(drop=True)
            )

            contagem_amostra = amostra_area["SG_UF_ESC"].value_counts()

            #cada linha da amostra guarda uma nota de uma área
            amostra_area = amostra_area.rename(
                columns={
                    "SG_UF_ESC": "UF_ESCOLA",
                    coluna_nota: "NOTA",
                }
            )
            amostra_area.insert(0, "ANO", ano)
            amostra_area["NOME_UF"] = amostra_area["UF_ESCOLA"].map(nome_por_uf)
            amostra_area["COD_UF_IBGE"] = amostra_area["UF_ESCOLA"].map(codigo_por_uf)
            amostra_area["REGIAO"] = amostra_area["UF_ESCOLA"].map(regiao_por_uf)
            amostra_area["AREA"] = area
            amostra_area["TOTAL_ELEGIVEIS_ESTRATO"] = amostra_area["UF_ESCOLA"].map(
                contagem_elegiveis
            )
            amostra_area["TAMANHO_AMOSTRA_ESTRATO"] = amostra_area["UF_ESCOLA"].map(
                contagem_amostra
            )
            amostra_area["PERCENTUAL_AMOSTRADO"] = (
                amostra_area["TAMANHO_AMOSTRA_ESTRATO"]
                / amostra_area["TOTAL_ELEGIVEIS_ESTRATO"]
                * 100
            )

            #acrescenta a amostra da área ao CSV e libera a memória em seguida
            amostra_area.to_csv(
                amostra_em_preparo,
                sep=";",
                encoding="utf-8-sig" if escrever_cabecalho else "utf-8",
                index=False,
                mode="w" if escrever_cabecalho else "a",
                header=escrever_cabecalho,
            )
            escrever_cabecalho = False

            #guarda as contagens de cada grupo de ano, UF e área.
            for uf, total in contagem_elegiveis.items():
                quantidade = int(contagem_amostra.get(uf, 0))
                linhas_validacao.append(
                    {
                        "ANO": ano,
                        "UF_ESCOLA": uf,
                        "REGIAO": regiao_por_uf[uf],
                        "AREA": area,
                        "REGISTROS_ELEGIVEIS": int(total),
                        "AMOSTRA": quantidade,
                        "PERCENTUAL": quantidade / int(total) * 100,
                    }
                )

            print(f"  {area}: {len(amostra_area):,} notas sorteadas")

            del elegiveis, amostra_area
            gc.collect()

        del dados_ano
        gc.collect()

    #salva as contagens em arquivos temporários
    pd.DataFrame(linhas_validacao).to_csv(
        ARQUIVO_VALIDACAO.with_suffix(".em_preparo.csv"),
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    pd.DataFrame(linhas_cobertura).to_csv(
        ARQUIVO_COBERTURA.with_suffix(".em_preparo.csv"),
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )

    #substitui os CSVs anteriores depois de concluir o processamento
    amostra_em_preparo.replace(ARQUIVO_AMOSTRA)
    ARQUIVO_VALIDACAO.with_suffix(".em_preparo.csv").replace(ARQUIVO_VALIDACAO)
    ARQUIVO_COBERTURA.with_suffix(".em_preparo.csv").replace(ARQUIVO_COBERTURA)
    print("Nova amostra concluída.")
else:
    print("A amostra pronta será carregada. Nenhum microdado será reprocessado.")

A amostra pronta será carregada. Nenhum microdado será reprocessado.


## 5. Carregar a amostra e conferir seu tamanho

Carregamos as notas sorteadas e a cobertura. No modo padrão, reconstruímos a validação com as contagens dos resumos completos. A diferença entre o tamanho sorteado e 10% dos elegíveis deve ser de até meia linha, por causa do arredondamento. Essa conferência verifica o tamanho; não prova a aleatoriedade nem elimina o viés de participação no ENEM.

Lemos as notas em precisão dupla e arredondamos para a casa decimal original, removendo resíduos de arquivos antigos. As medidas calculadas continuam com sua precisão: a mediana entre 535,2 e 535,3, por exemplo, é 535,25.


In [ ]:
#category e int ajudam a economizar memoria
tipos_amostra = {
    "ANO": "int16",
    "UF_ESCOLA": "category",
    "NOME_UF": "category",
    "COD_UF_IBGE": "string",
    "REGIAO": "category",
    "AREA": "category",
    "NOTA": "float64",
    "TOTAL_ELEGIVEIS_ESTRATO": "int32",
    "TAMANHO_AMOSTRA_ESTRATO": "int32",
    "PERCENTUAL_AMOSTRADO": "float32",
}

if PROCESSAR_MICRODADOS:
    dados_analise = pd.read_csv(
        ARQUIVO_AMOSTRA,
        sep=";",
        encoding="utf-8-sig",
        dtype=tipos_amostra,
    )

    validacao_amostra = pd.read_csv(
        ARQUIVO_VALIDACAO,
        sep=";",
        encoding="utf-8-sig",
    )

    cobertura_uf_escola = pd.read_csv(
        ARQUIVO_COBERTURA,
        sep=";",
        encoding="utf-8-sig",
    )

else:
    chaves_amostra = ['ANO', 'UF_ESCOLA', 'AREA']
    dados_analise = pd.read_parquet(PASTA_ENTRADA / 'amostra_notas_contexto.parquet',
                                   columns=chaves_amostra + ['NOTA', 'REGIAO'])
    contagens = (dados_analise.groupby(chaves_amostra, observed=True).size()
                 .rename('AMOSTRA').reset_index())
    completos = pd.read_csv(PASTA_ENTRADA / 'enem_uf_2020_2024.csv', sep=';')
    validacao_amostra = (completos[chaves_amostra + ['REGIAO', 'N']]
                         .rename(columns={'N': 'REGISTROS_ELEGIVEIS'})
                         .merge(contagens, on=chaves_amostra, validate='one_to_one', how='left'))
    validacao_amostra['PERCENTUAL'] = 100 * validacao_amostra.AMOSTRA / validacao_amostra.REGISTROS_ELEGIVEIS
    contexto_estrato = validacao_amostra.drop(columns='REGIAO').rename(columns={
        'AMOSTRA': 'TAMANHO_AMOSTRA_ESTRATO', 'REGISTROS_ELEGIVEIS': 'TOTAL_ELEGIVEIS_ESTRATO',
        'PERCENTUAL': 'PERCENTUAL_AMOSTRADO'})
    dados_analise = dados_analise.merge(contexto_estrato, on=chaves_amostra, validate='many_to_one')
    dados_analise['NOME_UF'] = dados_analise.UF_ESCOLA.map(nome_por_uf)
    dados_analise['COD_UF_IBGE'] = dados_analise.UF_ESCOLA.map(codigo_por_uf)
    dados_analise = dados_analise.astype(tipos_amostra)
    cobertura_uf_escola = pd.read_csv(PASTA_ENTRADA / 'cobertura_uf_escola.csv', sep=';')

#aceita a diferença de arredondamento no tamanho sorteado
validacao_amostra["DIFERENCA_DO_ESPERADO"] = (
    validacao_amostra["AMOSTRA"]
    - validacao_amostra["REGISTROS_ELEGIVEIS"] * FRACAO_AMOSTRA
).abs()

amostra_correta = validacao_amostra["DIFERENCA_DO_ESPERADO"].le(0.51).all()

print("Linhas da nova amostra:", f"{len(dados_analise):,}")
print("Estratos de ano × UF da escola × área:", len(validacao_amostra))
print("Amostragem de 10% correta, considerando o arredondamento:", amostra_correta)


# Remove resíduos decimais de CSVs antigos, gravados com float32.
dados_analise["NOTA"] = dados_analise["NOTA"].round(1)


Linhas da nova amostra: 1,910,983
Estratos de ano × UF da escola × área: 675
Amostragem de 10% correta, considerando o arredondamento: True


## 6. Conferir se as três tabelas correspondem

Antes dos gráficos, verificamos os anos, áreas, UFs, regiões, notas e valores ausentes. Também conferimos se:

- a contagem da amostra bate com a validação em cada ano, UF e área;
- a soma dos elegíveis por UF bate com a cobertura;
- os elegíveis com UF e sem UF somam o total elegível.

Se uma conferência falhar, a execução para, evitando misturar CSVs de versões diferentes. Não removemos linhas por terem a mesma nota: sem identificador, não sabemos se são registros duplicados ou pessoas diferentes. As contagens por área pressupõem uma linha por participante no arquivo de origem.


In [ ]:
chaves = ["ANO", "UF_ESCOLA", "AREA"]
assert not dados_analise.empty, "A amostra está vazia."
assert (
    dados_analise[chaves + ["REGIAO", "NOTA"]].notna().all().all()
), "Há ausentes na seleção."
assert dados_analise["UF_ESCOLA"].isin(ufs_validas).all(), "Há UF desconhecida."
assert dados_analise["ANO"].isin(ANOS).all(), "Há ano fora do período. "
assert dados_analise["AREA"].isin(AREAS).all(), "Há área desconhecida."
assert np.isfinite(dados_analise["NOTA"]).all(), "Há nota com valor não finito."
assert dados_analise["NOTA"].ge(0).all(), "Há nota negativa."
assert not validacao_amostra.duplicated(chaves).any(), "Estrato repetido na validação."
assert not cobertura_uf_escola.duplicated(["ANO", "AREA"]).any()
assert amostra_correta, "O tamanho não corresponde aos 10% definidos."
regioes_esperadas = dados_analise["UF_ESCOLA"].astype(str).map(regiao_por_uf)
assert (regioes_esperadas == dados_analise["REGIAO"].astype(str)).all()

#confere se o número de notas lidas bate com a tabela de validação
contagem_lida = (
    dados_analise.groupby(chaves, observed=True).size().rename("LIDAS").reset_index()
)
conferencia = validacao_amostra.merge(
    contagem_lida, on=chaves, how="outer", validate="one_to_one"
)
assert conferencia[["AMOSTRA", "LIDAS"]].notna().all().all()
assert (
    conferencia["AMOSTRA"].eq(conferencia["LIDAS"]).all()
), "Amostra e validação não correspondem."
assert validacao_amostra["REGISTROS_ELEGIVEIS"].ge(validacao_amostra["AMOSTRA"]).all()

#confere se a soma das UFs bate com a cobertura de cada ano e área
totais = (
    validacao_amostra.groupby(["ANO", "AREA"])["REGISTROS_ELEGIVEIS"]
    .sum()
    .reset_index()
)
totais = totais.merge(
    cobertura_uf_escola, on=["ANO", "AREA"], how="outer", validate="one_to_one"
)
assert totais.notna().all().all()
assert totais["REGISTROS_ELEGIVEIS"].eq(totais["NOTAS_COM_UF_ESCOLA"]).all()
total_conferido = totais["NOTAS_COM_UF_ESCOLA"] + totais["NOTAS_SEM_UF_ESCOLA"]
assert totais["NOTAS_VALIDAS"].eq(total_conferido).all()

cobertura_calculada = 100 * totais["NOTAS_COM_UF_ESCOLA"] / totais["NOTAS_VALIDAS"]
assert np.allclose(totais["COBERTURA_UF_ESCOLA"], cobertura_calculada)

print(
    "aprovada:", len(dados_analise), "notas;", len(conferencia), "estratos."
)
print(
    "A amostra descreve elegíveis com UF da escola informada; não todos os inscritos."
)


aprovada: 1910983 notas; 675 estratos.
A amostra descreve elegíveis com UF da escola informada; não todos os inscritos.


### Como os painéis são exibidos

As três funções abaixo cuidam apenas da apresentação: exibem o gráfico, formatam as tabelas e organizam os filtros. Os cálculos continuam nas células de cada análise. Cada gráfico usa uma área própria de HTML para evitar conflitos ao atualizar as saídas no Colab.


In [ ]:
def fixar_percentual(grafico, eixo='x', fracao=False):
    """Mantém proporções em 0–100%, inclusive após mudar filtros ou ampliar outro eixo."""
    limite = 1 if fracao else 100
    atualizar_eixo = grafico.update_xaxes if eixo == 'x' else grafico.update_yaxes
    atualizar_eixo(range=[0, limite], autorange=False, fixedrange=True,
                   tickmode='linear', tick0=0, dtick=limite / 5,
                   tickformat='.0%' if fracao else '.0f',
                   ticksuffix='' if fracao else '%')

    # Junto ao limite de 100%, ponha o rótulo dentro da barra.
    for barra in grafico.select_traces(selector=dict(type='bar')):
        eixo_valor = 'x' if barra.orientation == 'h' else 'y'
        if eixo_valor != eixo or (barra.text is None and barra.texttemplate is None):
            continue
        valores = barra.x if eixo == 'x' else barra.y
        barra.update(
            textposition=['inside' if valor is not None and valor >= .93 * limite else 'outside'
                          for valor in valores],
            insidetextfont=dict(color='#FFFFFF'),
            outsidetextfont=dict(color='#243447'),
            cliponaxis=False,
        )


def mostrar_figura(grafico):
    """Renderiza o gráfico interativo com download PNG da vista em exibição."""
    altura = grafico.layout.height or 500
    grafico.update_layout(autosize=True, width=None, height=altura, separators=",.")
    grafico.update_xaxes(automargin=True)
    grafico.update_yaxes(automargin=True)

    titulo = re.sub(r"<[^>]+>", " ", grafico.layout.title.text or "")
    nome = re.sub(r"[^\w-]+", "_", titulo).strip("_")[:120] or "grafico_ENEM"
    opcoes_png = dict(format="png", scale=2, filename=nome)
    acao = """
      const grafico = document.getElementById('{plot_id}');
      const botao = document.getElementById('baixar-png');
      const aviso = document.getElementById('estado-download');
      botao.disabled = false;
      botao.addEventListener('click', async () => {
        botao.disabled = true;
        aviso.textContent = 'Preparando PNG…';
        try {
          await Plotly.downloadImage(grafico, {
            ...OPCOES_PNG,
            width: grafico.clientWidth,
            height: grafico.clientHeight
          });
          aviso.textContent = 'PNG preparado.';
        } catch (erro) {
          console.error(erro);
          aviso.textContent = 'Não foi possível gerar o PNG. Tente novamente.';
        } finally {
          botao.disabled = false;
        }
      });
    """.replace("OPCOES_PNG", json.dumps(opcoes_png))
    pagina = grafico.to_html(
        include_plotlyjs="cdn",
        full_html=True,
        post_script=acao,
        config={
            "responsive": True,
            "displaylogo": False,
            "scrollZoom": False,
            "modeBarButtonsToRemove": ["select2d", "lasso2d"],
            "toImageButtonOptions": opcoes_png,
        },
    )
    estilo = """
    <meta charset="utf-8">
    <style>
      body {margin:0; font-family:Arial,sans-serif;}
      .barra-download {height:46px; box-sizing:border-box; padding:7px 14px;
        display:flex; align-items:center; justify-content:flex-end; gap:10px;
        background:#F3F7FA; border-bottom:1px solid #D9E3ED;}
      .barra-download button {background:#174B62; color:white; border:0;
        border-radius:5px; padding:7px 12px; font-size:13px; font-weight:600; cursor:pointer;}
      .barra-download button:focus-visible {outline:2px solid #1A66B7; outline-offset:2px;}
      .barra-download button:disabled {opacity:.62; cursor:wait;}
      #estado-download {font-size:12px; color:#334155;}
    </style>
    """
    barra = """
    <div class="barra-download">
      <span id="estado-download" role="status" aria-live="polite"></span>
      <button id="baixar-png" type="button" disabled>Baixar gráfico (PNG)</button>
    </div>
    """
    pagina = pagina.replace("<head>", "<head>" + estilo, 1).replace("<body>", "<body>" + barra, 1)
    quadro = IFrame(
        src="about:blank",
        width="100%",
        height=altura + 50,
        extras=[
            f'srcdoc="{escape(pagina)}"',
            'title="Gráfico interativo"',
            'style="min-width:540px;border:0;display:block;background:white;border-radius:8px;"',
        ],
    )
    display(HTML('<div style="width:100%;overflow-x:auto;">' + quadro._repr_html_() + '</div>'))


# Nome usado pelos painéis da Etapa 1.
exibir_grafico = mostrar_figura


def exibir_tabela(tabela):
    """Formata os detalhes sem ocupar uma saída inteira com linhas e colunas."""
    nomes = {
        "LOCALIZACAO": "Localização",
        "REDE": "Rede",
        "RECURSO": "Recurso",
        "PERCENTUAL": "Percentual (%)",
        "ESCOLAS": "Escolas no recorte",
        "INFORMADAS": "Escolas com resposta",
        "COM_RECURSO": "Escolas com recurso",
        "N_URBANA": "Notas urbanas",
        "N_RURAL": "Notas rurais",
        "MEDIANA_URBANA": "Mediana urbana",
        "MEDIANA_RURAL": "Mediana rural",
        "DIFERENCA": "Rural − urbana (pontos)",
        "PCT_RURAL": "Elegíveis em escolas rurais (%)",
        "PCT_PRIVADA": "Elegíveis na rede privada (%)",
        "NOTA_ALVO": "Mediana",
        "N_AMOSTRA": "Notas na amostra",
        "ANO": "Ano",
        "AREA": "Área",
        "UF_ESCOLA": "UF da escola",
        "NOME_UF": "Estado",
        "REGIAO": "Região",
        "N": "Notas na amostra",
        "REGISTROS_ELEGIVEIS": "Elegíveis",
        "AMOSTRA": "Amostrados",
        "MEDIA": "Média",
        "MEDIANA": "Mediana",
        "DESVIO_PADRAO": "Desvio-padrão",
        "ASSIMETRIA": "Assimetria",
        "MINIMO": "Mínimo",
        "MAXIMO": "Máximo",
    }
    formatada = (
        tabela.rename(columns=nomes)
        .style.hide(axis="index")
        .format(precision=2, decimal=",", thousands=".")
        .set_table_styles(
            [
                {
                    "selector": "",
                    "props": "border-collapse:collapse;font:14px Arial;color:#243447;background:white;width:100%;",
                },
                {
                    "selector": "th",
                    "props": "padding:12px;text-align:left;background:#edf3f7;white-space:nowrap;",
                },
                {
                    "selector": "td",
                    "props": "padding:10px 12px;border-bottom:1px solid #e2e8f0;white-space:nowrap;",
                },
            ]
        )
    )
    if "p-valor" in tabela.columns:
        formatada = formatada.format(
            {"p-valor": "{:.3g}", "W de Shapiro": "{:.4f}"},
            subset=["p-valor", "W de Shapiro"],
            decimal=",",
        )
    display(
        HTML(
            '<div style="overflow:auto;max-height:360px;margin:12px 0;">'
            + formatada.to_html()
            + "</div>"
        )
    )


def montar_painel(funcao, **controles):
    """Organiza os filtros e mantém apenas a saída do recorte atual."""
    if not MOSTRAR_FILTROS:
        funcao(**{nome: controle.value for nome, controle in controles.items()})
        return
    for controle in controles.values():
        controle.layout = widgets.Layout(width="310px", max_width="100%")
        controle.style.description_width = "initial"
    filtros = widgets.Box(
        list(controles.values()),
        layout=widgets.Layout(
            display="flex", flex_flow="row wrap", gap="8px", width="100%"
        ),
    )
    saida = widgets.interactive_output(funcao, controles)
    saida.layout = widgets.Layout(width="100%", height="auto")
    display(widgets.VBox([filtros, saida], layout=widgets.Layout(width="100%")))

## 7. Participantes elegíveis e cobertura da UF escolar

Escolha uma área e alterne **Mostrar** entre as contagens de participantes e a porcentagem com UF da escola informada. Exibimos um gráfico por vez. Marque **Ver tabela do ano** para consultar o ano selecionado.

As contagens vêm de antes do sorteio. Cada participante é contado uma vez na área e no ano. São totais de elegíveis, e a cobertura é a parcela deles com UF conhecida — não é uma taxa de comparecimento. Mudanças na cobertura podem alterar o perfil do grupo comparado.

Nos painéis, cada `def` reúne os passos de um gráfico. `montar_painel` organiza os filtros e atualiza a saída quando uma escolha muda.


In [ ]:
# @title
def mostrar_cobertura(area, ano, visualizacao="Participantes", ver_tabela=False):
    tabela = cobertura_uf_escola[cobertura_uf_escola["AREA"] == area].sort_values("ANO")
    if visualizacao == "Participantes":
        barras = tabela.melt(
            id_vars="ANO",
            value_vars=["NOTAS_COM_UF_ESCOLA", "NOTAS_SEM_UF_ESCOLA"],
            var_name="Situação",
            value_name="Participantes elegíveis",
        )
        barras["Situação"] = barras["Situação"].replace(
            {
                "NOTAS_COM_UF_ESCOLA": "Com UF escolar",
                "NOTAS_SEM_UF_ESCOLA": "Sem UF escolar",
            }
        )
        grafico = px.bar(
            barras,
            x="ANO",
            y="Participantes elegíveis",
            color="Situação",
            barmode="group",
            title=f"Quantas pessoas fizeram a prova?<br><sup>{area}</sup>",
            labels={"ANO": "Ano do ENEM", "Participantes elegíveis": "Participantes com nota válida"},
            color_discrete_map={
                "Com UF escolar": "#0056B3",
                "Sem UF escolar": "#C34109",
            },
        )
        grafico.update_layout(
            height=480,
            legend=dict(orientation="h", y=-0.2, title_text=""),
            margin=dict(t=90, b=110, l=80, r=35),
        )
        grafico.update_xaxes(dtick=1)
        exibir_grafico(grafico)
    else:
        linha = px.line(
            tabela,
            x="ANO",
            y="COBERTURA_UF_ESCOLA",
            markers=True,
            hover_data={"NOTAS_COM_UF_ESCOLA": ":,", "NOTAS_VALIDAS": ":,"},
            title=f"Em quantas notas sabemos o estado da escola?<br><sup>{area} · entre as notas válidas</sup>",
            labels={
                "ANO": "Ano do ENEM",
                "COBERTURA_UF_ESCOLA": "Notas válidas com estado da escola (%)",
            },
        )
        fixar_percentual(linha, eixo="y")
        linha.update_xaxes(dtick=1)
        linha.update_layout(height=460, margin=dict(t=85, b=65, l=75, r=35))
        exibir_grafico(linha)
    detalhe = tabela[tabela["ANO"] == ano].rename(
        columns={
            "NOTAS_VALIDAS": "Elegíveis totais",
            "NOTAS_COM_UF_ESCOLA": "Com UF escolar",
            "NOTAS_SEM_UF_ESCOLA": "Sem UF escolar",
            "COBERTURA_UF_ESCOLA": "Cobertura (%)",
        }
    )
    if ver_tabela:
        exibir_tabela(detalhe.round(2))


montar_painel(
    mostrar_cobertura,
    visualizacao=widgets.Dropdown(
        options=["Participantes", "Cobertura (%)"], description="Mostrar:"
    ),
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela do ano"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Detalhe ano:"),
)


## 8. Onde se concentram os participantes de cada área?

Escolha ano, área, território e região. O gráfico soma os elegíveis ou os amostrados e ordena as barras. Alterne entre contagem e proporção. O percentual mostra a participação de cada barra no total exibido.

O total de elegíveis vem da contagem completa com UF da escola, sem multiplicar a amostra por dez. Mais participantes não significam maior taxa de participação: para calcular essa taxa, precisaríamos de uma população de referência compatível.


In [ ]:
# @title
def mostrar_participantes(ano, area, territorio, regiao, contagem, ver_tabela=False, unidade="Contagem"):
    selecao = (validacao_amostra["ANO"] == ano) & (validacao_amostra["AREA"] == area)
    tabela = validacao_amostra[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    coluna = "REGIAO" if territorio == "Região" else "UF_ESCOLA"
    medida = (
        "REGISTROS_ELEGIVEIS" if contagem == "Elegíveis antes do sorteio" else "AMOSTRA"
    )
    grupos = tabela.groupby(coluna, observed=True)
    tabela = grupos[["REGISTROS_ELEGIVEIS", "AMOSTRA"]].sum().reset_index()
    tabela["Participação no recorte (%)"] = 100 * tabela[medida] / tabela[medida].sum()
    coluna_valor = medida if unidade == "Contagem" else "Participação no recorte (%)"
    tabela = tabela.sort_values(coluna_valor)
    grupo = "Região da escola" if coluna == "REGIAO" else "Estado da escola"
    tipo = "Notas válidas" if medida == "REGISTROS_ELEGIVEIS" else "Notas da amostra"
    grafico = px.bar(
        tabela,
        x=coluna_valor,
        y=coluna,
        orientation="h",
        text=coluna_valor,
        title=f"{tipo} por {grupo.lower()}<br><sup>{area} · {ano} · {regiao}</sup>",
        hover_data={"Participação no recorte (%)": ":.2f", medida: ":,"},
        labels={medida: "Participantes na área", coluna: territorio},
    )
    grafico.update_traces(
        texttemplate="%{text:,.0f}" if unidade == "Contagem" else "%{text:.1f}%",
        textposition="outside", cliponaxis=False
    )
    grafico.update_layout(
        height=max(420, 27 * len(tabela) + 130), margin=dict(l=115, r=95, t=100, b=65)
    )
    grafico.update_yaxes(title=grupo)
    grafico.update_xaxes(title="Número de notas" if unidade == "Contagem" else "Parcela do total exibido (%)")
    if unidade == "Contagem":
        grafico.update_xaxes(range=[0, max(1, tabela[coluna_valor].max() * 1.18)])
    else:
        fixar_percentual(grafico)
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(tabela.sort_values(medida, ascending=False).round(2))


montar_painel(
    mostrar_participantes,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    territorio=widgets.Dropdown(
        options=["Região", "UF da escola"], description="Território:"
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    unidade=widgets.Dropdown(options=["Contagem", "Proporção (%)"], value="Proporção (%)", description="Unidade:"),
    contagem=widgets.Dropdown(
        options=["Elegíveis antes do sorteio", "Amostrados"],
        description="Contagem:",
        layout=widgets.Layout(width="370px"),
    ),
)


## 9. Qual medida representa o desempenho típico?

Definimos previamente o que queremos resumir: **o centro da distribuição das notas no recorte**. Usamos a mediana, ou percentil 50, de forma consistente em todos os estados, anos e áreas. Ela depende da ordenação das notas e é menos sensível a valores muito altos ou muito baixos. Seu uso não exige normalidade.

Exemplo didático: em `[400, 450, 500, 550, 900]`, a média é 560 e a mediana é 500. A nota 900 desloca a média, enquanto a mediana preserva a posição central. Isso não torna a média errada: ela responde à pergunta sobre a soma das notas dividida pela quantidade. Escolhemos a mediana por nosso objetivo de descrever o centro com menor sensibilidade às caudas.

Nenhuma medida resume sozinha toda a distribuição. Se houver dois grupos muito diferentes, até a mediana pode esconder essa divisão. Por isso, sempre a acompanhamos de histograma, quartis, percentis e N. Maior N não corrige automaticamente viés na seleção dos participantes.

### Por que não usar Shapiro para escolher entre média e mediana?

O Shapiro–Wilk testa a hipótese de uma distribuição normal. Ele **não compara a qualidade da média e da mediana**. Um p-valor pequeno não torna a média inválida; um p-valor grande não prova normalidade. Trocar a medida de estado para estado com base no teste ainda mistura alvos diferentes no mesmo mapa.

O diagnóstico fica separado adiante. Não adotamos qui-quadrado para escolher a nota: categorizar notas em faixas e testar frequências não resolve a escolha da medida central.

Referências: [medidas de localização — NIST](https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm) e [Shapiro–Wilk — SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html).

### Como ler as medidas

| Medida | Significado e uso |
|---|---|
| N | Quantidade de notas da amostra no recorte; não é o total de inscritos |
| Média | Soma das notas dividida por N |
| Mediana / P50 | Centro da ordenação; com N par, média dos dois valores centrais |
| Q1 / P25 e Q3 / P75 | Limites aproximados dos 50% centrais |
| P5 e P95 | Delimitam aproximadamente os 90% centrais; não são intervalo de confiança |
| Desvio-padrão | Dispersão em torno da média, na unidade de pontos |
| Q3 − Q1 | Dispersão dos 50% centrais; amplitude interquartil |
| Assimetria | Sinal positivo sugere cauda à direita; negativo, à esquerda; zero não prova normalidade |
| Mínimo e máximo | Extremos observados; não significam erro de registro por si só |

Empates, comuns em Redação, podem fazer mais da metade das notas ficar exatamente na mediana. Percentis descrevem a distribuição; eles não medem a incerteza da estimativa.

## 10. Calcular as medidas de cada UF da escola

Criamos uma linha para cada ano, UF da escola e área. `groupby` separa os grupos, `agg` calcula as medidas e `quantile` encontra os percentis. Depois, juntamos os resultados e as contagens pela chave do grupo.

Todas as notas da amostra entram nos cálculos, inclusive as caudas. `NOTA_REPRESENTATIVA` recebe sempre a mediana; o teste de normalidade não altera essa escolha.


In [ ]:
# @title
#agrupa as notas por ano, UF da escola e área
colunas_grupo = ["ANO", "UF_ESCOLA", "NOME_UF", "COD_UF_IBGE", "REGIAO", "AREA"]
grupos_uf = dados_analise.groupby(colunas_grupo, observed=True)["NOTA"]

#medidas de posição, dispersão e forma
resumo_uf = grupos_uf.agg(
    MEDIA="mean",
    MEDIANA="median",
    N="count",
    DESVIO_PADRAO="std",
    ASSIMETRIA="skew",
    MINIMO="min",
    MAXIMO="max",
).reset_index()

#percentis usados para observar o centro e as caudas
percentis_uf = (
    grupos_uf.quantile([0.05, 0.25, 0.75, 0.95])
    .unstack()
    .rename(columns={0.05: "P5", 0.25: "Q1", 0.75: "Q3", 0.95: "P95"})
    .reset_index()
)

resumo_uf = resumo_uf.merge(percentis_uf, on=colunas_grupo)
resumo_uf["AMPLITUDE_INTERQUARTIL"] = resumo_uf["Q3"] - resumo_uf["Q1"]

#a nota representativa é sempre a mediana
resumo_uf["NOTA_REPRESENTATIVA"] = resumo_uf["MEDIANA"]


#junta o tamanho da amostra e o total de elegíveis de cada grupo
resumo_uf = resumo_uf.merge(
    validacao_amostra[chaves + ["REGISTROS_ELEGIVEIS"]],
    on=chaves,
    validate="one_to_one",
)
assert resumo_uf["N"].sum() == len(dados_analise)
print(f"Resumo estadual: {len(resumo_uf)} grupos de ano, UF e área.")


Resumo estadual: 675 grupos de ano, UF e área.


## 11. Calcular os resumos regionais e nacional

Para resumir uma região, agrupamos diretamente as notas sorteadas de suas UFs. Cada nota tem o mesmo peso; UFs com mais participantes contribuem com mais notas. Não calculamos a média ou a mediana das medianas estaduais. O resumo nacional também parte das notas, por ano e área.

Como o sorteio é de aproximadamente 10% em cada grupo, a composição da amostra acompanha aproximadamente a dos elegíveis com UF conhecida. O arredondamento gera pequenas diferenças nessa fração. Uma inferência populacional exigiria considerar pesos e incerteza da amostragem. Aqui, o resumo nacional continua restrito aos participantes com UF da escola informada.


In [ ]:
# @title
#calculo as mesmas medidas juntando as UFs de cada região
colunas_regiao = ["ANO", "REGIAO", "AREA"]
grupos_regiao = dados_analise.groupby(colunas_regiao, observed=True)["NOTA"]

resumo_regiao = grupos_regiao.agg(
    MEDIA="mean",
    MEDIANA="median",
    N="count",
    DESVIO_PADRAO="std",
    ASSIMETRIA="skew",
).reset_index()

percentis_regiao = (
    grupos_regiao.quantile([0.05, 0.25, 0.75, 0.95])
    .unstack()
    .rename(columns={0.05: "P5", 0.25: "Q1", 0.75: "Q3", 0.95: "P95"})
    .reset_index()
)

resumo_regiao = resumo_regiao.merge(percentis_regiao, on=colunas_regiao)
resumo_regiao["AMPLITUDE_INTERQUARTIL"] = resumo_regiao["Q3"] - resumo_regiao["Q1"]
resumo_regiao["NOTA_REPRESENTATIVA"] = resumo_regiao["MEDIANA"]

#resumo nacional por ano e área
resumo_brasil = (
    dados_analise.groupby(["ANO", "AREA"], observed=True)["NOTA"]
    .agg(MEDIA="mean", MEDIANA="median", N="count", DESVIO_PADRAO="std")
    .reset_index()
)


## 12. Distribuição das notas por estado, área e ano

Escolha UF, ano e área para ver onde as notas se concentram. O histograma mostra percentuais; as linhas marcam a média e a mediana. O filtro de faixas muda a aparência das barras, sem alterar as medidas.

Marque **Ver tabela detalhada** para consultar tamanho da amostra, elegíveis, medidas centrais, dispersão e extremos. Todas as notas do grupo entram. P5 e P95 descrevem a dispersão das notas, não um intervalo de confiança.


In [ ]:
# @title
def mostrar_distribuicao(ano, area, uf_escola, faixas, ver_tabela=False):
    selecao = (
        (dados_analise["ANO"] == ano)
        & (dados_analise["AREA"] == area)
        & (dados_analise["UF_ESCOLA"] == uf_escola)
    )
    notas = dados_analise.loc[selecao, "NOTA"]
    if notas.empty:
        print("Não há notas neste recorte.")
        return
    limites = np.linspace(notas.min(), notas.max() + 0.001, faixas + 1)
    frequencias, limites = np.histogram(notas, bins=limites)
    tabela = pd.DataFrame(
        {
            "Centro da faixa": (limites[:-1] + limites[1:]) / 2,
            "Percentual": frequencias / len(notas) * 100,
            "Notas": frequencias,
            "Início": limites[:-1],
            "Fim": limites[1:],
        }
    )
    grafico = px.bar(
        tabela,
        x="Centro da faixa",
        y="Percentual",
        hover_data={"Início": ":.1f", "Fim": ":.1f", "Notas": ":,", "Percentual": ":.2f"},
        title=f"Em quais faixas as notas se concentram?<br><sup>{nome_por_uf[uf_escola]} · {area} · {ano}</sup>",
    )
    grafico.update_traces(width=(limites[1] - limites[0]) * 0.95)
    grafico.add_vline(x=notas.mean(), line_color="#C34109", line_dash="dot")
    grafico.add_vline(x=notas.median(), line_color="#007763", line_dash="dash")
    grafico.update_layout(height=480, margin=dict(t=95, b=65, l=70, r=35))
    grafico.update_xaxes(title="Nota na prova (pontos)")
    grafico.update_yaxes(title="Notas da amostra em cada faixa (%)")
    fixar_percentual(grafico, eixo="y")
    exibir_grafico(grafico)
    display(
        Markdown(
            f"**{len(notas):,} notas** · Média: **{notas.mean():.2f}** "
            f"· Mediana: **{notas.median():.2f}** pontos.  \n"
            "Laranja pontilhado: média. Verde tracejado: mediana."
        )
    )
    selecao_resumo = (
        (resumo_uf["ANO"] == ano)
        & (resumo_uf["AREA"] == area)
        & (resumo_uf["UF_ESCOLA"] == uf_escola)
    )
    tabela_resumo = resumo_uf[selecao_resumo]
    if ver_tabela:
        exibir_tabela(
            tabela_resumo[
                [
                    "N",
                    "REGISTROS_ELEGIVEIS",
                    "MEDIA",
                    "MEDIANA",
                    "DESVIO_PADRAO",
                    "ASSIMETRIA",
                    "P5",
                    "Q1",
                    "Q3",
                    "P95",
                    "MINIMO",
                    "MAXIMO",
                ]
            ].round(2)
        )


montar_painel(
    mostrar_distribuicao,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    uf_escola=widgets.Dropdown(
        options=sorted(ufs_validas), value="SP", description="UF escola:"
    ),
    faixas=widgets.IntSlider(
        min=10,
        max=80,
        step=10,
        value=40,
        description="Faixas:",
        continuous_update=False,
    ),
)


## 15. Diagnóstico opcional: Shapiro–Wilk e gráfico Q–Q

Escolha uma prova objetiva e marque **Executar**. Usamos até 5.000 notas, sorteadas com a mesma semente, no teste e no gráfico Q–Q. Esse limite segue a ressalva do SciPy sobre a precisão do p-valor acima de 5.000 observações; não garante todos os pressupostos do teste.

W é a estatística do Shapiro–Wilk. Um p-valor menor que 0,05 indica evidência contra normalidade, sob as hipóteses do teste. Um valor maior ou igual a 0,05 indica que o teste não a rejeitou; não prova que a distribuição seja normal. O p-valor também não é a probabilidade de normalidade nem a confiança da mediana.

No Q–Q, pontos próximos da linha sugerem aproximação ao padrão normal. Curvas e desvios nas pontas ajudam a observar diferenças e caudas.

O diagnóstico é exploratório: amostras grandes detectam desvios pequenos, há notas arredondadas e empatadas, e participantes compartilham escolas. Assim, independência e continuidade são aproximações. Redação fica fora deste teste por sua escala discreta e seus muitos empates, mas permanece nos outros painéis. Repetir o teste em várias UFs não equivale a compará-las estatisticamente.

**A mediana continua sendo a medida principal, qualquer que seja o resultado.**


In [ ]:
# @title
def mostrar_normalidade(ano, area, uf_escola, executar):
    if not executar:
        print("Marque Executar para ver o diagnóstico e o gráfico Q–Q.")
        return
    selecao = (
        (dados_analise["ANO"] == ano)
        & (dados_analise["AREA"] == area)
        & (dados_analise["UF_ESCOLA"] == uf_escola)
    )
    notas = dados_analise.loc[selecao, "NOTA"]
    if len(notas) < 3 or notas.nunique() < 2:
        print("Quantidade ou variação insuficiente para o diagnóstico.")
        return
    notas_teste = notas.sample(n=min(5000, len(notas)), random_state=SEMENTE)
    resultado = shapiro(notas_teste)
    exibir_tabela(
        pd.DataFrame(
            {
                "N total do recorte": [len(notas)],
                "N no diagnóstico": [len(notas_teste)],
                "W de Shapiro": [resultado.statistic],
                "p-valor": [resultado.pvalue],
            }
        )
    )
    if resultado.pvalue < 0.05:
        print("Evidência contra normalidade (5%).")
    else:
        print("O teste não rejeitou normalidade (5%); isso não a comprova.")
    print("Diagnóstico exploratório; a medida principal continua sendo a mediana.")
    #compara quantis observados com quantis teóricos de uma distribuição normal
    (quantis_normais, notas_ordenadas), (inclinacao, intercepto, correlacao) = probplot(
        notas_teste, dist="norm"
    )
    grafico = go.Figure()
    grafico.add_scatter(
        x=quantis_normais,
        y=notas_ordenadas,
        mode="markers",
        name="Notas observadas",
        marker=dict(size=4),
    )
    grafico.add_scatter(
        x=quantis_normais,
        y=inclinacao * quantis_normais + intercepto,
        mode="lines",
        name="Referência normal",
    )
    grafico.update_layout(
        title=f"As notas seguem o formato de uma curva normal?<br><sup>{uf_escola} · {area} · {ano}</sup>",
        xaxis_title="Posição esperada na curva normal",
        yaxis_title="Nota observada (pontos)",
        height=470,
        margin=dict(t=100, b=110, l=80, r=35),
        legend=dict(orientation="h", y=-0.22),
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_normalidade,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS[:-1],
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    uf_escola=widgets.Dropdown(
        options=sorted(ufs_validas), value="SP", description="UF escola:"
    ),
    executar=widgets.Checkbox(value=False, description="Executar"),
)


## 16. Mapa da mediana por UF da escola

O código do IBGE liga a tabela ao desenho de cada estado. A cor representa a mediana. Ao mudar o ano, mantemos a escala de cor da área para permitir a comparação no tempo.

Passe o cursor para consultar tamanho da amostra, média, mediana e quartis. A UF é a da escola, não a da residência ou da prova. Uma mediana maior descreve esse recorte; não mede a qualidade de todo o sistema educacional.

Use `malha_ufs_plotly.geojson`, que já tem a orientação dos polígonos preparada para o Plotly. Sem esse arquivo, os outros painéis continuam funcionando.


In [ ]:
# @title
#sem malha, apenas o painel do mapa fica indisponível.
mapa_brasil = None
if ARQUIVO_MAPA.exists():
    with open(ARQUIVO_MAPA, encoding="utf-8") as arquivo:
        mapa_brasil = json.load(arquivo)

#manntém a mesma escala de cor ao trocar o ano dentro de cada área
limites_cores = resumo_uf.groupby("AREA", observed=True)["MEDIANA"].agg(["min", "max"])


def mostrar_mapa(ano, area):
    if mapa_brasil is None:
        print(
            "Coloque malha_ufs_plotly.geojson na pasta e execute esta célula novamente. Os outros painéis funcionam sem a malha."
        )
        return
    dados_mapa = resumo_uf[
        (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    ].copy()

    grafico = px.choropleth(
        dados_mapa,
        geojson=mapa_brasil,
        locations="COD_UF_IBGE",
        featureidkey="properties.codarea",
        color="MEDIANA",
        color_continuous_scale=ESCALA_SEQUENCIAL,
        range_color=(limites_cores.loc[area, "min"], limites_cores.loc[area, "max"]),
        hover_name="NOME_UF",
        hover_data={
            "COD_UF_IBGE": False,
            "UF_ESCOLA": True,
            "REGIAO": True,
            "MEDIANA": ":.2f",
            "MEDIA": ":.2f",
            "Q1": ":.2f",
            "Q3": ":.2f",
            "N": ":,.0f",
        },
        labels={
            "UF_ESCOLA": "UF da escola",
            "REGIAO": "Região",
            "MEDIANA": "Mediana",
            "MEDIA": "Média",
            "Q1": "1º quartil",
            "Q3": "3º quartil",
            "N": "Notas na amostra",
        },
    )

    grafico.update_geos(fitbounds="geojson", projection_type="mercator", visible=False)
    grafico.update_traces(marker_line_color="white", marker_line_width=0.7)
    grafico.update_layout(
        title={
            "text": (
                "<b>Qual é a nota do meio em cada estado?</b><br>"
                f"<span style='font-size:17px'>{area} — {ano}</span><br>"
                "<span style='font-size:13px;color:#555'>A cor mostra a mediana das notas</span>"
            ),
            "x": 0.5,
            "xanchor": "center",
        },
        coloraxis_colorbar_title="Mediana (pontos)",
        height=570,
        margin={"l": 10, "r": 20, "t": 110, "b": 15},
        paper_bgcolor="white",
        font={"family": "Arial, sans-serif", "color": "#222"},
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_mapa,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="330px"),
    ),
)


## 17. Evolução regional: uma área ou todas

Cada ponto mostra a mediana das notas da região em um ano e área. Escolha o período e compare as regiões ou selecione apenas uma. A visualização começa em Matemática. A opção **Todas** abre um painel por área, com eixos próprios.

As linhas ligam amostras de pessoas diferentes. Mudanças podem refletir composição dos participantes, cobertura escolar e contexto do exame. As escalas de Redação e das provas objetivas não devem ser interpretadas como equivalentes.


In [ ]:
# @title
def mostrar_evolucao(area, regiao, periodo):
    tabela = resumo_regiao[resumo_regiao["ANO"].between(periodo[0], periodo[1])].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    if area != "Todas":
        tabela = tabela[tabela["AREA"] == area]
    tabela = tabela.sort_values("ANO")
    grafico = px.line(
        tabela,
        x="ANO",
        y="MEDIANA",
        color="REGIAO",
        markers=True,
        facet_col="AREA" if area == "Todas" else None,
        facet_col_spacing=0.12,
        facet_col_wrap=2 if area == "Todas" else 0,
        color_discrete_map=cores_regioes,
        category_orders={"AREA": AREAS, "REGIAO": REGIOES},
        hover_data={"N": True, "Q1": ":.1f", "Q3": ":.1f"},
        labels={"MEDIANA": "Nota do meio (mediana, em pontos)", "ANO": "Ano do ENEM", "REGIAO": "Região"},
        title=f"Como as notas mudaram em cada região?<br><sup>{area}</sup>",
    )
    grafico.update_xaxes(dtick=1, showticklabels=True)
    grafico.update_yaxes(matches=None, showticklabels=True)
    grafico.update_layout(
        height=940 if area == "Todas" else 500,
        margin=dict(t=110, b=120, l=75, r=40),
        legend=dict(orientation="h", y=-0.12),
    )
    # Deixa só o nome da área no título de cada painel.
    for anotacao in grafico.layout.annotations:
        anotacao.text = anotacao.text.replace("AREA=", "")
    exibir_grafico(grafico)


montar_painel(
    mostrar_evolucao,
    area=widgets.Dropdown(
        options=["Todas"] + AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    periodo=widgets.IntRangeSlider(
        min=2020,
        max=2024,
        value=[2020, 2024],
        step=1,
        description="Período:",
        continuous_update=False,
    ),
)


## 18. Ranking de UFs no Brasil ou em uma região

Escolha ano, área e região para ordenar as UFs pela mediana. Empates recebem a mesma posição. A ordem usa a precisão do resultado, embora o gráfico mostre duas casas decimais. A tabela mantém tamanho da amostra, elegíveis, média e quartis.

O ranking descreve a amostra. A primeira posição não prova superioridade estatística ou melhor educação. Diferenças pequenas podem mudar em outro sorteio, e ainda não calculamos intervalos de confiança. Compare também a média e a distribuição das notas.


In [ ]:
# @title
def mostrar_ranking(ano, area, regiao, ordem, quantidade, ver_tabela=False):
    selecao = (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    tabela = resumo_uf[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    tabela["Posição no recorte"] = (
        tabela["MEDIANA"].rank(method="min", ascending=False).astype(int)
    )
    menores_primeiro = ordem == "Menores"
    tabela = tabela.sort_values(
        ["MEDIANA", "UF_ESCOLA"], ascending=[menores_primeiro, True]
    ).head(quantidade)
    grafico = px.bar(
        tabela.sort_values("MEDIANA"),
        x="MEDIANA",
        y="UF_ESCOLA",
        orientation="h",
        color="REGIAO",
        color_discrete_map=cores_regioes,
        text="MEDIANA",
        hover_data={"N": True, "Q1": ":.2f", "Q3": ":.2f", "Posição no recorte": True},
        labels={
            "UF_ESCOLA": "UF da escola",
            "MEDIANA": "Nota do meio (mediana, em pontos)",
            "REGIAO": "Região",
        },
        title=f"{ordem} notas do meio entre os estados<br><sup>{area} · {ano} · {regiao}</sup>",
    )
    grafico.update_traces(
        texttemplate="%{text:.2f}", textposition="outside", cliponaxis=False
    )
    grafico.update_layout(
        height=max(470, 27 * len(tabela) + 200),
        margin=dict(t=100, r=100, b=110, l=75),
        legend=dict(orientation="h", y=-0.12, title_text=""),
    )
    grafico.update_xaxes(range=[0, tabela["MEDIANA"].max() * 1.18])
    grafico.update_yaxes(categoryorder="total ascending", title="Estado da escola")
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(
            tabela[
                [
                    "Posição no recorte",
                    "UF_ESCOLA",
                    "NOME_UF",
                    "REGIAO",
                    "N",
                    "REGISTROS_ELEGIVEIS",
                    "MEDIANA",
                    "MEDIA",
                    "Q1",
                    "Q3",
                ]
            ].round(2)
        )


montar_painel(
    mostrar_ranking,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    ordem=widgets.Dropdown(options=["Maiores", "Menores"], description="Ordem:"),
    quantidade=widgets.IntSlider(
        min=1, max=27, value=10, description="Exibir:", continuous_update=False
    ),
)


## 19. Evolução dos estados em cada região

Escolha região, área e período para acompanhar as medianas estaduais. Marque **Ver tabela detalhada** para consultar todas as UFs empatadas na maior mediana de cada ano.

São amostras de participantes diferentes ao longo do tempo. Mudanças nas linhas podem refletir composição, cobertura e distribuição das notas; o gráfico sozinho não permite atribuir causas.


In [ ]:
# @title
def mostrar_estados(regiao, area, periodo, ver_tabela=False):
    selecao = (
        (resumo_uf["REGIAO"] == regiao)
        & (resumo_uf["AREA"] == area)
        & resumo_uf["ANO"].between(periodo[0], periodo[1])
    )
    tabela = resumo_uf[selecao].sort_values("ANO").copy()
    grafico = px.line(
        tabela,
        x="ANO",
        y="MEDIANA",
        color="UF_ESCOLA",
        markers=True,
        hover_data={"N": True, "REGISTROS_ELEGIVEIS": True},
        labels={"ANO": "Ano do ENEM", "MEDIANA": "Nota do meio (mediana, em pontos)", "UF_ESCOLA": "UF escolar"},
        title=f"Como as notas mudaram nos estados de {regiao}?<br><sup>{area}</sup>",
    )
    grafico.update_xaxes(dtick=1)
    grafico.update_layout(
        height=500,
        margin=dict(t=100, b=100, l=75, r=35),
        legend=dict(orientation="h", y=-0.2, title_text=""),
    )
    exibir_grafico(grafico)
    maior_por_ano = tabela.groupby("ANO")["MEDIANA"].transform("max")
    #mostra  todas as UFs empatadas na maior mediana de cada ano
    destaques = tabela.loc[
        tabela["MEDIANA"] == maior_por_ano,
        ["ANO", "UF_ESCOLA", "MEDIANA", "N"],
    ]
    if ver_tabela:
        exibir_tabela(destaques.round(2))


montar_painel(
    mostrar_estados,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    regiao=widgets.Dropdown(options=REGIOES, value="Sudeste", description="Região:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    periodo=widgets.IntRangeSlider(
        min=2020,
        max=2024,
        value=[2020, 2024],
        description="Período:",
        continuous_update=False,
    ),
)


## 20. Posição das regiões em cada área

Cada coluna mostra uma área. A cor indica a posição da região pela mediana dentro daquela coluna: 1 é a maior. Empates recebem a mesma posição. O número escrito é a mediana em pontos, e o cursor mostra a posição.

As posições permitem comparar a ordem das regiões sem misturar as escalas das áreas. A cor não mostra o tamanho da diferença: confira as medianas, pois posições consecutivas podem estar próximas ou distantes em pontos.


In [ ]:
# @title
def mostrar_painel_areas(ano):
    tabela = resumo_regiao[resumo_regiao["ANO"] == ano]
    tabela = tabela.pivot(index="REGIAO", columns="AREA", values="MEDIANA")
    tabela = tabela.reindex(index=REGIOES, columns=AREAS)
    posicoes = tabela.rank(axis=0, ascending=False, method="min")
    grafico = go.Figure(
        go.Heatmap(
            z=posicoes.to_numpy(),
            x=AREAS,
            y=REGIOES,
            customdata=tabela.to_numpy(),
            text=[[f'{v:.2f}'.replace('.', ',') for v in linha] for linha in tabela.to_numpy()],
            texttemplate="%{text}",
            colorscale=ESCALA_SEQUENCIAL, reversescale=True,
            zmin=1,
            zmax=5,
            colorbar=dict(title="Posição", tickvals=[1, 2, 3, 4, 5], thickness=14),
            xgap=4,
            ygap=4,
            textfont=dict(size=16),
            hovertemplate="%{y}<br>%{x}<br>Mediana: %{customdata:.2f}<br>Posição: %{z}<extra></extra>",
        )
    )
    grafico.update_layout(
        title=f"Como as regiões se comparam nas cinco áreas? · {ano}<br><sup>O número é a mediana. A cor mostra a posição da região.</sup>",
        height=480,
        margin=dict(t=100, b=100, l=120, r=70),
    )
    grafico.update_xaxes(
        title="Área da prova",
        tickvals=AREAS,
        ticktext=[
            "Linguagens",
            "Ciências<br>Humanas",
            "Ciências da<br>Natureza",
            "Matemática",
            "Redação",
        ],
        tickangle=0,
    )
    grafico.update_yaxes(autorange="reversed", title="Região da escola")
    exibir_grafico(grafico)


montar_painel(
    mostrar_painel_areas,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
)


## 21. Quantos alcançam uma nota escolhida?

Escolha ano, área, região e uma nota de referência. Para cada UF, calculamos a porcentagem da amostra com nota maior ou igual ao limite. O denominador é o tamanho da amostra daquela UF, mostrado na tabela e no cursor.

O limite serve para explorar os dados; não é uma nota oficial de aprovação ou proficiência. Como as áreas têm escalas diferentes, usar o mesmo limite entre elas exige justificativa.


In [ ]:
# @title
def mostrar_percentual(ano, area, regiao, limite, ver_tabela=False):
    selecao = (dados_analise["ANO"] == ano) & (dados_analise["AREA"] == area)
    tabela = dados_analise[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    tabela["ATINGIU"] = tabela["NOTA"] >= limite
    grupos = tabela.groupby("UF_ESCOLA", observed=True)["ATINGIU"]
    resumo = grupos.agg(N="count", Quantidade="sum").reset_index()
    resumo["Percentual"] = 100 * resumo["Quantidade"] / resumo["N"]
    grafico = px.bar(
        resumo.sort_values("Percentual"),
        x="Percentual",
        y="UF_ESCOLA",
        orientation="h",
        text="Percentual",
        hover_data=["N", "Quantidade"],
        title=f"Que parcela atingiu {limite} pontos ou mais?<br><sup>{area} · {ano} · {regiao}</sup>",
        labels={"Percentual": "Notas da amostra que atingiram o corte (%)", "UF_ESCOLA": "UF escolar"},
    )
    grafico.update_traces(
        texttemplate="%{text:.1f}%", textposition="outside", cliponaxis=False
    )
    fixar_percentual(grafico)
    grafico.update_yaxes(title="Estado da escola")
    grafico.update_layout(
        height=max(420, 27 * len(resumo) + 120), margin=dict(t=100, r=75, b=65, l=75)
    )
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(resumo.sort_values("Percentual", ascending=False).round(2))


montar_painel(
    mostrar_percentual,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    limite=widgets.IntSlider(
        min=0,
        max=1000,
        step=20,
        value=600,
        description="Nota ≥",
        continuous_update=False,
    ),
)


## 22. Mais participantes estão associados a maior mediana?

Cada ponto representa uma UF. O eixo horizontal mostra os elegíveis com UF da escola, contados antes do sorteio; o vertical mostra a mediana da amostra. A cor identifica a região, e o cursor mostra o estado. A escala logarítmica ajuda a visualizar UFs com volumes menores.

O gráfico permite observar uma possível associação entre volume e mediana. Ele não demonstra causa nem explica a participação por população, renda ou cultura sem outros dados.


In [ ]:
# @title
def mostrar_volume_nota(ano, area, escala_log):
    selecao = (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    tabela = resumo_uf[selecao]
    grafico = px.scatter(
        tabela,
        x="REGISTROS_ELEGIVEIS",
        y="MEDIANA",
        color="REGIAO",
        hover_name="UF_ESCOLA",
        color_discrete_map=cores_regioes,
        log_x=escala_log,
        hover_data={"N": True, "NOME_UF": True},
        labels={
            "REGISTROS_ELEGIVEIS": "Notas válidas com estado da escola",
            "MEDIANA": "Nota do meio (mediana, em pontos)",
            "REGIAO": "Região",
        },
        title=f"Estados com mais participantes têm notas mais altas?<br><sup>{area} · {ano}</sup>",
    )
    grafico.update_traces(marker=dict(size=10))
    grafico.update_layout(
        height=520,
        margin=dict(t=100, b=110, l=80, r=40),
        legend=dict(orientation="h", y=-0.22, title_text=""),
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_volume_nota,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    escala_log=widgets.Checkbox(value=True, description="Volume em escala logarítmica"),
)


## 23. Preparar tabelas para as próximas etapas

Deixamos prontos os resumos por UF, região e Brasil, com uma linha por ano e área em cada território. A base estadual também informa a fração amostral e a medida principal. Ative `EXPORTAR_TABELAS` para salvar os três CSVs em `tabelas_descritivas`.

Mantemos as medidas, as contagens e as chaves territoriais. O Shapiro não entra como atributo para prever nota, pois é calculado a partir da própria distribuição das notas.

Um modelo com unidade UF × ano terá apenas 135 combinações nos cinco anos. As áreas não são cinco observações territoriais independentes. A cobertura disponível é nacional por ano e área; não podemos apresentá-la como cobertura de cada UF, pois desconhecemos a UF de quem não a informou.

**Pergunta para a Etapa 2.** Que informações sobre quem faz o ENEM e sobre os territórios precisamos integrar para compreender melhor as diferenças que acabamos de descrever?

In [ ]:
# @title
base_proxima_etapa = resumo_uf.copy()
base_proxima_etapa["FRACAO_AMOSTRAL"] = (
    base_proxima_etapa["N"] / base_proxima_etapa["REGISTROS_ELEGIVEIS"]
)
base_proxima_etapa["MEDIDA_PRINCIPAL"] = "Mediana da amostra"
assert not base_proxima_etapa.duplicated(["ANO", "UF_ESCOLA", "AREA"]).any()
assert not resumo_regiao.duplicated(["ANO", "REGIAO", "AREA"]).any()
print("Base estadual pronta:", base_proxima_etapa.shape)
print("Base regional pronta:", resumo_regiao.shape)

if EXPORTAR_TABELAS:
    pasta_exportacao = PASTA_DADOS / "tabelas_descritivas"
    pasta_exportacao.mkdir(exist_ok=True)
    base_proxima_etapa.to_csv(
        pasta_exportacao / "enem_descritiva_uf_escola.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    resumo_regiao.to_csv(
        pasta_exportacao / "enem_descritiva_regiao.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    resumo_brasil.to_csv(
        pasta_exportacao / "enem_descritiva_amostra_nacional.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    print("Tabelas salvas em:", pasta_exportacao)
else:
    print(
        "Tabelas disponíveis na memória. Ative EXPORTAR_TABELAS na configuração para salvar CSVs."
    )


Base estadual pronta: (675, 22)
Base regional pronta: (125, 14)
Tabelas disponíveis na memória. Ative EXPORTAR_TABELAS na configuração para salvar CSVs.


# Etapa 2 · Quem faz o ENEM e quais condições acompanham o desempenho?

Na Etapa 1, descrevemos as distribuições e localizamos diferenças. Agora perguntamos se estamos comparando grupos semelhantes, como variam os perfis dos participantes e quais condições territoriais acompanham as notas.

1. **Quem entra na comparação?** Cobertura da UF escolar, elegibilidade e situação escolar.
2. **Onde e quando as notas mudam?** Distribuição, trajetória e mudança na composição.
3. **Quais perfis aparecem entre as áreas?** Duas medidas complementares usando quatro notas da mesma pessoa.
4. **Que informações externas ajudam a interpretar as diferenças?** Integração auditada, disponibilidade de recursos e estabilidade das associações.
5. **O que pode seguir para a Etapa 3?** Alvo, atributos candidatos e calendário de disponibilidade.

| Unidade | Onde entra | Universo |
|---|---|---|
| Inscrição por edição | Situação escolar entre inscritos | Arquivo público de cada edição |
| Nota por inscrição e área | Distribuições e comparações de redes | Amostra de 10% de elegíveis com UF escolar |
| Nota por inscrição e área | Cobertura, evolução e decomposição | Resumos de todos os elegíveis do recorte |
| Inscrição com quatro provas | Perfis relativos | Amostra conjunta de 10% por UF e ano |
| Escola por ano | Oferta e infraestrutura | Escolas ativas com ensino médio no Censo |
| UF por ano e área | Associações e alvo futuro | Resumos do ENEM ligados a contexto territorial |

As notas das cinco áreas permanecem separadas. As edições são cortes sucessivos de participantes. Veja a [revisão por célula](REVISAO_ETAPA2.md) para rastrear decisões e dependências.

In [ ]:
# @title
from pathlib import Path
import json
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown
from scipy.stats import gaussian_kde, spearmanr
from scipy.signal import find_peaks


if not PASTA_ENTRADA.is_dir():
    raise FileNotFoundError('Ajuste PASTA_ESTUDO para a pasta que contém dados/.')

PASTA_RESULTADOS = PASTA_ESTUDO / 'resultados_etapa2'
PASTA_RESULTADOS.mkdir(exist_ok=True)

def ler(nome):
    return pd.read_csv(PASTA_ENTRADA / nome, sep=';')

def salvar(tabela, nome):
    tabela.to_csv(PASTA_RESULTADOS / nome, sep=';', index=False, encoding='utf-8-sig')

enem_uf = ler('enem_uf_2020_2024.csv')
enem_regiao = ler('enem_regiao_2020_2024.csv')
enem_nacional = ler('enem_nacional_2020_2024.csv')
enem_redes = ler('enem_rede_local_2020_2024.csv')
enem_estadual_completo = ler('enem_estadual_completo_2020_2024.csv')
cobertura = ler('cobertura_uf_escola.csv')
amostra = pd.read_parquet(PASTA_ENTRADA / 'amostra_notas_contexto.parquet')
escolas = pd.read_parquet(PASTA_ENTRADA / 'escolas_ensino_medio.parquet')
perfis = pd.read_parquet(PASTA_ENTRADA / 'perfis_contexto_escolar.parquet')

ANOS_ESTUDO = list(range(2020, 2025))
AREAS_ESTUDO = ['Linguagens', 'Ciências Humanas', 'Ciências da Natureza', 'Matemática', 'Redação']
REGIOES_ESTUDO = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
UFS_ESTUDO = sorted(enem_uf.UF_ESCOLA.unique())
TERRITORIOS = ['Brasil (UF informada)'] + REGIOES_ESTUDO + UFS_ESTUDO
REDES_ESTUDO = ['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada', 'Não informada']
CORES_REGIOES = cores_regioes

print('Pasta:', PASTA_ESTUDO)
print('Período: 2020–2024 | 27 UFs | 5 áreas')

Pasta: /content/drive/MyDrive/ENEM/estudo_enem_revisado
Período: 2020–2024 | 27 UFs | 5 áreas


### Navegação dos painéis

Use **Área**, **Ano** ou **Período** onde o indicador admite esses recortes. Quando um painel tem mais de uma pergunta, **Vista** alterna o gráfico em largura inteira. Passe o mouse para ver o valor e, quando aplicável, a contagem e o denominador. Clique na legenda para ocultar uma série ou dê dois cliques para isolá-la.

Os painéis de proporções usam **0% a 100%** para comparar recortes na mesma escala. Na seção **Distribuições, território e mudanças no tempo**, as frequências e as caudas usam limites ajustados aos valores observados, com as duas caudas na mesma escala por recorte. As visões gerais das cinco áreas e dos recursos escolares ajustam os limites das cores à matriz exibida. Ao mudar os filtros, consulte o valor de cada célula e a escala da legenda. Notas, diferenças, contagens e correlações usam a unidade indicada no eixo.

Cada figura tem um botão **Baixar gráfico (PNG)**. Ele salva o recorte exibido, com os filtros e as séries visíveis no momento do clique. A barra de ferramentas permite ampliar os eixos de notas e restaurar a vista.


In [ ]:
# @title
def escolher(opcoes, valor, descricao):
    return widgets.Dropdown(options=opcoes, value=valor, description=descricao,
        style={'description_width': '100px'}, layout=widgets.Layout(width='360px'))

def periodo():
    return widgets.IntRangeSlider(value=[2020, 2024], min=2020, max=2024,
        description='Período:', continuous_update=False, layout=widgets.Layout(width='360px'))

def painel(funcao, **controles):
    if not MOSTRAR_FILTROS:
        funcao(**{nome: controle.value for nome, controle in controles.items()})
        return
    saida = widgets.interactive_output(funcao, controles)
    linha = widgets.HBox(list(controles.values()), layout=widgets.Layout(flex_flow='row wrap'))
    display(widgets.VBox([linha, saida]))

def recortar(tabela, territorio='Brasil (UF informada)', rede='Todas'):
    if territorio in UFS_ESTUDO:
        tabela = tabela[tabela.UF_ESCOLA.eq(territorio)]
    elif territorio in REGIOES_ESTUDO:
        tabela = tabela[tabela.REGIAO.eq(territorio)]
    if rede == 'Públicas':
        tabela = tabela[tabela.REDE.isin(['Federal', 'Estadual', 'Municipal'])]
    elif rede != 'Todas':
        tabela = tabela[tabela.REDE.eq(rede)]
    return tabela

def tabela_opcional(dados, mostrar):
    if mostrar:
        exibir_tabela(dados.round(2).reset_index(drop=True))

def configurar_painel(fig, titulo, detalhe=None, altura=455):
    """Padroniza títulos, espaçamento e legendas dos gráficos da Etapa 2."""
    texto = titulo if not detalhe else f'{titulo}<br><sup>{detalhe}</sup>'
    fig.update_layout(title_text=texto, height=altura,
                      margin=dict(l=60, r=46, t=83 if detalhe else 62, b=62),
                      legend=dict(orientation='h', yanchor='top', y=-.18,
                                  xanchor='left', x=0))
    return fig


def limites_cores_percentuais(porcentagens):
    """Usa uma escala comum a todas as células da matriz no recorte."""
    valores = porcentagens[np.isfinite(porcentagens)]
    if np.isclose(valores.min(), valores.max()):
        return 0, 100
    minimo = max(0, 5 * np.floor(valores.min() / 5))
    maximo = min(100, 5 * np.ceil(valores.max() / 5))
    return minimo, maximo


## 1. Cobertura, elegibilidade e situação escolar

### Auditoria nacional · o que muda quando exigimos UF escolar?

**Vale manter este gráfico?** Sim. A comparação mostra se o grupo que conseguimos localizar tem notas parecidas com as do grupo sem UF escolar. Isso ajuda a avaliar até onde podemos generalizar a análise territorial. A diferença mede um contraste descritivo entre grupos observados. Cobertura, composição dos participantes e regras de publicação podem variar entre as edições.

**Três recortes, três funções**

| Recorte | Para que usamos | Unidade e denominador |
|---|---|---|
| Todos os elegíveis, com e sem UF escolar | Auditoria nacional de cobertura e seleção | Nota por inscrição, edição e área. A cobertura divide as notas com UF por todas as notas elegíveis. Para a média, dividimos a soma das notas do grupo pela sua contagem. |
| Amostra limpa de aproximadamente 10%, com UF escolar | Histogramas, percentis, caudas, comparações por rede e localização | Notas sorteadas por edição × UF × área. Os perfis territoriais usam a amostra conjunta de participantes com as quatro provas válidas. |
| Resumos completos dos elegíveis com UF escolar | Trajetórias territoriais, indicadores de desempenho e integração com as fontes por UF | As estatísticas são calculadas a partir de todas as notas elegíveis no território e na área. Os resumos entram na junção pelas chaves de UF e ano. |

A elegibilidade exige presença e nota disponível na área. Em Redação, exige também situação regular. O filtro de UF vale para a amostra e para as bases territoriais. Os registros sem UF formam um grupo agregado nacional, fora da junção por estado. A amostragem descreve a população identificável sob o desenho do sorteio. A participação no ENEM e a ausência de UF limitam sua representatividade para outros públicos.

**Como ler os gráficos.** Compare primeiro a cobertura em 0–100%. Em seguida, use **Médias dos grupos** para conferir as duas séries em pontos e **Diferença de médias** para examinar o contraste com − sem UF. Acima de zero, o grupo com UF tem média maior. O cursor informa a contagem usada em cada cálculo. As edições representam cortes sucessivos de participantes.

**Exemplo em Matemática.** Entre 2020 e 2024, a cobertura passou de 20,8% para 39,9%. No mesmo período, a diferença com − sem UF passou de +25,1 para −10,8 pontos. A mudança de sinal reforça que o conjunto territorial observado deve ser interpretado edição por edição. Atribuir o contraste à cobertura exigiria separar mudanças de composição e de desempenho.

Em 2024, o Inep voltou a disponibilizar código de escola obtido da matrícula no Censo Escolar, com proteção para escolas com poucos participantes. A edição também passou a publicar arquivos de participantes e resultados sem chave comum. Essas mudanças entram na leitura da série. [Leia-me de 2024, páginas 7–8](fontes/etapa_anterior/Leia_Me_Enem_2024.pdf)


In [ ]:
# @title
notas_situacao = ler('notas_situacao_escolar.csv')
situacao_com_cobertura = notas_situacao.assign(
    COBERTURA=np.where(notas_situacao.UF_ESCOLA.eq('Sem UF escolar'), 'UF ausente', 'UF conhecida')
)
CORES_COBERTURA = {'UF conhecida': '#0056B3', 'UF ausente': '#C34109'}


def barras_taxa(d, titulo, eixo, cor='#0056B3', eixo_y='Ano do ENEM'):
    # Cada barra usa o denominador do seu próprio grupo.
    d = d.loc[d.TOTAL.gt(0)].copy()
    if d.empty:
        print('Sem observações para este recorte.')
        return
    taxas = 100 * d.N / d.TOTAL
    fig = go.Figure(go.Bar(
        x=taxas, y=d.ROTULO.astype(str), orientation='h',
        marker_color=cor, width=.61, customdata=d[['N', 'TOTAL']],
        text=[f'{v:.2f}%' if 0 < v < 1 else f'{v:.1f}%' for v in taxas],
        textposition='outside', cliponaxis=False,
        hovertemplate=('<b>%{y}</b><br>Proporção: %{x:.2f}%'
                       '<br>Contagem: %{customdata[0]:,.0f}'
                       '<br>Denominador: %{customdata[1]:,.0f}<extra></extra>'),
    ))
    fig.update_layout(title=titulo, height=max(310, 142 + 37 * len(d)), showlegend=False)
    fig.update_yaxes(title=eixo_y, type='category', autorange='reversed', automargin=True)
    fig.update_xaxes(title=eixo)
    fixar_percentual(fig)
    mostrar_figura(fig)


def mostrar_cobertura_etapa2(area, intervalo=(2020, 2024)):
    d = cobertura.loc[cobertura.AREA.eq(area) & cobertura.ANO.between(*intervalo)].sort_values('ANO')
    barras_taxa(d.rename(columns={'ANO': 'ROTULO', 'NOTAS_COM_UF_ESCOLA': 'N', 'NOTAS_VALIDAS': 'TOTAL'}),
                f'Em quantas notas sabemos o estado da escola? · {area}', 'Notas válidas com estado da escola (%)')


def mostrar_medias_cobertura(area, intervalo=(2020, 2024), vista='Médias dos grupos'):
    base = situacao_com_cobertura
    d = (base.loc[base.AREA.eq(area) & base.ANO.between(*intervalo)]
         .groupby(['ANO', 'COBERTURA'], as_index=False)[['N', 'SOMA']].sum())
    if d.empty:
        print('Sem notas elegíveis neste recorte.')
        return

    d['MEDIA'] = d.SOMA / d.N
    grupos = list(CORES_COBERTURA)
    com_uf, sem_uf = grupos
    contagens = d.pivot(index='ANO', columns='COBERTURA', values='N').reindex(columns=grupos).fillna(0)
    medias = d.pivot(index='ANO', columns='COBERTURA', values='MEDIA').reindex(columns=grupos)
    diferenca = medias[com_uf] - medias[sem_uf]
    total = contagens.sum(axis=1)
    fig = go.Figure()

    if vista == 'Diferença de médias':
        rotulos = diferenca.map(lambda x: f'{x:+.1f}'.replace('.', ',') if pd.notna(x) else '')
        fig.add_bar(
            x=diferenca.index, y=diferenca,
            marker_color=np.where(diferenca.ge(0), CORES_COBERTURA[com_uf], CORES_COBERTURA[sem_uf]),
            text=rotulos, textposition='outside', cliponaxis=False,
            customdata=np.column_stack([medias[com_uf], medias[sem_uf],
                                        contagens[com_uf], contagens[sem_uf]]),
            hovertemplate=('<b>Edição %{x:.0f}</b><br>Com − sem UF: %{y:+.2f} pontos'
                           '<br>Com UF: %{customdata[0]:.2f} pontos · %{customdata[2]:,.0f} notas'
                           '<br>Sem UF: %{customdata[1]:.2f} pontos · %{customdata[3]:,.0f} notas'
                           '<extra></extra>'),
        )
        fig.update_yaxes(title='Média com UF menos média sem UF (pontos)', zeroline=True, zerolinewidth=2)
        subtitulo = 'Qual foi a diferença entre as médias dos dois grupos?'
    else:
        for rotulo, cor in CORES_COBERTURA.items():
            serie = d.loc[d.COBERTURA.eq(rotulo)]
            total_edicao = serie.ANO.map(total)
            fig.add_scatter(
                x=serie.ANO, y=serie.MEDIA, name=rotulo, mode='lines+markers',
                line=dict(color=cor, width=2.5, dash='solid' if rotulo == com_uf else 'dash'),
                marker=dict(size=8, symbol='circle' if rotulo == com_uf else 'square'),
                customdata=np.column_stack([serie.N, total_edicao, 100 * serie.N / total_edicao]),
                hovertemplate=('<b>Edição %{x:.0f}</b><br>Média: %{y:.2f} pontos'
                               '<br>Notas elegíveis no grupo: %{customdata[0]:,.0f}'
                               '<br>Total nacional da área: %{customdata[1]:,.0f}'
                               '<br>Parcela do total: %{customdata[2]:.1f}%'
                               '<extra>%{fullData.name}</extra>'),
            )
        fig.update_yaxes(title='Nota média (pontos)')
        subtitulo = 'As médias mudam quando o estado da escola é conhecido?'

    fig.update_layout(title=f'{subtitulo} · {area}<br><sup>Brasil · grupos com e sem estado da escola</sup>',
                      height=390, margin=dict(t=85, b=48))
    fig.update_xaxes(title='Ano do ENEM', dtick=1)
    mostrar_figura(fig)

    comparacao = pd.DataFrame({
        'Notas com UF': contagens[com_uf].astype(int),
        'Notas sem UF': contagens[sem_uf].astype(int),
        'Total elegível': total.astype(int),
        'Cobertura (%)': 100 * contagens[com_uf] / total,
        'Média com UF': medias[com_uf],
        'Média sem UF': medias[sem_uf],
        'Diferença (pontos)': diferenca,
    })
    display(comparacao.round(2))


painel(mostrar_cobertura_etapa2, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       intervalo=periodo())
painel(mostrar_medias_cobertura, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       intervalo=periodo(), vista=escolher(['Médias dos grupos', 'Diferença de médias'],
                                          'Médias dos grupos', 'Vista:'))


### Quem se inscreve e quem aparece nas notas?

Os dicionários de 2020–2024 usam o mesmo código para `TP_ST_CONCLUSAO`: **1** já concluiu, **2** cursa e declara concluir no ano, **3** cursa e declara concluir depois e **4** ainda não concluiu e está fora do ensino médio. Somar 2 e 3 dá a parcela que declara estar cursando. Essas são declarações na inscrição. Em 2024 a variável está em `PARTICIPANTES_2024`; UF escolar, rede e notas estão em `RESULTADOS_2024`. O Inep publica as tabelas sem chave comum. Por isso, 2024 entra no painel nacional de inscritos e fica indisponível no cruzamento situação × nota. [Microdados e dicionários do Inep](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) · [Leia-me de 2024, pp. 7–8](fontes/etapa_anterior/Leia_Me_Enem_2024.pdf)

- **Inscrições nacionais:** cada barra compara a mesma situação escolar em uma edição. O denominador inclui os inscritos ausentes da prova.
- **Auditoria nacional de seleção por situação escolar:** selecione a edição, a área e a situação. Cada barra usa o total de notas elegíveis no respectivo grupo de cobertura.
- **Faixas de nota por situação escolar:** selecione a edição, a área, o território e a rede. Cada barra usa o total de notas elegíveis daquela situação no recorte. O painel abre com os elegíveis com UF escolar. A seleção de todos os elegíveis ou de sem UF serve à leitura nacional. Os limites 500 e 600 são referências descritivas em pontos da prova. As contagens continuam na tabela para grupos com menos de 30 observações.

In [ ]:
inscritos = ler('inscritos_situacao_escolar.csv')
brasil_inscritos = (inscritos.groupby(['ANO', 'SITUACAO_ESCOLAR']).N.sum()
                    .unstack(fill_value=0))
salvar(brasil_inscritos.reset_index(), 'inscricoes_nacionais_por_situacao.csv')
CONCLUINTE = 'Concluinte declarado do ano'
FAIXAS_SITUACAO = ['Abaixo de 500', '500 a menos de 600', '600 ou mais']


def mostrar_inscritos_situacao(situacao, intervalo):
    d = brasil_inscritos.loc[intervalo[0]:intervalo[1]]
    quadro = pd.DataFrame({'ROTULO': d.index, 'N': d[situacao], 'TOTAL': d.sum(axis=1)})
    barras_taxa(quadro, f'Quem se inscreveu no ENEM? · {situacao}', 'Inscrições nessa situação (%)')


def mostrar_elegiveis_situacao(ano, area, situacao):
    d = situacao_com_cobertura.query('ANO == @ano and AREA == @area')
    contagens = d.pivot_table(index='COBERTURA', columns='SITUACAO_ESCOLAR', values='N',
                             aggfunc='sum', fill_value=0)
    total = contagens.sum(axis=1)
    quadro = pd.DataFrame({'ROTULO': total.index, 'N': contagens.get(situacao, 0), 'TOTAL': total})
    cores = quadro.ROTULO.map(CORES_COBERTURA).tolist()
    barras_taxa(quadro, f'Quem fez a prova? · {situacao} · {area} · {ano}', 'Notas válidas nessa situação (%)', cores,
                eixo_y='Informação sobre o estado da escola')


def mostrar_situacao(ano, area, territorio, rede, faixa='600 ou mais'):
    d = notas_situacao.query('ANO == @ano and AREA == @area')
    if territorio == 'Brasil (UF informada)':
        d = d[d.UF_ESCOLA.ne('Sem UF escolar')]
    elif territorio != 'Brasil (todos os elegíveis)':
        d = d[d.UF_ESCOLA.eq(territorio)]
    if rede != 'Todas':
        d = d[d.REDE.eq(rede)]
    if d.empty:
        print('Sem notas para este recorte.')
        return
    quadro = d.pivot_table(index='SITUACAO_ESCOLAR', columns='FAIXA_NOTA', values='N',
                           aggfunc='sum', fill_value=0).reindex(columns=FAIXAS_SITUACAO, fill_value=0)
    total = quadro.sum(axis=1)
    grupo = pd.DataFrame({'ROTULO': total.index, 'N': quadro[faixa], 'TOTAL': total})
    barras_taxa(grupo.loc[grupo.TOTAL.ge(30)],
                f'Quem ficou na faixa {faixa.lower()}? · {area} · {ano}<br><sup>{territorio} · {rede}</sup>',
                'Notas na faixa escolhida (%)', '#007763', eixo_y='Situação escolar')
    faixas = 100 * quadro.div(total, axis=0)
    faixas.loc[total.lt(30)] = np.nan
    media = d.groupby('SITUACAO_ESCOLAR').SOMA.sum() / total
    display(faixas.round(1).assign(N=total, MEDIA=media.where(total.ge(30)).round(1)))


painel(mostrar_inscritos_situacao,
       situacao=escolher(list(brasil_inscritos.columns), CONCLUINTE, 'Situação:'),
       intervalo=periodo())
painel(mostrar_elegiveis_situacao,
       ano=escolher(ANOS_ESTUDO[:-1], 2023, 'Ano:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       situacao=escolher(list(brasil_inscritos.columns), CONCLUINTE, 'Situação:'))
painel(mostrar_situacao,
       ano=escolher(ANOS_ESTUDO[:-1], 2023, 'Ano:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       territorio=escolher(['Brasil (todos os elegíveis)', TERRITORIOS[0], 'Sem UF escolar', *UFS_ESTUDO],
                          TERRITORIOS[0], 'Território:'),
       rede=escolher([r for r in REDES_ESTUDO if r != 'Públicas'], 'Todas', 'Rede:'),
       faixa=escolher(FAIXAS_SITUACAO, '600 ou mais', 'Faixa:'))

### O sorteio preserva a composição observável?

Reexecutamos a seleção a partir dos CSVs oficiais. A auditoria confere contagens por área, cobertura, mediana estadual do recorte geral, contagens por rede/localização, o multiconjunto de notas sorteadas com seu contexto e a amostra conjunta. Notas repetidas são comparadas pela frequência. Os identificadores servem para verificar unicidade durante a extração e ficam fora das tabelas exportadas.

Comparamos a participação de cada rede e localização nas notas amostradas e nas notas completas **dentro de cada ano × UF × área**. A diferença é medida em pontos percentuais. A fração de 10% é definida por estrato territorial e área. Redes e localização são verificações posteriores. Em células pequenas, o arredondamento e o sorteio afetam mais as proporções.

In [ ]:
auditoria_bruta = ler('auditoria_reprocessamento_enem.csv')
chaves_descritivas = ['ANO', 'UF_ESCOLA', 'AREA']
chaves_composicao = chaves_descritivas + ['REDE', 'LOCALIZACAO']
amostral = amostra.groupby(chaves_composicao, observed=True).size().rename('N_AMOSTRA').reset_index()
qualidade_sorteio = enem_redes[chaves_composicao + ['N']].merge(
    amostral, on=chaves_composicao, how='outer', validate='one_to_one')
assert qualidade_sorteio.N.notna().all()
qualidade_sorteio.N_AMOSTRA = qualidade_sorteio.N_AMOSTRA.fillna(0).astype(int)
for coluna in ['N', 'N_AMOSTRA']:
    qualidade_sorteio['PCT_' + coluna] = 100 * qualidade_sorteio[coluna] / (
        qualidade_sorteio.groupby(chaves_descritivas)[coluna].transform('sum'))
qualidade_sorteio['DIF_PP'] = qualidade_sorteio.PCT_N_AMOSTRA - qualidade_sorteio.PCT_N
salvar(qualidade_sorteio, 'auditoria_composicao_amostral.csv')
display(auditoria_bruta.groupby('ANO')[['ELEGIVEIS_QUATRO_PROVAS_COM_UF', 'AMOSTRA_CONJUNTA']].first())
resumo_sorteio = qualidade_sorteio.assign(DIF_ABS=lambda d: d.DIF_PP.abs()).groupby('ANO').agg(
    MEDIANA_ERRO_PP=('DIF_ABS', 'median'), P95_ERRO_PP=('DIF_ABS', lambda s: s.quantile(.95)),
    MAX_ERRO_PP=('DIF_ABS', 'max'))
display(resumo_sorteio.round(2))
salvar(resumo_sorteio.reset_index(), 'resumo_composicao_amostral.csv')
print(f'{len(amostra):,} notas em {len(amostral[chaves_descritivas].drop_duplicates())} estratos. '
      f'A amostra conjunta reúne {len(perfis):,} inscrições com quatro notas.')
print(f'Na comparação rede × localização dentro dos estratos, a diferença absoluta mediana '
      f'ficou entre {resumo_sorteio.MEDIANA_ERRO_PP.min():.2f} e {resumo_sorteio.MEDIANA_ERRO_PP.max():.2f} p.p. '
      f'por edição. O maior desvio observado foi {resumo_sorteio.MAX_ERRO_PP.max():.2f} p.p.')


,ELEGIVEIS_QUATRO_PROVAS_COM_UF,AMOSTRA_CONJUNTA
ANO,,
2020,537909,53790
2021,602451,60243
2022,692915,69291
2023,721429,72142
2024,1193432,119343


,MEDIANA_ERRO_PP,P95_ERRO_PP,MAX_ERRO_PP
ANO,,,
2020,0.19,1.79,4.63
2021,0.16,1.71,5.58
2022,0.18,1.86,6.40
2023,0.17,1.76,4.98
2024,0.14,1.27,3.12


1,910,983 notas em 675 estratos. A amostra conjunta reúne 374,809 inscrições com quatro notas.
Na comparação rede × localização dentro dos estratos, a diferença absoluta mediana ficou entre 0.14 e 0.19 p.p. por edição. O maior desvio observado foi 6.40 p.p.


## 2. Distribuições, território e mudanças no tempo

### O que a distribuição mostra além da mediana?

**Pergunta:** o centro resume bem as notas do recorte ou esconde grupos distintos? O histograma usa a amostra por área da etapa 1, agora com contexto da escola. Mude ano, território, rede e largura das faixas. As linhas mostram média e mediana; a faixa sombreada vai de Q1 a Q3. O eixo de frequência começa em zero e se ajusta à maior barra do recorte.

Nas provas objetivas, a segunda figura desenha três curvas suavizadas sobre as notas positivas. Usamos até 4.000 notas, com semente fixa. Aumentar a largura espalha a contribuição de cada nota por uma vizinhança maior. Isso reduz oscilações, mas também pode esconder dois grupos reais: um pico desaparecer não prova que ele era falso. A quantidade de picos é apenas um diagnóstico exploratório, não um teste conclusivo de bimodalidade. Notas zero permanecem no histograma e nas medidas; são separadas somente nas curvas.

Redação tem notas discretas: seus degraus podem criar picos artificiais numa curva contínua. Nesse caso, mostramos as frequências das notas. A mediana continua sendo o centro por ordenação; **normalidade não é um requisito para usar a média nem um critério automático para escolher a mediana**. Havendo grupos distintos, devemos descrevê-los também.

**Como ler a legenda:** o número é a quantidade de elevações destacadas por um critério de altura relativa, não uma contagem definitiva de grupos de alunos. Em **Vista → Sensibilidade da densidade**, a largura é um fator multiplicado pelo desvio-padrão das notas da curva; não cria notas novas. O critério exige proeminência de 8% da altura máxima e distância de 15 posições na grade. A escolha desses valores afeta a contagem. Não identificamos perfis de alunos apenas por esse desenho.

In [ ]:
# @title
def mostrar_forma(ano, area, territorio, rede, largura, vista):
    d = recortar(amostra, territorio, rede)
    notas = d.loc[d.ANO.eq(ano) & d.AREA.eq(area), 'NOTA'].dropna().to_numpy()
    n = len(notas)
    if n < 30:
        print(f'Recorte com {n} notas. Amplie a seleção para examinar a distribuição.')
        return

    q1, mediana, q3 = np.quantile(notas, [.25, .5, .75])
    media = notas.mean()
    contexto = f'{territorio} · {ano} · {area} · {rede} · n={n:,}'
    fig = go.Figure()
    if vista == 'Frequência por faixa':
        bordas = np.arange(0, max(1000, notas.max()) + largura, largura)
        contagem, _ = np.histogram(notas, bins=bordas)
        fig.add_bar(x=(bordas[:-1] + bordas[1:]) / 2, y=100 * contagem / n,
                    width=.94 * largura, marker_color='#0056B3',
                    customdata=np.column_stack([bordas[:-1], bordas[1:], contagem]),
                    hovertemplate='Faixa: %{customdata[0]:.0f} a %{customdata[1]:.0f}'
                                  '<br>%{y:.2f}% · %{customdata[2]:,.0f} notas<extra></extra>')
        fig.add_vrect(x0=q1, x1=q3, fillcolor='#F5C66D', opacity=.19, layer='below', line_width=0)
        fig.add_vline(x=media, line_dash='dash', line_color='#C34109')
        fig.add_vline(x=mediana, line_color='#243747')
        titulo = 'Em quais faixas as notas se concentram?'
        eixo_y = 'Notas da amostra em cada faixa (%)'
    elif area == 'Redação':
        valores, contagem = np.unique(notas, return_counts=True)
        fig.add_bar(x=valores, y=100 * contagem / n, width=13,
                    marker_color='#007763', customdata=contagem,
                    hovertemplate='Nota: %{x:.0f}<br>%{y:.2f}% · %{customdata:,} notas<extra></extra>')
        titulo = 'Quais notas de Redação aparecem mais?'
        eixo_y = 'Parcela da amostra com cada nota (%)'
    else:
        positivas = notas[notas > 0]
        if len(positivas) > 4000:
            positivas = np.random.default_rng(42).choice(positivas, 4000, replace=False)
        if len(positivas) < 3 or np.ptp(positivas) == 0:
            print('Amostra de notas positivas sem variação suficiente para a densidade.')
            return
        eixo = np.linspace(*np.quantile(positivas, [.005, .995]), 300)
        for largura_kde, cor in zip([.15, .25, .35], ['#0056B3', '#007763', '#C34109']):
            densidade = gaussian_kde(positivas, bw_method=largura_kde)(eixo)
            picos, _ = find_peaks(densidade, prominence=.08 * densidade.max(), distance=15)
            fig.add_scatter(x=eixo, y=densidade, mode='lines', line=dict(color=cor, width=2.5),
                            name=f'Largura {largura_kde:.2f} · {len(picos)} destaque(s)',
                            hovertemplate='Nota: %{x:.1f}<br>Densidade: %{y:.4f}')
        titulo = f'Onde as notas acima de zero se concentram?'
        eixo_y = 'Concentração de notas (densidade)'
    fig.update_xaxes(title='Nota na prova (pontos)')
    fig.update_yaxes(title=eixo_y, rangemode='tozero')
    if eixo_y != 'Concentração de notas (densidade)':
        fig.update_yaxes(ticksuffix='%')
    mostrar_figura(configurar_painel(fig, titulo, contexto))
    print(f'Média: {media:.1f} (tracejada no histograma) · mediana: {mediana:.1f} (linha sólida) '
          f'· Q1–Q3: {q1:.1f} a {q3:.1f} · zeros: {100 * (notas == 0).mean():.2f}%.')
painel(mostrar_forma, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
    rede=escolher(REDES_ESTUDO, 'Todas', 'Rede:'),
    largura=escolher([20, 40, 80], 40, 'Faixa:'),
    vista=escolher(['Frequência por faixa', 'Sensibilidade da densidade'], 'Frequência por faixa', 'Vista:'))

### Uma mediana parecida pode esconder duas distribuições bem diferentes

Use **Entre regiões** para comparar as regiões, **Entre UFs da região** para comparar os estados da região selecionada ou **Entre edições da região** para acompanhar a distribuição de 2020 a 2024. A área e a rede valem para toda comparação. O filtro de edição vale para as comparações territoriais. Cada região é resumida a partir das notas individuais reunidas de suas UFs. Nesse recorte, uma UF com mais notas tem mais peso no resultado regional. O seletor **Vista** alterna entre as figuras e as tabelas. **Todas as vistas** exibe as figuras em sequência.

- **Distribuição acumulada.** Em cada nota do eixo horizontal, a curva informa a parcela da amostra com nota menor ou igual a ela. Curvas que se cruzam indicam que a comparação depende de onde se olha na distribuição.
- **Centro e dispersão.** O ponto marca a mediana. O traço grosso liga P25 e P75 e mostra a dispersão no centro. O traço fino liga P10 e P90 e mostra a distância entre as posições mais baixas e mais altas. Empates, sobretudo em Redação, podem fazer mais de uma dessas posições coincidir.
- **Caudas em gráficos separados.** Usamos os cortes P20 e P80 da amostra nacional da área e da edição de referência, com todas as redes e UF escolar informada. A cauda inferior tem notas estritamente abaixo de P20 e a superior, estritamente acima de P80. Os empates no corte ficam no meio. São referências descritivas comuns aos grupos, sem equivalência com níveis de proficiência. Em comparações entre edições, os cortes ficam fixos. As duas caudas usam o mesmo eixo horizontal, partindo de zero até um limite ajustado à maior proporção observada.
- **Contribuição à média.** Para cada faixa, a tabela divide a soma dos desvios das notas em relação à média de referência pelo total de notas do grupo. As três contribuições somam a diferença entre a média do grupo e a média de referência. Assim é possível distinguir quantas notas estão nas caudas e quanto elas deslocam a média. Se mudarem apenas as notas das posições extremas, preservada a ordem, as notas das posições centrais permanecem iguais e a mediana também.

As comparações descrevem a amostra de elegíveis com UF escolar informada. Grupos com menos de 30 notas ficam fora dos gráficos. Os intervalos de percentis descrevem dispersão. Diferenças entre territórios e edições também podem refletir cobertura, composição e características da prova.

**Exemplo.** Selecione Redação de 2023, todas as redes e **Entre UFs da região → Centro-Oeste**. Na amostra, DF (1.694 notas) e GO (2.855 notas) têm mediana 660. O P10 é 480 no DF e 400 em GO. No outro extremo, o P90 é 900 no DF e 920 em GO. Nesse recorte, a mesma mediana convive com uma cauda inferior mais baixa e uma cauda superior mais alta em GO.


In [ ]:
def mostrar_distribuicoes_territoriais(comparacao, regiao, ano, area, rede, ano_base, vista='Todas as vistas'):
    por_edicao = comparacao == 'Entre edições da região'
    filtro = amostra.AREA.eq(area)
    if not por_edicao:
        filtro &= amostra.ANO.eq(ano)
    d = recortar(amostra.loc[filtro, ['ANO', 'UF_ESCOLA', 'REGIAO', 'REDE', 'NOTA']],
                 'Brasil (UF informada)' if comparacao == 'Entre regiões' else regiao, rede)
    coluna = {'Entre regiões': 'REGIAO', 'Entre UFs da região': 'UF_ESCOLA',
              'Entre edições da região': 'ANO'}[comparacao]
    d = d[np.isfinite(d.NOTA)]
    grupos = list(d.groupby(coluna, sort=True, observed=True).NOTA)
    elegiveis = [(str(nome), notas.to_numpy()) for nome, notas in grupos if len(notas) >= 30]
    if not elegiveis:
        print('Nenhum grupo atingiu 30 notas. Ajuste a rede, a região ou a edição.')
        return

    ref = amostra.loc[amostra.AREA.eq(area) & amostra.ANO.eq(ano_base), 'NOTA'].dropna()
    if ref.empty:
        print('A amostra de referência está vazia. Escolha outra edição ou área.')
        return
    limite_inf, limite_sup = np.quantile(ref, [.2, .8])
    media_ref = ref.mean()
    territorio = 'Brasil (UF escolar informada)' if comparacao == 'Entre regiões' else regiao
    periodo_titulo = f'{min(ANOS_ESTUDO)}–{max(ANOS_ESTUDO)}' if por_edicao else str(ano)
    contexto = f'{area} · {periodo_titulo} · {territorio} · {rede}'
    cores_grupo = PALETA_SERIES
    curvas = go.Figure()
    linhas = []
    for i, (nome, x) in enumerate(elegiveis):
        valores, contagens = np.unique(x, return_counts=True)
        acumuladas = contagens.cumsum()
        curvas.add_scatter(x=valores, y=acumuladas / len(x), mode='lines', name=nome,
                           line=dict(shape='hv', color=cores_grupo[i % len(cores_grupo)], width=2),
                           customdata=acumuladas,
                           hovertemplate=f'<b>{nome}</b><br>Nota ≤ %{{x:.1f}}<br>'
                                         f'%{{y:.1%}} · %{{customdata:,d}} de {len(x):,} notas<extra></extra>')
        p10, p25, mediana, p75, p90 = np.quantile(x, [.1, .25, .5, .75, .9])
        abaixo = x < limite_inf
        acima = x > limite_sup
        meio = ~(abaixo | acima)
        linhas.append({'Grupo': nome, 'N': len(x), 'Média': x.mean(), 'P10': p10, 'P25': p25,
                       'Mediana': mediana, 'P75': p75, 'P90': p90, 'AIQ': p75 - p25,
                       'N abaixo': abaixo.sum(), 'N acima': acima.sum(),
                       'Cauda inferior (pontos)': (x[abaixo] - media_ref).sum() / len(x),
                       'Centro (pontos)': (x[meio] - media_ref).sum() / len(x),
                       'Cauda superior (pontos)': (x[acima] - media_ref).sum() / len(x),
                       'Média − referência (pontos)': x.mean() - media_ref})
    resumo = pd.DataFrame(linhas)
    ordem = resumo.Grupo.tolist()
    titulo_curvas = f'Que parcela tirou até cada nota?<br><sup>{contexto}</sup>'
    curvas.update_layout(title=titulo_curvas, xaxis_title='Nota na prova (pontos)', yaxis_title='Notas iguais ou abaixo desse valor (%)',
                         hovermode='closest',
                         legend_title_text=coluna.replace('UF_ESCOLA', 'UF').title())
    curvas.update_yaxes(range=[0, 1.02], tickformat='.0%')
    if vista in ['Curva acumulada', 'Todas as vistas']:
        mostrar_figura(curvas)

    intervalos = go.Figure()
    for baixo, alto, nome, cor, espessura in [
        ('P10', 'P90', 'P10 a P90', '#6083AA', 3),
        ('P25', 'P75', 'P25 a P75', '#00634E', 9),
    ]:
        for i, linha in resumo.iterrows():
            intervalos.add_scatter(x=[linha[baixo], linha[alto]], y=[linha.Grupo, linha.Grupo],
                                   mode='lines+markers', line=dict(color=cor, width=espessura),
                                   marker=dict(size=6, color=cor), name=nome, legendgroup=nome,
                                   showlegend=i == 0,
                                   hovertext=[f'{baixo} · {linha[baixo]:.1f} pontos',
                                              f'{alto} · {linha[alto]:.1f} pontos'],
                                   hovertemplate='<b>%{y}</b><br>%{hovertext}<extra></extra>')
    intervalos.add_scatter(x=resumo.Mediana, y=resumo.Grupo, mode='markers', name='Mediana',
                           marker=dict(size=11, color='#243747', symbol='diamond'),
                           customdata=resumo[['N', 'Média', 'P10', 'P25', 'P75', 'P90', 'AIQ']],
                           hovertemplate='<b>%{y}</b><br>Mediana: %{x:.1f} · Média: %{customdata[1]:.1f}'
                                         '<br>P10: %{customdata[2]:.1f} · P25: %{customdata[3]:.1f}'
                                         '<br>P75: %{customdata[4]:.1f} · P90: %{customdata[5]:.1f}'
                                         '<br>AIQ: %{customdata[6]:.1f} · N: %{customdata[0]:,.0f}<extra></extra>')
    intervalos.update_layout(title=f'Onde fica o meio e quanto as notas variam?<br><sup>{contexto}</sup>',
                             xaxis_title='Nota na prova (pontos)', yaxis=dict(title='Grupo comparado', categoryorder='array',
                                 categoryarray=ordem, autorange='reversed'),
                             height=max(360, 42 * len(ordem) + 160))
    if vista in ['Centro e dispersão', 'Todas as vistas']:
        mostrar_figura(intervalos)

    caudas = resumo[['N abaixo', 'N acima']].div(resumo.N, axis=0)
    limite_caudas = max(.05, min(1, 1.15 * caudas.to_numpy().max()))
    for contagem, nome, detalhe, cor in [
        ('N abaixo', 'Cauda inferior', f'Nota < {limite_inf:.2f}', '#B93217'),
        ('N acima', 'Cauda superior', f'Nota > {limite_sup:.2f}', '#00634E'),
    ]:
        fig = go.Figure(go.Bar(x=caudas[contagem], y=resumo.Grupo, orientation='h', marker_color=cor,
                               texttemplate='%{x:.1%}',
                               textposition=['inside' if v >= .93 else 'outside' for v in caudas[contagem]],
                               insidetextfont=dict(color='white'), cliponaxis=False,
                               customdata=resumo[[contagem, 'N']],
                               hovertemplate='<b>%{y}</b><br>%{x:.1%}<br>'
                                             '%{customdata[0]:,.0f} de %{customdata[1]:,.0f} notas<extra></extra>'))
        fig.update_layout(title=f'Notas {"abaixo" if contagem == "N abaixo" else "acima"} do corte · {detalhe}<br><sup>{contexto} · corte de {ano_base}</sup>',
                          xaxis_title='Notas do grupo além do corte (%)',
                          yaxis=dict(title='Grupo comparado', categoryorder='array', categoryarray=ordem, autorange='reversed'),
                          margin=dict(r=78), height=max(320, 38 * len(ordem) + 145))
        fig.update_xaxes(range=[0, limite_caudas], tickformat='.0%')
        if vista in [nome, 'Todas as vistas']:
            mostrar_figura(fig)

    resumo['% abaixo'] = 100 * caudas['N abaixo']
    resumo['% acima'] = 100 * caudas['N acima']
    display(Markdown(f'**Referência nacional da amostra em {ano_base}** · {len(ref):,} notas · '
                     f'Média {media_ref:.1f} · P20 {limite_inf:.2f} · P80 {limite_sup:.2f}'))
    resumo = resumo.round(2).set_index('Grupo')
    if vista in ['Tabelas e contribuição', 'Todas as vistas']:
        display(resumo[['N', 'Média', 'Mediana', 'P10', 'P90', 'AIQ', 'N abaixo', '% abaixo', 'N acima', '% acima']])
        display(resumo[['Cauda inferior (pontos)', 'Centro (pontos)', 'Cauda superior (pontos)',
                        'Média − referência (pontos)']])
    omitidos = [f'{nome} (N={len(notas)})' for nome, notas in grupos if len(notas) < 30]
    if omitidos:
        print('Grupos abaixo de 30 notas: ' + ', '.join(omitidos))
    if len(resumo) > 1:
        ordenado = resumo.sort_values('Mediana').reset_index()
        pos = int(ordenado.Mediana.diff().idxmin())
        a, b = ordenado.iloc[pos - 1], ordenado.iloc[pos]
        display(Markdown(f'**As medianas mais próximas no recorte** · {a.Grupo} tem mediana '
                         f'{a.Mediana:.1f} e P10 {a.P10:.1f}. {b.Grupo} tem mediana '
                         f'{b.Mediana:.1f} e P10 {b.P10:.1f}. A diferença no P10 é de '
                         f'{abs(b.P10 - a.P10):.1f} pontos, com {abs(b.Mediana - a.Mediana):.1f} na mediana.'))


painel(mostrar_distribuicoes_territoriais,
       comparacao=escolher(['Entre regiões', 'Entre UFs da região', 'Entre edições da região'],
                           'Entre regiões', 'Comparação:'),
       regiao=escolher(REGIOES_ESTUDO, 'Nordeste', 'Região:'),
       ano=escolher(ANOS_ESTUDO, 2024, 'Edição:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       rede=escolher(REDES_ESTUDO, 'Todas', 'Rede:'),
       ano_base=escolher(ANOS_ESTUDO, 2024, 'Corte de:'),
       vista=escolher(['Centro e dispersão', 'Curva acumulada', 'Cauda inferior', 'Cauda superior',
                       'Tabelas e contribuição', 'Todas as vistas'], 'Centro e dispersão', 'Vista:'))

### Visão geral das cinco áreas na amostra com UF escolar

O mapa de calor mostra a parcela de notas na cauda escolhida. Na vista **Regiões**, as linhas comparam as cinco regiões no ano selecionado. Na vista **Edições**, as linhas comparam 2020–2024 no território selecionado. As cinco áreas usam uma legenda de cor comum, ajustada ao menor e ao maior percentual exibido. Ao comparar recortes de filtros diferentes, leia os números nas células.

Cada área tem seu próprio corte, calculado na **amostra nacional com UF escolar informada**, com todas as redes no ano de referência. A cauda inferior reúne notas estritamente abaixo do P20 e a superior, notas estritamente acima do P80. As notas iguais ao corte ficam fora da cauda. Na vista Edições, o corte da área permanece fixo. Assim comparamos a frequência em faixas de referência próprias de cada área. As diferenças ao longo do tempo podem refletir características da prova, cobertura e composição do público.

O denominador de cada célula é o número de notas amostrais válidas da área, no recorte exibido. O cursor mostra o corte em pontos, a contagem na cauda e o denominador. Grupos com menos de 30 notas ficam em branco.

In [ ]:
def mostrar_visao_geral_caudas(vista, cauda, ano_base, territorio, rede):
    por_regiao = vista == 'Regiões'
    inferior = cauda == 'Inferior (abaixo do P20)'
    referencia = amostra.loc[amostra.ANO.eq(ano_base) & np.isfinite(amostra.NOTA)]
    cortes = referencia.groupby('AREA', observed=True).NOTA.quantile(.2 if inferior else .8)
    d = recortar(amostra, 'Brasil (UF informada)' if por_regiao else territorio, rede)
    if por_regiao:
        d = d[d.ANO.eq(ano_base)]
    d = d.loc[np.isfinite(d.NOTA), ['REGIAO', 'ANO', 'AREA', 'NOTA']].copy()
    d['CORTE'] = d.AREA.map(cortes).astype(float)
    d['NA_CAUDA'] = d.NOTA.lt(d.CORTE) if inferior else d.NOTA.gt(d.CORTE)
    grupo = 'REGIAO' if por_regiao else 'ANO'
    grupos = REGIOES_ESTUDO if por_regiao else ANOS_ESTUDO
    r = d.groupby([grupo, 'AREA'], observed=True).NA_CAUDA.agg(CONTAGEM='sum', N='size').reset_index()
    r['PCT'] = 100 * r.CONTAGEM / r.N.where(r.N.ge(30))

    def matriz(coluna):
        return r.pivot(index=grupo, columns='AREA', values=coluna).reindex(index=grupos, columns=AREAS_ESTUDO)

    porcentagens = matriz('PCT').to_numpy(float)
    if not np.isfinite(porcentagens).any():
        print('Nenhum grupo atingiu 30 notas válidas neste recorte.')
        return
    cortes_por_area = np.broadcast_to(cortes.reindex(AREAS_ESTUDO).to_numpy(float), porcentagens.shape)
    minimo, maximo = limites_cores_percentuais(porcentagens)
    fig = go.Figure(go.Heatmap(
        x=AREAS_ESTUDO, y=list(map(str, grupos)), z=porcentagens,
        customdata=np.stack([matriz('CONTAGEM').to_numpy(float), matriz('N').to_numpy(float), cortes_por_area], axis=-1),
        text=[[f'{v:.1f}%'.replace('.', ',') if np.isfinite(v) else '' for v in linha] for linha in porcentagens],
        texttemplate='%{text}', textfont=dict(size=13),
        colorscale=ESCALA_SEQUENCIAL, zmin=minimo, zmax=maximo,
        xgap=2, ygap=2, hoverongaps=False,
        colorbar=dict(title='Notas além do corte', ticksuffix='%', nticks=6),
        hovertemplate=('<b>%{y} · %{x}</b><br>Na cauda: %{z:.1f}%'
                       '<br>%{customdata[0]:,.0f} de %{customdata[1]:,.0f} notas'
                       '<br>Corte: %{customdata[2]:.1f} pontos<extra></extra>'),
    ))
    fig.update_xaxes(title='Área da prova')
    fig.update_yaxes(autorange='reversed', type='category', title='Região' if por_regiao else 'Ano do ENEM')
    recorte = f'Regiões em {ano_base}' if por_regiao else f'{territorio} · 2020–2024'
    mostrar_figura(configurar_painel(fig, 'Notas na parte mais baixa da distribuição' if inferior else 'Notas na parte mais alta da distribuição',
                                   f'{recorte} · {rede} · referência de {ano_base}', altura=410))


painel(mostrar_visao_geral_caudas,
       vista=escolher(['Regiões', 'Edições'], 'Regiões', 'Comparar:'),
       cauda=escolher(['Inferior (abaixo do P20)', 'Superior (acima do P80)'],
                      'Inferior (abaixo do P20)', 'Cauda:'),
       ano_base=escolher(ANOS_ESTUDO, 2024, 'Ano de corte:'),
       territorio=escolher(TERRITORIOS, TERRITORIOS[0], 'Nas edições:'),
       rede=escolher(REDES_ESTUDO, 'Todas', 'Rede:'))

### Trajetórias por UF e região

**Pergunta:** a queda está em todas as áreas e todos os anos? O painel mostra a trajetória e a mudança em pontos entre anos consecutivos. A variação de 2020 fica ausente porque 2019 não está no período; isso não equivale a crescimento zero. O intervalo Q1–Q3, disponível nos dados, descreve dispersão e não incerteza da mediana.
As medianas abaixo usam todos os elegíveis com UF escolar. O resumo regional foi calculado reunindo suas notas. A edição de 2020 foi aplicada em 2021: `ANO` é o ano da edição. A série acompanha grupos diferentes, não os mesmos alunos.

In [ ]:
# @title
def mostrar_trajetoria(area, nivel, regiao, intervalo, medida, vista):
    d = enem_regiao if nivel == 'Regiões' else enem_uf
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    d = d[d.AREA.eq(area)].sort_values([grupo, 'ANO'])
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)].copy()
    d['VARIACAO'] = d.groupby(grupo)[medida].diff()
    d = d[d.ANO.between(*intervalo)]
    coluna = medida if vista == 'Nível das notas' else 'VARIACAO'
    d = d.dropna(subset=[coluna])
    if d.empty:
        print('Selecione um período que inclua pelo menos uma edição com variação calculada.')
        return
    nome = 'Mediana' if medida == 'MEDIANA' else 'Média'
    eixo = ('Nota do meio (mediana, em pontos)' if medida == 'MEDIANA' else 'Nota média (pontos)') if coluna == medida else 'Mudança desde o ano anterior (pontos)'
    titulo = 'Como as notas mudaram ao longo do tempo?' if coluna == medida else 'Quanto a nota mudou de um ano para o outro?'
    fig = px.line(d, x='ANO', y=coluna, color=grupo, markers=True,
                  labels={'ANO': 'Ano do ENEM', coluna: eixo, grupo: nivel, 'N': 'Notas elegíveis'},
                  hover_data={'N': ':,', 'Q1': ':.1f', 'Q3': ':.1f', coluna: ':+.1f' if coluna == 'VARIACAO' else ':.1f'},
                  color_discrete_map=CORES_REGIOES if nivel == 'Regiões' else None)
    if coluna == 'VARIACAO':
        fig.add_hline(y=0, line_color='#6A7D92', line_dash='dot')
    fig.update_xaxes(tickvals=list(range(intervalo[0], intervalo[1] + 1)))
    mostrar_figura(configurar_painel(fig, titulo, f'{area} · {nome} · {nivel} · {regiao}'))

painel(mostrar_trajetoria, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'), intervalo=periodo(),
    medida=escolher([('Mediana', 'MEDIANA'), ('Média', 'MEDIA')], 'MEDIANA', 'Medida:'),
    vista=escolher(['Nível das notas', 'Mudança anual'], 'Nível das notas', 'Vista:'))

### Mudanças entre edições

Compare duas edições, por UF ou região, usando o conjunto de notas elegíveis com UF escolar informada. Passe o mouse para ver os valores e as contagens dos dois anos. A cor azul mostra aumento e a laranja, redução. A intensidade da cor se ajusta ao recorte, por isso compare as diferenças em pontos ao trocar os filtros.

O mapa reúne as cinco áreas. Escolha uma área para ler as mudanças diretamente em barras que partem de zero. Alterne entre a **mediana**, os percentis **P5** e **P95** e os quartis **Q1** e **Q3**. Por exemplo, se o P5 cair mais que a mediana, a parte inferior da distribuição mudou mais nesse ponto de referência. Isso descreve posições na distribuição de cada edição, com grupos de participantes que podem ter mudado.

In [ ]:
# @title
MEDIDAS_MUDANCA = {'Mediana': 'MEDIANA', 'P5': 'P5', 'Q1': 'Q1', 'Q3': 'Q3', 'P95': 'P95'}


def mostrar_mudancas(inicio, fim, nivel, regiao, area='Todas as áreas', medida='Mediana'):
    if inicio >= fim:
        print('Escolha um ano final posterior ao inicial.')
        return
    d = enem_regiao if nivel == 'Regiões' else enem_uf
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    metrica = MEDIDAS_MUDANCA[medida]
    d = d[d.ANO.isin([inicio, fim])]
    m = d.pivot(index=[grupo, 'AREA'], columns='ANO', values=metrica)
    n = d.pivot(index=[grupo, 'AREA'], columns='ANO', values='N')
    delta = m[fim] - m[inicio]
    nomes = {'Mediana': 'nota do meio (mediana)', 'P5': 'nota no percentil 5',
             'Q1': 'nota no percentil 25', 'Q3': 'nota no percentil 75', 'P95': 'nota no percentil 95'}
    titulo = f'Quanto mudou a {nomes[medida]} de {inicio} para {fim}?'
    tooltip = (f'<br>{inicio}: %{{customdata[0]:.1f}} pontos · N=%{{customdata[2]:,.0f}}'
               f'<br>{fim}: %{{customdata[1]:.1f}} pontos · N=%{{customdata[3]:,.0f}}')
    if area == 'Todas as áreas':
        matriz = delta.unstack('AREA').reindex(columns=AREAS_ESTUDO)
        limite = max(1, np.nanmax(abs(matriz.to_numpy())))
        matrizes = [x.unstack('AREA').reindex(index=matriz.index, columns=matriz.columns).to_numpy()
                    for x in (m[inicio], m[fim], n[inicio], n[fim])]
        fig = go.Figure(go.Heatmap(
            x=matriz.columns, y=matriz.index, z=matriz.to_numpy(),
            text=[[f'{v:+.1f}'.replace('.', ',') if np.isfinite(v) else '—'
                   for v in linha] for linha in matriz.to_numpy()],
            texttemplate='%{text}', customdata=np.stack(matrizes, axis=-1),
            colorscale=ESCALA_DIVERGENTE,
            zmin=-limite, zmax=limite, xgap=2, ygap=2,
            colorbar=dict(title='Mudança (pontos)'),
            hovertemplate=('<b>%{y} · %{x}</b><br>Variação: %{z:+.1f} pontos'
                           + tooltip + '<extra></extra>'),
        ))
        fig.update_xaxes(title='Área da prova')
        if medida == 'Mediana':
            salvar(matriz.reset_index(), 'mudancas_areas.csv')
    else:
        recorte = delta.xs(area, level='AREA').sort_values()
        linhas = m.xs(area, level='AREA').loc[recorte.index]
        volumes = n.xs(area, level='AREA').loc[recorte.index]
        fig = go.Figure(go.Bar(
            x=recorte, y=recorte.index, orientation='h',
            marker_color=np.where(recorte.ge(0), '#1554BD', '#B93217'),
            customdata=np.column_stack([linhas[inicio], linhas[fim], volumes[inicio], volumes[fim]]),
            text=[f'{v:+.1f}' for v in recorte], textposition='outside', cliponaxis=False,
            hovertemplate=('<b>%{y}</b><br>Variação: %{x:+.1f} pontos'
                           + tooltip + '<extra></extra>'),
        ))
        fig.add_vline(x=0, line_color='#7A8792', line_width=1)
        fig.update_xaxes(title='Nota do ano final menos nota do ano inicial (pontos)', zeroline=True)
        titulo += f' · {area}'
    fig.update_layout(title=titulo, height=max(350, 155 + 29 * d[grupo].nunique()))
    fig.update_yaxes(title='Região da escola' if nivel == 'Regiões' else 'Estado da escola', autorange='reversed', automargin=True)
    mostrar_figura(fig)


painel(mostrar_mudancas,
       inicio=escolher(ANOS_ESTUDO, 2023, 'De:'),
       fim=escolher(ANOS_ESTUDO, 2024, 'Até:'),
       nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
       regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'),
       area=escolher(['Todas as áreas'] + AREAS_ESTUDO, 'Todas as áreas', 'Área:'),
       medida=escolher(list(MEDIDAS_MUDANCA), 'Mediana', 'Medida:'))

In [ ]:
referencia = amostra.query('ANO == 2024 and AREA == "Matemática"').NOTA
mud_regional = (enem_regiao[enem_regiao.ANO.isin([2023, 2024])]
                .pivot(index=['REGIAO', 'AREA'], columns='ANO', values='MEDIANA'))
delta_regional = mud_regional[2024] - mud_regional[2023]
regiao_queda, area_queda = delta_regional.idxmin()
display(Markdown(
    f'Em Matemática 2024, as **{len(referencia):,} notas amostradas** com UF escolar têm média '
    f'{referencia.mean():.1f}, mediana {referencia.median():.1f} e Q1–Q3 '
    f'{referencia.quantile(.25):.1f} a {referencia.quantile(.75):.1f}. Na série completa, '
    f'{int(delta_regional.lt(0).sum())} das {len(delta_regional)} combinações região × área '
    f'caíram de 2023 para 2024. A maior queda foi **{area_queda} no {regiao_queda} '
    f'({delta_regional.min():+.1f} pontos)**. Para verificar se comparamos grupos semelhantes, '
    'agora separamos rede e localização.'))

Em Matemática 2024, as **119,759 notas amostradas** com UF escolar têm média 520.7, mediana 492.0 e Q1–Q3 428.4 a 600.4. Na série completa, 19 das 25 combinações região × área caíram de 2023 para 2024. A maior queda foi **Matemática no Sudeste (-55.2 pontos)**. Para verificar se comparamos grupos semelhantes, agora separamos rede e localização.

## 3. Composição dos participantes

### Comparações dentro da rede e localização

**Pergunta:** as diferenças entre territórios permanecem quando comparamos grupos escolares parecidos? Aqui entram rede e localização urbana/rural. Os filtros permitem qualquer par de UFs ou regiões; Ceará e Sudeste são apenas a seleção inicial.

O cálculo reúne as notas da amostra em cada território, ano e área. Não fazemos uma nova amostragem para cada filtro. Os quartis mostram a dispersão dentro de cada grupo, não a margem de erro da mediana. Grupos com menos de 30 notas não recebem uma curva; o tamanho continua na tabela. O limite é uma precaução de leitura, não garantia de precisão.

Rede privada não significa escola preparatória. Estes arquivos não informam frequência em cursinho nem intensidade da preparação para vestibulares. Uma diferença urbana/rural tampouco mede, sozinha, o efeito de isolamento geográfico.

In [ ]:
# @title
import plotly.graph_objects as go
from IPython.display import HTML, display

def mostrar_comparacao(
    area, territorio_a, territorio_b, rede,
    localizacao, intervalo, ver_tabela
):
    fig = go.Figure()
    resumos = []

    for indice, (territorio, cor) in enumerate(zip(
        [territorio_a, territorio_b],
        ['#0056B3', '#C34109']
    )):
        d = recortar(amostra, territorio, rede)
        d = d[d.AREA.eq(area) & d.ANO.between(*intervalo)]

        if localizacao != 'Todas':
            d = d[d.LOCALIZACAO.eq(localizacao)]

        r = (
            d.groupby('ANO', observed=True).NOTA
            .agg(
                N='size',
                MEDIANA='median',
                Q1=lambda x: x.quantile(.25),
                Q3=lambda x: x.quantile(.75)
            )
            .reset_index()
        )
        r['TERRITORIO'] = territorio
        resumos.append(r)

        r = r[r.N.ge(30)]
        if r.empty:
            continue

        grupo = str(indice)
        rgb = ', '.join(
            str(int(cor[i:i + 2], 16)) for i in (1, 3, 5)
        )

        # Faixa entre o primeiro e o terceiro quartil.
        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.Q1,
            mode='lines',
            line=dict(width=0),
            legendgroup=grupo,
            showlegend=False,
            hoverinfo='skip'
        ))

        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.Q3,
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor=f'rgba({rgb}, 0.16)',
            legendgroup=grupo,
            showlegend=False,
            hoverinfo='skip'
        ))

        # Mediana e valores exibidos ao passar o mouse.
        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.MEDIANA,
            mode='lines+markers',
            name=territorio,
            legendgroup=grupo,
            line=dict(color=cor),
            marker=dict(size=8),
            customdata=r[['N', 'Q1', 'Q3']].to_numpy(),
            hovertemplate=(
                '<b>%{fullData.name}</b>'
                '<br>Ano: %{x:.0f}'
                '<br>Mediana: %{y:.2f} pontos'
                '<br>Q1: %{customdata[1]:.2f} pontos'
                '<br>Q3: %{customdata[2]:.2f} pontos'
                '<br>Contagem: %{customdata[0]:.0f}'
                '<extra></extra>'
            )
        ))

    fig.update_layout(
        title=f'Como as notas se comparam entre os grupos?<br><sup>{area} · {rede} · {localizacao}</sup>',
        xaxis_title='Ano do ENEM',
        yaxis_title='Nota na prova (pontos)',
        template='plotly_white',
        height=480,
        hovermode='closest',
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(intervalo[0], intervalo[1] + 1))
        ),
        legend_title_text='Território'
    )

    if fig.data:
        mostrar_figura(fig)
    else:
        print('Nenhum território tem 30 notas no recorte selecionado.')

    resultado = pd.concat(resumos, ignore_index=True)
    tabela_opcional(resultado, ver_tabela)
    salvar(resultado, 'comparacao_territorios.csv')


painel(
    mostrar_comparacao,
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    territorio_a=escolher(TERRITORIOS, 'CE', 'Território A:'),
    territorio_b=escolher(TERRITORIOS, 'Sudeste', 'Território B:'),
    rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'),
    localizacao=escolher(
        ['Todas', 'Urbana', 'Rural', 'Não informada'],
        'Todas',
        'Localização:'
    ),
    intervalo=periodo(),
    ver_tabela=widgets.Checkbox(
        value=False,
        description='Ver contagens e quartis'
    )
)

### Mudança na participação das redes e decomposição da média

O primeiro gráfico acompanha a participação de uma rede em cada ano. A barra parte de zero e usa como denominador todas as notas elegíveis com UF escolar do recorte. Selecione território, área, rede e período.
O segundo gráfico compara duas edições. Nos grupos **UF × rede × localização** com pelo menos 30 notas nos dois anos, mostra a variação observada da média, a variação mantendo a composição inicial e a diferença entre as duas. Essa diferença indica quanto o resultado é sensível à troca dos pesos. O percentual de notas dos grupos comuns informa a abrangência em cada ano. Categorias sem rede ou localização informada permanecem como grupos próprios.

A decomposição é descritiva, usa a média e depende do ano inicial como referência. Ela compara grupos de edições diferentes. Fatores como renda, seleção dentro de cada grupo e mudanças no exame ainda podem contribuir para a variação.

In [ ]:
# @title

def decompor_composicao(territorio, area, inicio, fim):
    d = recortar(enem_redes, territorio)
    d = d[d.AREA.eq(area)]
    chave = ['UF_ESCOLA', 'REDE', 'LOCALIZACAO']
    a = d[d.ANO.eq(inicio)].set_index(chave)
    b = d[d.ANO.eq(fim)].set_index(chave)
    comum = a.index.intersection(b.index)
    comum = comum[(a.loc[comum, 'N'].ge(30) & b.loc[comum, 'N'].ge(30)).to_numpy()]
    if len(comum) == 0:
        return None
    ga, gb = a.loc[comum], b.loc[comum]
    wa, wb = ga.N / ga.N.sum(), gb.N / gb.N.sum()
    media_a, media_b = (wa * ga.MEDIA).sum(), (wb * gb.MEDIA).sum()
    media_fixa = (wa * gb.MEDIA).sum()
    return {'OBSERVADA': media_b - media_a, 'COMPOSICAO_INICIAL': media_fixa - media_a,
        'PARCELA_COMPOSICAO': media_b - media_fixa,
        'COBERTURA_INICIAL': 100 * ga.N.sum() / a.N.sum(),
        'COBERTURA_FINAL': 100 * gb.N.sum() / b.N.sum(), 'N_GRUPOS': len(comum)}


def mostrar_participacao_redes(territorio, area, rede, intervalo):
    d = recortar(enem_redes, territorio)
    d = d.loc[d.AREA.eq(area) & d.ANO.between(*intervalo)]
    total = d.groupby('ANO').N.sum()
    redes = ['Federal', 'Estadual', 'Municipal'] if rede == 'Públicas' else [rede]
    parte = d.loc[d.REDE.isin(redes)].groupby('ANO').N.sum().reindex(total.index, fill_value=0)
    quadro = pd.DataFrame({'ROTULO': total.index, 'N': parte, 'TOTAL': total})
    barras_taxa(quadro, f'Qual é a parcela de notas da rede {rede.lower()}? · {area}<br><sup>{territorio}</sup>',
                'Notas válidas dessa rede (%)', '#007763')


def mostrar_composicao(territorio, area, inicio, fim):
    if inicio >= fim:
        print('Escolha um ano final posterior ao inicial.')
        return
    resultado = decompor_composicao(territorio, area, inicio, fim)
    if resultado is None:
        print('Não há grupos comuns com 30 notas em ambos os anos.')
        return
    rotulos = ['Mudança observada', 'Com as proporções do ano inicial', 'Diferença ligada à composição']
    valores = [resultado[c] for c in ['OBSERVADA', 'COMPOSICAO_INICIAL', 'PARCELA_COMPOSICAO']]
    detalhes = np.tile([resultado['N_GRUPOS'], resultado['COBERTURA_INICIAL'],
                        resultado['COBERTURA_FINAL']], (len(rotulos), 1))
    fig = go.Figure(go.Bar(
        x=valores, y=rotulos, orientation='h',
        marker_color=['#17376D', '#0056B3', '#C34109'], width=.58,
        text=[f'{x:+.1f}' for x in valores], textposition='outside', cliponaxis=False,
        customdata=detalhes,
        hovertemplate=('<b>%{y}</b><br>Diferença: %{x:+.2f} pontos'
                       '<br>Grupos comuns: %{customdata[0]:.0f}'
                       f'<br>Cobertura em {inicio}: %{{customdata[1]:.2f}}%'
                       f'<br>Cobertura em {fim}: %{{customdata[2]:.2f}}%<extra></extra>'),
    ))
    fig.add_vline(x=0, line_color='#7A8792', line_width=1)
    fig.update_layout(title=f'Como a mudança no perfil dos participantes aparece na média?<br><sup>{area} · {inicio} a {fim} · {territorio}</sup>',
                      height=360)
    fig.update_xaxes(title='Mudança da nota média (pontos)')
    fig.update_yaxes(title='Cenário comparado', autorange='reversed', automargin=True)
    mostrar_figura(fig)
    print(f'{resultado["N_GRUPOS"]} grupos comuns. Cobertura: '
          f'{resultado["COBERTURA_INICIAL"]:.2f}% em {inicio} e {resultado["COBERTURA_FINAL"]:.2f}% em {fim}.')


painel(mostrar_participacao_redes,
       territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       rede=escolher(REDES_ESTUDO[1:], 'Estadual', 'Rede:'),
       intervalo=periodo())
painel(mostrar_composicao,
       territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       inicio=escolher(ANOS_ESTUDO, 2023, 'De:'),
       fim=escolher(ANOS_ESTUDO, 2024, 'Até:'))

In [ ]:
comp_mt = decompor_composicao(TERRITORIOS[0], 'Matemática', 2023, 2024)
decomposicoes = []
for area in AREAS_ESTUDO:
    for inicio, fim in zip(ANOS_ESTUDO, ANOS_ESTUDO[1:]):
        resultado = decompor_composicao(TERRITORIOS[0], area, inicio, fim)
        if resultado is not None:
            decomposicoes.append({'AREA': area, 'ANO_INICIAL': inicio, 'ANO_FINAL': fim, **resultado})
salvar(pd.DataFrame(decomposicoes), 'decomposicao_media_por_area_transicao.csv')

comparavel = (amostra.query('AREA == "Matemática" and REDE == "Estadual" and ANO == 2024')
              .groupby(['UF_ESCOLA', 'LOCALIZACAO'], observed=True).NOTA
              .agg(N='size', MEDIANA='median'))
ce_urbana = comparavel.loc[('CE', 'Urbana')]
ce_rural = comparavel.loc[('CE', 'Rural')]
display(Markdown(
    f'Na amostra estadual do Ceará em Matemática 2024, a mediana urbana foi '
    f'{ce_urbana.MEDIANA:.1f} (n={ce_urbana.N:,.0f}) e a rural, '
    f'{ce_rural.MEDIANA:.1f} (n={ce_rural.N:,.0f}). A comparação por rede ainda reúne '
    'participantes com características distintas. '
    f'No Brasil com UF escolar, a média completa nos grupos comparáveis variou '
    f'**{comp_mt["OBSERVADA"]:+.1f}** pontos entre 2023 e 2024. Com pesos de 2023, '
    f'a variação foi {comp_mt["COMPOSICAO_INICIAL"]:+.1f}; a diferença foi '
    f'{comp_mt["PARCELA_COMPOSICAO"]:+.1f}. Os grupos cobrem '
    f'{comp_mt["COBERTURA_INICIAL"]:.1f}% e {comp_mt["COBERTURA_FINAL"]:.1f}% '
    'dos respectivos recortes com UF. Essa mudança pode esconder diferenças entre áreas dentro '
    'da mesma pessoa, que examinamos a seguir.'))

Na amostra estadual do Ceará em Matemática 2024, a mediana urbana foi 468.6 (n=7,332) e a rural, 450.4 (n=496). A comparação por rede ainda reúne participantes com características distintas. No Brasil com UF escolar, a média completa nos grupos comparáveis variou **-26.5** pontos entre 2023 e 2024. Com pesos de 2023, a variação foi -12.6; a diferença foi -13.9. Os grupos cobrem 99.9% e 100.0% dos respectivos recortes com UF. Essa mudança pode esconder diferenças entre áreas dentro da mesma pessoa, que examinamos a seguir.

## 4. Desempenho relativo da mesma pessoa

### Diferença de posição entre pares de áreas

**Pergunta.** Um mesmo participante ocupa posições relativas diferentes nas quatro provas objetivas? Usamos a amostra conjunta de 10% por UF e edição, com semente fixa, entre elegíveis com as quatro provas válidas e UF escolar conhecida. Os sorteios separados da Etapa 1 servem às comparações por área, que aparecem em outros painéis.

Cada prova fornece um percentil na distribuição nacional do recorte elegível da edição. Comparamos a média dos percentis de MT/CN com a de LC/CH. Passe o mouse sobre um ponto para ver os quatro percentis. As diagonais tracejadas delimitam a faixa de posições próximas, e o corte pode variar de 10 a 20 pontos percentuais. Trata-se de uma regra descritiva de posição relativa.

A dispersão usa até 2.500 pontos. No gráfico de comparação, cada barra parte de zero e mostra a parcela de um único perfil. Use **UFs de uma região** para comparar os estados de uma mesma região ou **Regiões do Brasil** para comparar as regiões. As porcentagens são calculadas com todas as observações amostrais do grupo. O menu **Edições** compara anos quando o filtro de edição está em **Todas**. No recorte agrupado de várias edições, cada participante conserva seu percentil do próprio ano.

In [ ]:
ORDEM_DIFERENCAS = ['Maior posição em MT/CN', 'Posições próximas', 'Maior posição em LC/CH']
CORES_DIFERENCAS = ['#0056B3', '#46566D', '#C34109']
GRUPOS_PERFIL = {
    'Regiões do Brasil': 'REGIAO', 'UFs de uma região': 'UF_ESCOLA', 'UFs do Brasil': 'UF_ESCOLA',
    'Edições': 'ANO', 'Redes': 'REDE', 'Localização': 'LOCALIZACAO',
    'Renda familiar': 'GRUPO_RENDA_FAMILIAR',
}
EDICOES_PERFIL = ['Todas', *ANOS_ESTUDO]


def recorte_perfis(edicao, territorio, rede):
    d = recortar(perfis, territorio, rede)
    return d if edicao == 'Todas' else d.loc[d.ANO.eq(edicao)]


def comparacao_perfis(edicao, agrupar, regiao, rede):
    territorio = regiao if agrupar == 'UFs de uma região' else TERRITORIOS[0]
    d = recorte_perfis(edicao, territorio, rede)
    if agrupar == 'Renda familiar':
        d = d.loc[d.ANO.lt(2024)]
        print('A renda familiar ligada às notas está disponível até 2023 neste painel.')
    edicao_texto = 'Todas as edições disponíveis' if edicao == 'Todas' else str(edicao)
    detalhe = f'{territorio} · {edicao_texto} · Rede: {rede}'
    return d, GRUPOS_PERFIL[agrupar], detalhe


def grupo_perfis_regiao(nivel='Regiões do Brasil'):
    grupo = escolher(list(GRUPOS_PERFIL), nivel, 'Comparar:')
    regiao = escolher(REGIOES_ESTUDO, 'Nordeste', 'Região das UFs:')
    regiao.disabled = nivel != 'UFs de uma região'

    def atualizar_regiao(mudanca):
        regiao.disabled = mudanca['new'] != 'UFs de uma região'

    grupo.observe(atualizar_regiao, names='value')
    return grupo, regiao


def percentual_perfil(valor):
    if 0 < valor < .01:
        return '<0,01%'
    precisao = 2 if 0 < valor < 1 else 1
    return f'{valor:.{precisao}f}%'.replace('.', ',')


def barras_perfis(contagens, perfil, titulo, detalhe, cor, cronologico=False):
    totais = contagens.sum(axis=1)
    dados = pd.DataFrame({'N': totais, 'N_PERFIL': contagens[perfil]})
    dados = dados.loc[dados.N.ge(30)].copy()
    if dados.empty:
        print('Nenhum grupo deste recorte tem pelo menos 30 participantes.')
        return
    dados['PCT'] = 100 * dados.N_PERFIL / dados.N
    dados = dados.sort_index() if cronologico else dados.sort_values('PCT', ascending=False, kind='stable')
    rotulos = dados.PCT.map(percentual_perfil)
    detalhes = pd.DataFrame({
        'rotulo': rotulos,
        'contagem': dados.N_PERFIL.map(lambda n: f'{n:,}'.replace(',', '.')),
        'total': dados.N.map(lambda n: f'{n:,}'.replace(',', '.')),
    }).to_numpy()
    grupos = dados.index.astype(str).tolist()
    fig = go.Figure(go.Bar(
        x=dados.PCT, y=grupos, orientation='h', width=.55, marker_color=cor,
        text=rotulos, textposition='outside', cliponaxis=False, customdata=detalhes,
        hovertemplate=('<b>%{y}</b><br>Parcela no grupo: %{customdata[0]}'
                       '<br>Participantes com o perfil: %{customdata[1]}'
                       '<br>Participantes no grupo: %{customdata[2]}<extra></extra>'),
    ))
    configurar_painel(fig, titulo, detalhe, altura=max(290, 195 + 30 * len(grupos)))
    fig.update_layout(margin=dict(l=125, r=96))
    fig.update_yaxes(type='category', categoryorder='array', categoryarray=grupos,
                     autorange='reversed', automargin=True, title='Grupo comparado')
    fig.update_xaxes(title='Participantes com esse perfil (%)')
    fixar_percentual(fig)
    mostrar_figura(fig)
    omitidos = len(totais) - len(dados)
    if omitidos:
        print(f'Grupos com menos de 30 participantes omitidos: {omitidos}.')


def figura_dispersao_perfis(d, classificacao, ordem, cores, titulo, detalhe, n_maximo):
    pontos = d.sample(min(n_maximo, len(d)), random_state=42).assign(PERFIL_PONTO=classificacao)
    # A classificação é uma Series alinhada ao índice original; o sorteio preserva esse índice.
    fig = go.Figure()
    colunas_hover = ['UF_ESCOLA', 'PERCENTIL_MT', 'PERCENTIL_CN', 'PERCENTIL_LC', 'PERCENTIL_CH']
    for rotulo, cor in zip(ordem, cores):
        grupo = pontos.loc[pontos.PERFIL_PONTO.eq(rotulo)]
        if not grupo.empty:
            fig.add_trace(go.Scattergl(
                x=grupo.POS_LC_CH, y=grupo.POS_MT_CN, mode='markers', name=rotulo,
                marker=dict(color=cor, size=7, opacity=.68),
                customdata=grupo[colunas_hover].to_numpy(),
                hovertemplate=(
                    '<b>%{fullData.name}</b> · %{customdata[0]}<br>'
                    'Média LC/CH: %{x:.1f} · Média MT/CN: %{y:.1f}<br>'
                    'LC: %{customdata[3]:.1f} · CH: %{customdata[4]:.1f}<br>'
                    'MT: %{customdata[1]:.1f} · CN: %{customdata[2]:.1f}<extra></extra>'
                ),
            ))
    configurar_painel(fig, titulo, detalhe, altura=510)
    fig.update_xaxes(title='Posição média em Linguagens e Humanas (0 a 100)', range=[0, 100], constrain='domain')
    fig.update_yaxes(title='Posição média em Matemática e Natureza (0 a 100)', range=[0, 100], scaleanchor='x')
    return fig


def classificar_diferencas(d, limite):
    diferenca = d.POS_MT_CN - d.POS_LC_CH
    perfil = np.select(
        [diferenca.ge(limite), diferenca.le(-limite)],
        [ORDEM_DIFERENCAS[0], ORDEM_DIFERENCAS[2]], default=ORDEM_DIFERENCAS[1],
    )
    return pd.Series(perfil, index=d.index, name='PERFIL')


def mostrar_dispersao_diferencas(ano, territorio, rede, limite):
    d = recorte_perfis(ano, territorio, rede)
    if len(d) < 30:
        print(f'Recorte com {len(d)} participantes. Amplie os filtros.')
        return
    detalhe = f'{territorio} · {ano} · {rede} · Corte de ±{limite} p.p.'
    fig = figura_dispersao_perfis(
        d, classificar_diferencas(d, limite), ORDEM_DIFERENCAS, CORES_DIFERENCAS,
        'Como a mesma pessoa se saiu nos dois pares de provas?', detalhe, 2500,
    )
    linha = dict(color='#56697D', width=1.2, dash='dash')
    fig.add_shape(type='line', x0=0, y0=limite, x1=100-limite, y1=100, line=linha)
    fig.add_shape(type='line', x0=limite, y0=0, x1=100, y1=100-limite, line=linha)
    mostrar_figura(fig)


def mostrar_perfis(edicao, agrupar, regiao, rede, limite, perfil):
    d, grupo, detalhe = comparacao_perfis(edicao, agrupar, regiao, rede)
    classes = classificar_diferencas(d, limite)
    contagens = pd.crosstab(d[grupo], classes).reindex(columns=ORDEM_DIFERENCAS, fill_value=0)
    barras_perfis(contagens, perfil, f'Em quais grupos esse perfil é mais comum? · {perfil}', f'{detalhe} · Corte de ±{limite} p.p.',
                  CORES_DIFERENCAS[ORDEM_DIFERENCAS.index(perfil)], cronologico=grupo == 'ANO')


perfis['POS_MT_CN'] = perfis[['PERCENTIL_MT', 'PERCENTIL_CN']].mean(axis=1)
perfis['POS_LC_CH'] = perfis[['PERCENTIL_LC', 'PERCENTIL_CH']].mean(axis=1)

painel(mostrar_dispersao_diferencas, ano=escolher(ANOS_ESTUDO, 2023, 'Ano:'),
       territorio=escolher(TERRITORIOS, TERRITORIOS[0], 'Território:'),
       rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'), limite=escolher([10, 15, 20], 15, 'Diferença:'))

grupo_diferencas, regiao_diferencas = grupo_perfis_regiao()
painel(mostrar_perfis, edicao=escolher(EDICOES_PERFIL, 2023, 'Edição:'),
       agrupar=grupo_diferencas, regiao=regiao_diferencas, rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'),
       limite=escolher([10, 15, 20], 15, 'Diferença:'),
       perfil=escolher(ORDEM_DIFERENCAS, 'Posições próximas', 'Perfil:'))

### Posições próximas podem ser altas ou baixas

A diferença entre MT/CN e LC/CH mostra a direção do desequilíbrio. Para distinguir quem está alto ou baixo nas duas dimensões, cruzamos as médias dos percentis. O eixo horizontal é LC/CH e o vertical é MT/CN. O **corte 50** classifica cada dimensão como alta a partir de 50 e baixa abaixo de 50. Uma média de dois percentis pode ter distribuição diferente da de cada prova, por isso 50 pode deixar quantidades diferentes de pessoas de cada lado.

O **corte 40/60** usa ≤40 para baixo e ≥60 para alto. Quando uma posição fica entre os cortes, a pessoa entra no grupo intermediário. Compare os cortes para verificar se a conclusão depende da definição. A legenda da dispersão permite ocultar ou destacar perfis.

As barras comparam a prevalência de um perfil por vez, a partir de zero, com um eixo ajustado ao recorte. Compare **UFs de uma região** primeiro e depois **Regiões do Brasil**. Nos dois casos, o denominador são os participantes do grupo observado e o mínimo para exibição é 30. A proporção de uma região usa a contagem conjunta dos participantes e reflete o tamanho das amostras de cada UF. O hover traz numerador e denominador. Nas tabelas exportadas, a composição inversa indica de quais grupos vêm os participantes com um perfil.

As posições são relativas aos elegíveis com quatro provas e UF escolar. O grupo de renda familiar vem de `Q006` até 2023 e usa intervalos em salários mínimos da edição. Em 2024, renda e situação escolar ficam indisponíveis nesse cruzamento.

In [ ]:
ORDEM_QUADRANTES = ['Alto nos dois', 'Baixo nos dois', 'Alto MT/CN', 'Alto LC/CH', 'Intermediário']
CORES_QUADRANTES = ['#00634E', '#46566D', '#0056B3', '#C34109', '#855000']


def classificar_perfis(d, faixa):
    x, y = d.POS_LC_CH, d.POS_MT_CN
    alto_x, alto_y = x.ge(60 if faixa else 50), y.ge(60 if faixa else 50)
    baixo_x, baixo_y = (x.le(40), y.le(40)) if faixa else (x.lt(50), y.lt(50))
    return np.select([alto_x & alto_y, baixo_x & baixo_y, baixo_x & alto_y, alto_x & baixo_y],
                     ORDEM_QUADRANTES[:4], default=ORDEM_QUADRANTES[-1])


perfis['QUADRANTE_50'] = classificar_perfis(perfis, False)
perfis['QUADRANTE_40_60'] = classificar_perfis(perfis, True)
perfis.to_parquet(PASTA_RESULTADOS / 'participantes_perfis.parquet', index=False)


def ordem_quadrantes(corte):
    return ORDEM_QUADRANTES if corte == '40/60' else ORDEM_QUADRANTES[:4]


def controles_quadrante():
    corte = escolher(['50', '40/60'], '50', 'Corte:')
    perfil = escolher(ordem_quadrantes(corte.value), ORDEM_QUADRANTES[0], 'Perfil:')

    def atualizar_perfil(mudanca):
        anterior = perfil.value
        perfil.options = ordem_quadrantes(mudanca['new'])
        if anterior in perfil.options:
            perfil.value = anterior

    corte.observe(atualizar_perfil, names='value')
    return corte, perfil


def mostrar_dispersao_quadrantes(ano, territorio, rede, corte):
    d = recorte_perfis(ano, territorio, rede)
    if len(d) < 30:
        print(f'Recorte com {len(d)} participantes. Amplie os filtros.')
        return
    faixa = corte == '40/60'
    coluna = 'QUADRANTE_40_60' if faixa else 'QUADRANTE_50'
    detalhe = f'{territorio} · {ano} · {rede} · Corte {corte}'
    fig = figura_dispersao_perfis(
        d, d[coluna], ordem_quadrantes(corte), CORES_QUADRANTES,
        'Quem ficou em posição alta ou baixa nos dois pares?', detalhe, 3000,
    )
    linha = dict(line_color='#56697D', line_width=1.2, line_dash='dot')
    for limite in ([40, 60] if faixa else [50]):
        fig.add_vline(x=limite, **linha)
        fig.add_hline(y=limite, **linha)
    mostrar_figura(fig)


def mostrar_quadrantes(edicao, agrupar, regiao, rede, corte, perfil, ver_tabela=False):
    d, grupo, detalhe = comparacao_perfis(edicao, agrupar, regiao, rede)
    ordem = ordem_quadrantes(corte)
    if perfil not in ordem:
        return
    coluna = 'QUADRANTE_40_60' if corte == '40/60' else 'QUADRANTE_50'
    contagens = pd.crosstab(d[grupo], d[coluna]).reindex(columns=ordem, fill_value=0)
    totais = contagens.sum(axis=1)
    comparaveis = contagens.loc[totais.ge(30)]
    barras_perfis(contagens, perfil, f'Em quais grupos esse perfil é mais comum? · {perfil}', f'{detalhe} · Corte {corte}',
                  CORES_QUADRANTES[ordem.index(perfil)], cronologico=grupo == 'ANO')
    if comparaveis.empty:
        return
    n = totais.loc[comparaveis.index]
    prevalencia = 100 * comparaveis.div(n, axis=0)
    composicao = 100 * comparaveis.div(comparaveis.sum(axis=0).replace(0, np.nan), axis=1)
    if ver_tabela:
        pct_formatada = prevalencia.apply(lambda coluna: coluna.map(percentual_perfil))
        rotulos = comparaveis.astype(str) + ' · ' + pct_formatada
        display(rotulos.assign(N=n))
    salvar(prevalencia.round(3).assign(N=n).reset_index(), 'perfis_prevalencia_recorte.csv')
    salvar(composicao.round(3).reset_index(), 'perfis_composicao_recorte.csv')


painel(mostrar_dispersao_quadrantes, ano=escolher(ANOS_ESTUDO, 2023, 'Ano:'),
       territorio=escolher(TERRITORIOS, TERRITORIOS[0], 'Território:'),
       rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'), corte=escolher(['50', '40/60'], '50', 'Corte:'))

grupo_quadrantes, regiao_quadrantes = grupo_perfis_regiao('UFs de uma região')
corte_quadrantes, perfil_quadrantes = controles_quadrante()
painel(mostrar_quadrantes, edicao=escolher(EDICOES_PERFIL, 2023, 'Edição:'),
       agrupar=grupo_quadrantes, regiao=regiao_quadrantes, rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'),
       corte=corte_quadrantes, perfil=perfil_quadrantes,
       ver_tabela=widgets.Checkbox(value=False, description='Mostrar tabela de perfis', indent=False))

In [ ]:
est_2023 = perfis.query('ANO == 2023 and REDE == "Estadual"')
perfil_50 = 100 * est_2023.QUADRANTE_50.value_counts(normalize=True).reindex(ORDEM_QUADRANTES[:4], fill_value=0)
intermediarios = 100 * est_2023.QUADRANTE_40_60.eq('Intermediário').mean()
proximos = est_2023[est_2023.VANTAGEM_EXATAS_PP.abs().lt(15)]
ambos_niveis = 100 * proximos.QUADRANTE_50.value_counts(normalize=True)
por_uf = pd.crosstab(est_2023.UF_ESCOLA, est_2023.QUADRANTE_50)
por_uf = por_uf.loc[por_uf.sum(axis=1).ge(30)]
alto_uf = 100 * por_uf['Alto nos dois'] / por_uf.sum(axis=1)
display(Markdown(
    f'Em 2023, a rede estadual reúne **{len(est_2023):,} participantes** na amostra conjunta. '
    + ', '.join(f'{pct:.1f}% {nome.lower()}' for nome, pct in perfil_50.items())
    + f'. Entre os {len(proximos):,} com diferença inferior a 15 p.p., '
    f'{ambos_niveis["Alto nos dois"]:.1f}% estão altos nos dois e '
    f'{ambos_niveis["Baixo nos dois"]:.1f}% estão baixos nos dois. Com 40/60, '
    f'{intermediarios:.1f}% ficam intermediários. Entre as UFs com n ≥ 30, '
    f'a prevalência de alto nos dois varia de {alto_uf.min():.1f}% ({alto_uf.idxmin()}) '
    f'a {alto_uf.max():.1f}% ({alto_uf.idxmax()}). São posições relativas; '
    'para investigar por que a distribuição varia por território, precisamos descrever seu contexto.'))
renda_perfis = pd.crosstab(est_2023.GRUPO_RENDA_FAMILIAR, est_2023.QUADRANTE_50)
renda_perfis = renda_perfis.reindex(columns=ORDEM_QUADRANTES[:4], fill_value=0)
n_renda = renda_perfis.sum(axis=1)
display((100 * renda_perfis.div(n_renda, axis=0)).where(n_renda.ge(30)).round(1).assign(N=n_renda))
print('Renda familiar na amostra estadual de 2023. Cada linha mostra percentuais dentro da faixa.')


Em 2023, a rede estadual reúne **45,867 participantes** na amostra conjunta. 24.6% alto nos dois, 54.6% baixo nos dois, 9.0% alto mt/cn, 11.8% alto lc/ch. Entre os 28,581 com diferença inferior a 15 p.p., 29.1% estão altos nos dois e 63.2% estão baixos nos dois. Com 40/60, 39.0% ficam intermediários. Entre as UFs com n ≥ 30, a prevalência de alto nos dois varia de 12.4% (PI) a 35.5% (RS). São posições relativas; para investigar por que a distribuição varia por território, precisamos descrever seu contexto.

QUADRANTE_50,Alto nos dois,Baixo nos dois,Alto MT/CN,Alto LC/CH,N
GRUPO_RENDA_FAMILIAR,,,,,
"Até 1,5 salário mínimo",16.9,64.3,7.9,10.9,27511
"Mais de 1,5 até 4 salários mínimos",33.8,42.5,10.5,13.3,14677
Mais de 4 salários mínimos,45.6,30.3,11.2,12.8,3679


Renda familiar na amostra estadual de 2023. Cada linha mostra percentuais dentro da faixa.


### Auditoria nacional · situação escolar e perfis

Entre 2020 e 2023, a amostra conjunta com UF escolar é formada integralmente por concluintes declarados. Para comparar situações escolares, ampliamos o universo a **todos os elegíveis com quatro provas válidas no país**, incluindo quem não tem UF escolar. Esta análise agregada usa novos percentis nacionais calculados nesse universo completo, ano a ano, antes da divisão por situação. Os percentuais deste painel e dos quadrantes territoriais têm referências diferentes. Este recorte nacional ajuda a explicar quem fica fora da análise com UF e permanece separado dos perfis e das integrações territoriais.

Cada linha mostra a distribuição de perfis **dentro da situação escolar** e o número de inscritos elegíveis. O filtro de cobertura permite ver quanta diversidade de situação aparece fora do recorte territorial. Em 2024, os arquivos públicos separados impedem classificar os perfis por situação.

Escolha um perfil para comparar sua proporção entre as situações escolares.
Ao passar o mouse, veja a contagem, o total do grupo e a porcentagem.

In [ ]:
perfis_nacionais = ler('perfis_nacionais_por_situacao.csv')


def mostrar_perfis_situacao(ano, cobertura_uf, corte, perfil):
    d = perfis_nacionais.query('ANO == @ano and CORTE == @corte')
    if cobertura_uf != 'Todos os elegíveis':
        d = d.loc[d.UF_CONHECIDA.eq(cobertura_uf == 'Com UF escolar')]
    ordem = ordem_quadrantes(corte)
    if perfil not in ordem:
        return
    contagens = d.pivot_table(index='SITUACAO_ESCOLAR', columns='PERFIL', values='N',
                              aggfunc='sum', fill_value=0).reindex(columns=ordem, fill_value=0)
    detalhe = f'{ano} · {cobertura_uf} · Corte {corte} · Referência nacional'
    barras_perfis(contagens, perfil, f'Como esse perfil varia com a situação escolar? · {perfil}', detalhe,
                  CORES_QUADRANTES[ordem.index(perfil)])


corte_situacao, perfil_situacao = controles_quadrante()
painel(mostrar_perfis_situacao, ano=escolher(ANOS_ESTUDO[:-1], 2023, 'Ano:'),
       cobertura_uf=escolher(['Todos os elegíveis', 'Com UF escolar', 'Sem UF escolar'],
                             'Todos os elegíveis', 'Cobertura:'),
       corte=corte_situacao, perfil=perfil_situacao)

### Documentação do período e diagnóstico da TRI

Os resultados precisam ser confrontados com a documentação do período. Os eventos abaixo são **contexto histórico**, não colunas de tratamento atribuídas aos participantes.

| Referência | O que está documentado | Como entra na leitura |
|---|---|---|
| 2020–2021: interrupção das aulas presenciais | O Inep documentou ampla suspensão e alterações no calendário escolar. [Nota informativa do Ideb 2021](https://download.inep.gov.br/educacao_basica/portal_ideb/planilhas_para_download/2021/nota_informativa_ideb_2021.pdf) | Torna plausível investigar consequências posteriores; não identifica quanto cada participante perdeu nem um atraso específico até 2024. |
| 2023: criação do Programa Escola em Tempo Integral | A política federal prevê apoio à ampliação de matrículas. [Notícia de 31/07/2023](https://www.gov.br/planalto/pt-br/acompanhe-o-planalto/noticias/2023/07/escola-de-qualidade-e-base-para-sociedade-menos-desigual-diz-lula) | Motiva observar oferta e jornada. Não confundimos anúncio com execução nem toda matrícula integral com adesão ao novo programa. |
| 2024: apoio ao acesso ao ENEM no Ceará | Houve divulgação de transporte gratuito em metrô e VLT nos dias de prova. [Governo do Ceará, 31/10/2024](https://www.ce.gov.br/2024/10/31/inscritos-no-enem-contarao-com-metro-e-vlts-gratuitamente-nos-dois-domingos-de-prova/) | Pode motivar estudos de participação. Não sabemos quem utilizou o transporte; a notícia não demonstra aumento de nota. |
| ENEM: notas e comparabilidade | A TRI considera características dos itens e padrão de respostas; Redação segue critérios próprios. [Guia do participante sobre as notas](https://download.inep.gov.br/educacao_basica/enem/guia_participante/2013/guia_do_participante_notas.pdf) | Quantidades iguais de acertos não exigem notas iguais. Uma diferença entre anos não comprova mudança do método de correção. |

**Conclusão sobre a pandemia:** nossos dados não permitem atribuir a queda de 2024 a um “lag da pandemia”. Não temos acompanhamento individual, exposição escolar detalhada nem um grupo contrafactual adequado. Cobertura, composição e resultados diferem entre edições. Criar uma coluna `pandemia = 1` para todo mundo não resolveria isso.

**Conclusão sobre escolas preparatórias:** não há uma variável que identifique essa atividade de forma comparável em todas as UFs e anos. A hipótese fica pendente; não usamos rede privada como substituto.

O próximo gráfico é um diagnóstico complementar nacional de **2023 e 2024**, anos cujas respostas e gabaritos já foram conferidos nesta preparação. O recorte reúne notas elegíveis com e sem UF escolar para examinar a relação nacional entre acertos e nota. Os resultados ficam separados da integração por estado. Essa limitação está restrita ao anexo de acertos; a base territorial e os demais painéis cobrem 2020–2024. Não ampliamos a conclusão sobre TRI a anos ou territórios que não foram processados nesse diagnóstico.

In [ ]:
# @title
acertos = ler('notas_por_numero_acertos_2023_2024.csv')
CODIGOS_AREAS = {'Linguagens': 'LC', 'Ciências Humanas': 'CH', 'Ciências da Natureza': 'CN', 'Matemática': 'MT'}

def mostrar_acertos(area, ano):
    d = acertos[acertos.AREA.eq(CODIGOS_AREAS[area]) & acertos.N.ge(100)].copy()
    if ano != 'Comparar 2023 e 2024':
        d = d[d.ANO.eq(ano)]
    d = d.sort_values(['ANO', 'ACERTOS'])
    d['Edição'] = d.ANO.astype(str)
    fig = px.line(d, x='ACERTOS', y='NOTA_MEDIA', color='Edição', markers=True,
                  labels={'ACERTOS': 'Número de questões certas', 'NOTA_MEDIA': 'Nota média (pontos)',
                          'N': 'Notas', 'MINIMA': 'Mínima', 'MAXIMA': 'Máxima'},
                  hover_data={'N': ':,', 'NOTA_MEDIA': ':.1f', 'MINIMA': ':.1f', 'MAXIMA': ':.1f'})
    mostrar_figura(configurar_painel(fig, 'Qual é a nota média para cada número de acertos?', f'{area} · {ano}'))
    print('Elegíveis nacionais, inclusive sem UF. Cada ponto reúne pelo menos 100 notas. '
          'O hover mostra a menor e a maior nota observadas no grupo.')

painel(mostrar_acertos, area=escolher(list(CODIGOS_AREAS), 'Matemática', 'Área:'),
       ano=escolher(['Comparar 2023 e 2024', 2023, 2024], 'Comparar 2023 e 2024', 'Ano:'))

## 5. O que acrescentam as fontes externas?

### Fontes, unidades e chaves

Cada linha do resultado territorial representa **UF da escola × edição do ENEM × área**. Por exemplo, Ceará, 2022, Matemática. Partimos dos resumos completos das notas elegíveis com UF escolar conhecida. As condições de ensino e renda entram como contexto da UF no ano correspondente.

| Fonte | Unidade antes da junção | O que acrescenta |
|---|---|---|
| ENEM (Inep) | UF escolar × ano × área | Contagem de notas elegíveis, média, mediana e quartis |
| Censo Escolar (Inep) | UF × ano, após agregar as escolas de ensino médio | Oferta de recursos, matrículas e jornada integral |
| PNAD Contínua (IBGE, tabela 7531) | UF × ano | Rendimento domiciliar médio por pessoa |
| Indicadores educacionais (Inep) | UF × ano × rede, antes de selecionar Total ou Estadual | Adequação da formação docente, horas-aula e rendimento escolar |

No recorte estadual, selecionamos as linhas da rede estadual do ENEM, do Censo Escolar e dos indicadores educacionais. A PNAD descreve toda a população da UF em ambos os recortes. Sua variável 10824 informa o rendimento médio mensal real domiciliar por pessoa, a preços médios do próprio ano. Reexpressamos a renda em preços de 2024 para a descrição e guardamos a série em preços de 2020 para a preparação da Etapa 3.

O vínculo é territorial. Padronizamos `SG_UF_ESC` do ENEM e `SG_UF` do Censo como `UF_ESCOLA`. A correspondência diz que as informações se referem à mesma UF no mesmo ano. As escolas do Censo descrevem a oferta de ensino médio no território. Essa unidade de análise não identifica a escola frequentada, a renda familiar ou a exposição de cada participante aos recursos.

Primeiro conferimos a disponibilidade de UFs por ano e fonte. Depois verificamos se cada chave encontra correspondência e se os campos da linha correspondente têm valores preenchidos.

Fontes oficiais. [Microdados do ENEM](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) · [Censo Escolar](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar) · [PNAD Contínua, tabela 7531](https://sidra.ibge.gov.br/tabela/7531) · [IPCA, tabela 1737](https://sidra.ibge.gov.br/tabela/1737) · [Indicadores educacionais do Inep](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais)


In [ ]:
# @title
censo_uf = ler('censo_uf_2020_2024.csv')
renda_uf = ler('renda_uf_2020_2024.csv')
formacao = ler('indicador_afd_uf_rede.csv')
jornada = ler('indicador_had_uf_rede.csv')
rendimento = ler('indicador_rendimento_uf_rede.csv')

fontes_anuais = {'ENEM': enem_uf, 'Censo Escolar': censo_uf, 'IBGE/PNAD': renda_uf,
                'Formação docente': formacao, 'Horas-aula': jornada, 'Rendimento': rendimento}
disponibilidade = pd.DataFrame({nome: tabela.groupby('ANO').UF_ESCOLA.nunique()
                               for nome, tabela in fontes_anuais.items()}).reindex(ANOS_ESTUDO)
assert disponibilidade.eq(27).all().all(), 'Confira as UFs ausentes antes de seguir.'
display(disponibilidade.rename_axis('Ano').style.set_caption('Número de UFs com dados em cada fonte'))
salvar(disponibilidade.reset_index(), 'cobertura_fontes.csv')

,ENEM,Censo Escolar,IBGE/PNAD,Formação docente,Horas-aula,Rendimento
Ano,,,,,,
2020,27,27,27,27,27,27
2021,27,27,27,27,27,27
2022,27,27,27,27,27,27
2023,27,27,27,27,27,27
2024,27,27,27,27,27,27


### Exclusão do FUNDEB mensal

Aumentar a quantidade de fontes não garante uma base melhor. O FUNDEB disponível neste pacote descreve dezembro, não o investimento anual no ensino médio estadual. Dividir esse valor pelas matrículas não corrige a incompatibilidade. Retiramos esse indicador dos gráficos de associação, da junção principal e das entradas de previsão.

Uma ampliação futura precisa de gasto executado anual, rede e etapa compatíveis, correção monetária e data de divulgação conhecida. O mesmo cuidado vale para renda individual e indicadores de aprendizagem de outras edições. A tabela abaixo registra exclusões para que elas não pareçam esquecimento.

In [ ]:
# @title

exclusoes = pd.DataFrame([
    ['FUNDEB apenas em dezembro', 'Mês isolado não mede investimento anual ou gasto na escola', 'Buscar execução anual com rede e etapa compatíveis'],
    ['Renda individual entre versões incompatíveis do ENEM', 'Não substituir renda familiar por renda média da UF', 'Auditar questionários e acesso antes de integrar'],
    ['Notas e perfis do próprio ano-alvo', 'Contêm a resposta que se deseja prever', 'Usar apenas histórico disponível antes da previsão'],
    ['Escolher uma UF por ter maior correlação', 'Favorece seleção oportunista e reduz a série', 'Examinar cobertura, comparabilidade e necessidade pública']],
    columns=['Informação', 'Por que não entra agora', 'Condição para reconsiderar'])
display(exclusoes.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
salvar(exclusoes, 'exclusoes_justificadas.csv')


Informação,Por que não entra agora,Condição para reconsiderar
FUNDEB apenas em dezembro,Mês isolado não mede investimento anual ou gasto na escola,Buscar execução anual com rede e etapa compatíveis
Renda individual entre versões incompatíveis do ENEM,Não substituir renda familiar por renda média da UF,Auditar questionários e acesso antes de integrar
Notas e perfis do próprio ano-alvo,Contêm a resposta que se deseja prever,Usar apenas histórico disponível antes da previsão
Escolher uma UF por ter maior correlação,Favorece seleção oportunista e reduz a série,"Examinar cobertura, comparabilidade e necessidade pública"


### Como as bases são unidas

**Base territorial, todas as redes.** A tabela do ENEM fica à esquerda do `merge`. Juntamos a ela o Censo Escolar agregado, a renda da PNAD e os indicadores educacionais da categoria Total, pela chave `ANO` + `UF_ESCOLA`. Cada fonte de contexto fornece uma linha por UF e ano. As cinco áreas do ENEM compartilham o mesmo contexto territorial para essa chave. É uma junção **muitos para um**.

**Validação.** O código checa duplicatas nas chaves de contexto, exige correspondência em cada junção à esquerda e confere que a chave do resultado `ANO` + `UF_ESCOLA` + `AREA` permanece única. Preservamos **675 linhas em 2020–2024**, ou 135 por edição. Também verificamos que a coluna `N`, que conta as notas elegíveis em cada recorte do ENEM, mantém seus valores. A auditoria de preenchimento distingue a correspondência da chave da presença de valores em cada indicador.

**Onde entram os recortes.** Registros sem UF escolar participam do diagnóstico nacional de cobertura. A amostra por área continua sendo a base dos gráficos de distribuição e serve para comparar as estimativas amostrais com os resumos completos. Na junção territorial, cada nota que entra em `N` pertence ao conjunto de elegíveis com UF escolar conhecida. Os indicadores acrescentam contexto sem criar participantes ou áreas.

O FUNDEB de dezembro permanece no catálogo para rastreabilidade, com a justificativa de exclusão da junção na célula anterior.


In [ ]:
# @title
chaves_territorio = ['ANO', 'UF_ESCOLA']
chaves_resultado = ['ANO', 'UF_ESCOLA', 'AREA']
base_integrada = enem_uf.copy()
auditoria_juncoes = []

def juntar_contexto(base, contexto, nome):
    assert not contexto.duplicated(chaves_territorio).any(), f'Chave repetida: {nome}'
    total_antes = len(base)
    nova = base.merge(contexto, on=chaves_territorio, how='left', validate='many_to_one', indicator=True)
    for ano, grupo in nova.groupby('ANO'):
        auditoria_juncoes.append({'FONTE': nome, 'ANO': ano, 'LINHAS_ENEM': len(grupo),
                                  'COM_CORRESPONDENCIA': int(grupo._merge.eq('both').sum())})
    assert len(nova) == total_antes
    assert nova._merge.eq('both').all(), f'Há chaves sem correspondência em {nome}'
    return nova.drop(columns='_merge')

base_integrada = juntar_contexto(base_integrada, censo_uf, 'Censo Escolar')
base_integrada = juntar_contexto(base_integrada, renda_uf, 'IBGE/PNAD')
for nome, tabela in [('Formação docente', formacao), ('Horas-aula', jornada), ('Rendimento', rendimento)]:
    contexto = tabela[tabela.REDE.eq('Total')].drop(columns='REDE')
    base_integrada = juntar_contexto(base_integrada, contexto, nome)

base_integrada = base_integrada.sort_values(chaves_resultado).reset_index(drop=True)
base_integrada['AMPLITUDE_INTERQUARTIL'] = base_integrada.Q3 - base_integrada.Q1
assert len(base_integrada) == 675
assert not base_integrada.duplicated(chaves_resultado).any()
pd.testing.assert_series_equal(base_integrada.N, enem_uf.sort_values(chaves_resultado).reset_index(drop=True).N)
auditoria_juncoes = pd.DataFrame(auditoria_juncoes)
salvar(auditoria_juncoes, 'auditoria_juncoes.csv')
salvar(base_integrada, 'base_territorial_2020_2024.csv')
display(auditoria_juncoes.pivot(index='ANO', columns='FONTE', values='COM_CORRESPONDENCIA'))
print('135 linhas por ano: 27 UFs × 5 áreas. A junção preservou as 675 linhas e as contagens do ENEM.')

FONTE,Censo Escolar,Formação docente,Horas-aula,IBGE/PNAD,Rendimento
ANO,,,,,
2020,135,135,135,135,135
2021,135,135,135,135,135
2022,135,135,135,135,135
2023,135,135,135,135,135
2024,135,135,135,135,135


135 linhas por ano: 27 UFs × 5 áreas. A junção preservou as 675 linhas e as contagens do ENEM.


### Recorte da rede estadual

A integração geral descreve todas as redes. Para estudar uma unidade próxima da atuação das secretarias estaduais, usamos notas, infraestrutura e indicadores da **rede estadual**. A renda continua descrevendo toda a UF. Essa delimitação vem da pergunta de gestão e da compatibilidade das fontes; não foi escolhida para obter uma correlação maior.

Recalculamos mediana, média e quartis diretamente de todas as notas elegíveis estaduais de 2020–2024. Não juntamos medianas urbanas e rurais. A amostra é mantida para comparar a diferença entre a estimativa amostral e o resumo completo e explorar distribuições; ela não define mais o alvo estadual.

As contagens são confrontadas com os resumos completos anteriores. Também conferimos a média pela soma ponderada dos grupos, operação válida para a média, mas não para a mediana. As bases geral e estadual compartilham participantes e não devem ser empilhadas como exemplos independentes.

**Sequência das junções estaduais.** Primeiro ligamos o resumo amostral (`N_AMOSTRA` e `MEDIANA_AMOSTRA`) ao resumo completo pela chave `ANO` + `UF_ESCOLA` + `AREA`, com validação **um para um**. Em seguida, acrescentamos pela chave `ANO` + `UF_ESCOLA` os contextos do Censo Escolar e dos indicadores do Inep filtrados para a rede estadual, além da renda da PNAD para toda a UF. Essa segunda junção é **muitos para um**. `N` e `MEDIANA` continuam sendo do recorte completo. Os campos com sufixo `_AMOSTRA` permitem conferir a amostra, inclusive quando houver zero notas sorteadas em algum recorte.


In [ ]:
# @title
chaves_territorio = ['ANO', 'UF_ESCOLA']
chaves_resultado = ['ANO', 'UF_ESCOLA', 'AREA']

amostra_estadual = amostra[amostra.REDE.eq('Estadual')]
resumo_amostra = amostra_estadual.groupby(chaves_resultado, observed=True).NOTA.agg(
    N_AMOSTRA='size', MEDIANA_AMOSTRA='median').reset_index()
base_estadual = enem_estadual_completo.merge(resumo_amostra, on=chaves_resultado, how='left', validate='one_to_one')
grupos = enem_redes[enem_redes.REDE.eq('Estadual')].copy()
grupos['SOMA_NOTAS'] = grupos.N * grupos.MEDIA
conferencia = grupos.groupby(chaves_resultado, as_index=False).agg(N_ANTERIOR=('N', 'sum'), SOMA_NOTAS=('SOMA_NOTAS', 'sum'))
conferencia = base_estadual.merge(conferencia, on=chaves_resultado, how='left', validate='one_to_one')
assert conferencia.N.eq(conferencia.N_ANTERIOR).all()
assert np.allclose(conferencia.MEDIA, conferencia.SOMA_NOTAS / conferencia.N_ANTERIOR)
censo_rede = ler('censo_uf_rede_2020_2024.csv')
for contexto in [censo_rede[censo_rede.REDE.eq('Estadual')].drop(columns='REDE'), renda_uf,
                 formacao[formacao.REDE.eq('Estadual')].drop(columns='REDE'),
                 jornada[jornada.REDE.eq('Estadual')].drop(columns='REDE'),
                 rendimento[rendimento.REDE.eq('Estadual')].drop(columns='REDE')]:
    base_estadual = base_estadual.merge(contexto, on=chaves_territorio, how='left',
        validate='many_to_one', indicator=True)
    assert base_estadual._merge.eq('both').all()
    base_estadual = base_estadual.drop(columns='_merge')
base_estadual['N_AMOSTRA'] = base_estadual.N_AMOSTRA.fillna(0).astype(int)
base_estadual['REDE'] = 'Estadual'
base_estadual['AMPLITUDE_INTERQUARTIL'] = base_estadual.Q3 - base_estadual.Q1
assert len(base_estadual) == 675 and not base_estadual.duplicated(chaves_resultado).any()
salvar(base_estadual, 'base_rede_estadual_2020_2024.csv')
print('Rede estadual: 675 resultados completos. Menor número de elegíveis:', base_estadual.N.min())

Rede estadual: 675 resultados completos. Menor número de elegíveis: 816


### A chave existe. Os indicadores têm informação suficiente?

O cruzamento preserva as 675 observações dos dois recortes. A auditoria abaixo separa chave correspondente de valor preenchido. Contamos ausentes por variável em **UF × ano**, sem repetir o contexto cinco vezes. Recalculamos também internet, laboratório e tempo integral a partir das escolas. Para os recursos, cada escola com resposta vale uma unidade. Para a jornada, o numerador soma matrículas integrais e o denominador soma matrículas de ensino médio. A oferta de um recurso não mede acesso efetivo, qualidade ou uso.

Os indicadores educacionais descrevem o ensino médio por rede. O Grupo 1 da formação docente diz respeito à adequação da formação à disciplina lecionada no ensino médio, sem separar áreas do ENEM nesta extração. A renda PNAD é média domiciliar per capita da UF, com coeficiente de variação disponível. O ajuste de preços usa IPCA nacional. A cobertura do ENEM é por área e inclui participantes de composição variável. Nenhuma dessas junções identifica exposição individual.

In [ ]:
# @title
RECURSOS = {'IN_AGUA_POTAVEL': 'Água potável', 'IN_ENERGIA_REDE_PUBLICA': 'Energia da rede',
    'IN_ESGOTO_REDE_PUBLICA': 'Esgoto da rede', 'IN_REFEITORIO': 'Refeitório',
    'IN_SALA_DIRETORIA': 'Sala da direção', 'IN_SALA_ATENDIMENTO_ESPECIAL': 'Atendimento especial',
    'IN_COMPUTADOR': 'Computador', 'IN_ALIMENTACAO': 'Alimentação',
    'IN_INTERNET_ALUNOS': 'Internet para alunos', 'IN_BIBLIOTECA_SALA_LEITURA': 'Biblioteca / leitura',
    'IN_LABORATORIO_CIENCIAS': 'Laboratório de ciências', 'IN_LABORATORIO_INFORMATICA': 'Laboratório de informática',
    'IN_QUADRA_ESPORTES': 'Quadra de esportes', 'IN_BANHEIRO_PNE': 'Banheiro acessível'}

auditoria_preenchimento = []
auditoria_denominadores = []
for universo, base, cadastro in [('Todas as redes', base_integrada, escolas),
                                 ('Rede estadual', base_estadual, escolas[escolas.REDE.eq('Estadual')])]:
    contexto = base.drop_duplicates(chaves_territorio).set_index(chaves_territorio).sort_index()
    colunas_contexto = [coluna for coluna in contexto if coluna in renda_uf.columns
                        or coluna in censo_uf.columns or coluna in formacao.columns
                        or coluna in jornada.columns or coluna in rendimento.columns]
    for ano, d in contexto.groupby(level='ANO'):
        auditoria_preenchimento.append(d[colunas_contexto].isna().sum().rename('N_AUSENTES')
                                      .rename_axis('VARIAVEL').reset_index()
                                      .assign(UNIVERSO=universo, ANO=ano, N_UFS=len(d)))
    grupo = cadastro.groupby(chaves_territorio, observed=True)
    assert not cadastro.duplicated(['ANO', 'CO_ENTIDADE']).any()
    assert grupo.size().eq(contexto.N_ESCOLAS_EM).all()
    total_matriculas = grupo.QT_MAT_MED.sum(min_count=1)
    integral = grupo.QT_MAT_MED_INT.sum(min_count=1)
    np.testing.assert_allclose(100 * integral / total_matriculas, contexto.PCT_MATRICULAS_INTEGRAL_EM)
    for recurso in RECURSOS:
        n_validas = grupo[recurso].count()
        assert n_validas.eq(contexto['N_INFO_' + recurso]).all()
        np.testing.assert_allclose(100 * grupo[recurso].mean(), contexto['PCT_' + recurso])
        auditoria_denominadores.append(pd.DataFrame({
            'N_ESCOLAS': grupo.size(), 'N_VALIDAS': n_validas,
            'PCT_RESPOSTA': 100 * n_validas / grupo.size()
        }).reset_index().assign(UNIVERSO=universo, RECURSO=recurso))
auditoria_preenchimento = pd.concat(auditoria_preenchimento, ignore_index=True)
auditoria_denominadores = pd.concat(auditoria_denominadores, ignore_index=True)
salvar(auditoria_preenchimento, 'auditoria_preenchimento.csv')
salvar(auditoria_denominadores, 'auditoria_denominadores_censo.csv')
display(auditoria_preenchimento.groupby(['UNIVERSO', 'ANO']).N_AUSENTES.sum().unstack())
if auditoria_denominadores.PCT_RESPOSTA.lt(100).any():
    display(auditoria_denominadores.nsmallest(10, 'PCT_RESPOSTA').round(2))
else:
    print('Nas tabelas preparadas, os recursos auditados têm respostas válidas em todas as escolas do recorte.')
print('Os 14 recursos e a proporção de matrículas integrais foram conferidos a partir do Censo.')

ANO,2020,2021,2022,2023,2024
UNIVERSO,,,,,
Rede estadual,0,0,0,0,0
Todas as redes,0,0,0,0,0


Nas tabelas preparadas, os recursos auditados têm respostas válidas em todas as escolas do recorte.
Os 14 recursos e a proporção de matrículas integrais foram conferidos a partir do Censo.


### Como ficaram as bases integradas?

As prévias mostram as primeiras cinco linhas após ordenar por UF, área e ano, o que ajuda a ler a sequência de edições dentro da mesma UF e área. Selecionamos colunas de notas, oferta e renda para caber na tela. O painel de contagem por edição confirma a cobertura de todas as redes e do recorte estadual. As duas tabelas completas são gravadas em `base_territorial_2020_2024.csv` e `base_rede_estadual_2020_2024.csv`.

In [ ]:
# @title
print(f'Base territorial · {len(base_integrada):,} linhas × {base_integrada.shape[1]} colunas')
display(base_integrada.sort_values(['UF_ESCOLA', 'AREA', 'ANO']).head()[
    ['ANO', 'UF_ESCOLA', 'AREA', 'N', 'MEDIANA', 'PCT_MATRICULAS_INTEGRAL_EM',
     'RENDA_PC_REAIS_2024_IPCA', 'PCT_DOCENTES_ADEQUADOS_EM']
].rename(columns={
    'N': 'Notas elegíveis', 'MEDIANA': 'Mediana (pontos)', 'PCT_MATRICULAS_INTEGRAL_EM': 'Tempo integral (%)',
    'RENDA_PC_REAIS_2024_IPCA': 'Renda por pessoa (R$ de 2024)', 'PCT_DOCENTES_ADEQUADOS_EM': 'Formação adequada (%)'
}).round(2))

print(f'Base estadual · {len(base_estadual):,} linhas × {base_estadual.shape[1]} colunas')
display(base_estadual.sort_values(['UF_ESCOLA', 'AREA', 'ANO']).head()[
    ['ANO', 'UF_ESCOLA', 'AREA', 'N', 'MEDIANA', 'N_AMOSTRA', 'MEDIANA_AMOSTRA',
     'PCT_IN_INTERNET_ALUNOS']
].rename(columns={
    'N': 'Notas elegíveis', 'MEDIANA': 'Mediana (pontos)', 'N_AMOSTRA': 'Notas na amostra',
    'MEDIANA_AMOSTRA': 'Mediana amostral', 'PCT_IN_INTERNET_ALUNOS': 'Internet nas escolas (%)'
}).round(2))

linhas_por_edicao = pd.concat({
    'Todas as redes': base_integrada.groupby('ANO').size(),
    'Rede estadual': base_estadual.groupby('ANO').size()
}, axis=1)
display(linhas_por_edicao.rename_axis('Edição').style.set_caption('Linhas por edição em cada base'))


Base territorial · 675 linhas × 58 colunas


,ANO,UF_ESCOLA,AREA,Notas elegíveis,Mediana (pontos),Tempo integral (%),Renda por pessoa (R$ de 2024),Formação adequada (%)
0,2020,AC,Ciências Humanas,2511,486.50,10.10,1135.63,52.6
135,2021,AC,Ciências Humanas,2396,495.40,10.45,1046.19,49.4
270,2022,AC,Ciências Humanas,3222,502.15,12.81,1112.36,49.3
405,2023,AC,Ciências Humanas,3109,503.70,17.71,1111.51,45.5
540,2024,AC,Ciências Humanas,6466,477.65,22.51,1250.00,44.2


Base estadual · 675 linhas × 58 colunas


,ANO,UF_ESCOLA,AREA,Notas elegíveis,Mediana (pontos),Notas na amostra,Mediana amostral,Internet nas escolas (%)
27,2020,AC,Ciências Humanas,1873,470.5,186,466.25,19.74
162,2021,AC,Ciências Humanas,1749,480.3,168,469.70,20.76
297,2022,AC,Ciências Humanas,2435,492.0,241,494.10,24.51
432,2023,AC,Ciências Humanas,2359,490.0,235,498.70,30.30
567,2024,AC,Ciências Humanas,5467,469.4,546,472.15,28.41


,Todas as redes,Rede estadual
Edição,,
2020,135,135
2021,135,135
2022,135,135
2023,135,135
2024,135,135


## 6. Oferta escolar e associações com o desempenho

### Infraestrutura escolar por território

Escolha um recurso, ano, rede e localização. Compare as regiões ou as UFs de uma região. Cada barra parte de zero e representa a proporção de escolas de ensino médio com o recurso entre as escolas que responderam ao item. O cursor mostra quantas escolas têm o recurso e quantas responderam.

A oferta de um recurso não descreve diretamente o acesso de cada participante. O Censo Escolar e o ENEM são integrados no nível territorial e as escolas têm o mesmo peso nesta proporção.


In [ ]:
# @title
def mostrar_infraestrutura(ano, nivel, regiao, rede, localizacao, recurso, ver_tabela):
    d = recortar(escolas[escolas.ANO.eq(ano)], regiao, rede)
    if localizacao != 'Todas':
        d = d[d.LOCALIZACAO.eq(localizacao)]
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    coluna = next(c for c, nome in RECURSOS.items() if nome == recurso)
    r = d.groupby(grupo, observed=True)[coluna].agg(COM_RECURSO='sum', N_VALIDAS='count').reset_index()
    r = r[r.N_VALIDAS.gt(0)].copy()
    if r.empty:
        print('Nenhuma escola respondeu ao item neste recorte.')
        return
    r['PCT'] = 100 * r.COM_RECURSO / r.N_VALIDAS
    r = r.sort_values('PCT')
    fig = go.Figure(go.Bar(x=r.PCT, y=r[grupo], orientation='h', marker_color='#007763',
                           text=r.PCT, texttemplate='%{text:.1f}%', textposition='outside', cliponaxis=False,
                           customdata=r[['COM_RECURSO', 'N_VALIDAS']].to_numpy(),
                           hovertemplate='<b>%{y}</b><br>%{x:.2f}% com o recurso'
                                         '<br>%{customdata[0]:,.0f} de %{customdata[1]:,.0f} escolas com resposta'
                                         '<extra></extra>'))
    fig.update_xaxes(title='Escolas com o recurso (%)')
    fig.update_yaxes(title='Região da escola' if nivel == 'Regiões' else 'Estado da escola')
    fixar_percentual(fig)
    mostrar_figura(configurar_painel(fig, f'Em quais lugares há mais escolas com {recurso.lower()}?',
        f'{ano} · {rede} · {localizacao} · {regiao}', max(420, 27 * len(r) + 150)))
    tabela_opcional(r, ver_tabela)

painel(mostrar_infraestrutura, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Brasil (UF informada)'] + REGIOES_ESTUDO, 'Brasil (UF informada)', 'Região:'),
    rede=escolher(['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada'], 'Públicas', 'Rede:'),
    localizacao=escolher(['Todas', 'Urbana', 'Rural'], 'Todas', 'Localização:'),
    recurso=escolher(list(RECURSOS.values()), 'Internet para alunos', 'Recurso:'),
    ver_tabela=widgets.Checkbox(value=False, description='Ver contagens'))

### Visão geral dos recursos escolares

Compare os recursos do Censo Escolar no mesmo painel, por região ou por UF da região selecionada. Cada célula mostra a porcentagem de escolas de ensino médio com o recurso **entre as escolas que responderam ao item**. O número de respostas pode variar entre os recursos de um mesmo território. Passe o mouse para conferir as duas contagens.

A legenda de cor acompanha o intervalo dos percentuais exibidos. Compare os valores impressos ao mudar o ano, a rede ou a localização. O painel descreve a oferta na rede e na localização selecionadas. Cada escola tem o mesmo peso no cálculo.

In [ ]:
def mostrar_visao_geral_recursos(ano, nivel, regiao, rede, localizacao):
    por_regiao = nivel == 'Regiões'
    d = recortar(escolas[escolas.ANO.eq(ano)], 'Brasil (UF informada)' if por_regiao else regiao, rede)
    if localizacao != 'Todas':
        d = d[d.LOCALIZACAO.eq(localizacao)]
    if d.empty:
        print('Nenhuma escola neste recorte.')
        return
    grupo = 'REGIAO' if por_regiao else 'UF_ESCOLA'
    colunas = list(RECURSOS)
    grupos = d.groupby(grupo, observed=True)[colunas]
    numerador = grupos.sum(min_count=1).T
    denominador = grupos.count().T
    ordem = [r for r in REGIOES_ESTUDO if r in numerador.columns] if por_regiao else sorted(numerador.columns)
    numerador, denominador = numerador[ordem], denominador[ordem]
    porcentagens = 100 * numerador.div(denominador.where(denominador.gt(0))).to_numpy(float)
    if not np.isfinite(porcentagens).any():
        print('Nenhuma resposta válida aos itens neste recorte.')
        return

    minimo, maximo = limites_cores_percentuais(porcentagens)
    fig = go.Figure(go.Heatmap(
        x=ordem, y=[RECURSOS[c] for c in colunas], z=porcentagens,
        customdata=np.stack([numerador.to_numpy(float), denominador.to_numpy(float)], axis=-1),
        text=[[f'{v:.1f}%'.replace('.', ',') if np.isfinite(v) else '' for v in linha] for linha in porcentagens],
        texttemplate='%{text}', textfont=dict(size=12),
        colorscale=ESCALA_SEQUENCIAL, zmin=minimo, zmax=maximo,
        xgap=2, ygap=2, hoverongaps=False,
        colorbar=dict(title='Escolas com o recurso', ticksuffix='%', nticks=6),
        hovertemplate=('<b>%{y} · %{x}</b><br>Com o recurso: %{z:.1f}%'
                       '<br>%{customdata[0]:,.0f} de %{customdata[1]:,.0f} escolas com resposta<extra></extra>'),
    ))
    fig.update_xaxes(title='Recurso da escola')
    fig.update_yaxes(title='Região da escola' if por_regiao else 'Estado da escola', autorange='reversed', automargin=True)
    recorte = nivel if por_regiao else f'{nivel} · {regiao}'
    mostrar_figura(configurar_painel(fig, 'Quais recursos estão disponíveis nas escolas?',
                                   f'{ano} · {recorte} · {rede} · {localizacao}',
                                   altura=max(440, 30 * len(colunas) + 170)))


painel(mostrar_visao_geral_recursos,
       ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
       nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
       regiao=escolher(REGIOES_ESTUDO, 'Nordeste', 'Região (UFs):'),
       rede=escolher(['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada'], 'Públicas', 'Rede:'),
       localizacao=escolher(['Todas', 'Urbana', 'Rural'], 'Todas', 'Localização:'))

### Evolução da oferta escolar

Ver a infraestrutura em apenas um ano não permite acompanhar sua evolução. Aqui seguimos um recurso ou o percentual de matrículas em tempo integral entre 2020 e 2024, com filtros de rede, região e localização. O tempo integral usa matrículas como peso; os demais recursos usam escolas com resposta válida. A diferença de denominadores fica indicada no eixo.

O crescimento da oferta não demonstra sozinho um efeito nas notas. O Censo registra a situação da escola na referência anual; não sabemos quanto tempo cada participante esteve exposto ao recurso. Além disso, o programa federal Escola em Tempo Integral foi criado em 2023, enquanto iniciativas estaduais e escolas de tempo integral já existiam. Não chamamos todo aumento de adesão a esse programa.

In [ ]:
# @title
def mostrar_oferta(indicador, nivel, regiao, rede, localizacao, intervalo):
    d = recortar(escolas, regiao, rede)
    d = d[d.ANO.between(*intervalo)]
    if localizacao != 'Todas':
        d = d[d.LOCALIZACAO.eq(localizacao)]
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    if indicador == 'Tempo integral':
        r = d.groupby(['ANO', grupo], observed=True)[['QT_MAT_MED', 'QT_MAT_MED_INT']].sum(min_count=1)
        r['VALOR'] = 100 * r.QT_MAT_MED_INT / r.QT_MAT_MED.replace(0, np.nan)
        numerador, denominador = 'QT_MAT_MED_INT', 'QT_MAT_MED'
        eixo = 'Matrículas do ensino médio em tempo integral (%)'
    else:
        coluna = next(c for c, nome in RECURSOS.items() if nome == indicador)
        r = d.groupby(['ANO', grupo], observed=True)[coluna].agg(COM_RECURSO='sum', N_VALIDAS='count')
        r['VALOR'] = 100 * r.COM_RECURSO / r.N_VALIDAS.replace(0, np.nan)
        numerador, denominador = 'COM_RECURSO', 'N_VALIDAS'
        eixo = 'Escolas com o recurso (%)'
    r = r.reset_index()
    if r.VALOR.notna().sum() == 0:
        print('Sem respostas válidas para este recorte.')
        return
    fig = px.line(r, x='ANO', y='VALOR', color=grupo, markers=True,
                  labels={'ANO': 'Ano do Censo Escolar', 'VALOR': eixo, grupo: nivel,
                          numerador: 'Numerador', denominador: 'Denominador'},
                  hover_data={'VALOR': ':.2f', numerador: ':,.0f', denominador: ':,.0f'},
                  color_discrete_map=CORES_REGIOES if nivel == 'Regiões' else None)
    fig.update_xaxes(tickvals=list(range(intervalo[0], intervalo[1] + 1)))
    fixar_percentual(fig, eixo='y')
    mostrar_figura(configurar_painel(fig, f'Como a oferta mudou ao longo dos anos? · {indicador}',
                                   f'{regiao} · {rede} · {localizacao}'))
    salvar(r, 'evolucao_oferta.csv')

painel(mostrar_oferta, indicador=escolher(['Tempo integral'] + list(RECURSOS.values()), 'Tempo integral', 'Indicador:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Brasil (UF informada)'] + REGIOES_ESTUDO, 'Brasil (UF informada)', 'Região:'),
    rede=escolher(['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada'], 'Públicas', 'Rede:'),
    localizacao=escolher(['Todas', 'Urbana', 'Rural'], 'Todas', 'Localização:'), intervalo=periodo())

In [ ]:
censo_est_2024 = base_estadual.query('ANO == 2024 and AREA == "Matemática"')
linhas_recursos = []
for recurso in ['PCT_IN_INTERNET_ALUNOS', 'PCT_IN_LABORATORIO_CIENCIAS', 'PCT_MATRICULAS_INTEGRAL_EM']:
    extremos = censo_est_2024.set_index('UF_ESCOLA')[recurso].dropna()
    linhas_recursos.append({'INDICADOR': recurso, 'UF_MENOR': extremos.idxmin(), 'MENOR_PCT': extremos.min(),
                            'UF_MAIOR': extremos.idxmax(), 'MAIOR_PCT': extremos.max()})
display(pd.DataFrame(linhas_recursos).round(1))
print('Rede estadual em 2024. A variação observada motiva examinar a associação com as notas das UFs.')

,INDICADOR,UF_MENOR,MENOR_PCT,UF_MAIOR,MAIOR_PCT
0,PCT_IN_INTERNET_ALUNOS,PA,3.1,ES,97.6
1,PCT_IN_LABORATORIO_CIENCIAS,RR,19.1,CE,88.2
2,PCT_MATRICULAS_INTEGRAL_EM,RR,4.0,PE,71.2


Rede estadual em 2024. A variação observada motiva examinar a associação com as notas das UFs.


### Associações territoriais

**Pergunta:** UFs com determinado indicador mais alto também tendem a ter maior desempenho? Cada ponto é uma UF, dentro de um mesmo ano e área. Os filtros alternam todas as redes e o recorte estadual. Cor identifica região; a correlação de Spearman resume se as posições dos estados nos dois eixos tendem a caminhar juntas.

O painel examina uma característica de cada vez. Uma associação entre renda e nota não demonstra um efeito individual da renda; uma associação com laboratório pode refletir diferenças de renda, rede, participação e outras condições. Não há ajuste estatístico nem teste causal aqui. Não usamos dezenas de p-valores para declarar “fatores comprovados”.

O gráfico à direita compara os anos para ver se a relação persiste. A formação adequada, o tempo integral, a jornada e o abandono representam dimensões diferentes da oferta. Aprovação, reprovação e abandono são relacionados entre si; não devem entrar todos automaticamente num modelo.

**Revisão:** Corrigir rótulos após recalcular o alvo.

In [ ]:
# @title
ATRIBUTOS = {
    'Renda por pessoa (R$ de 2024)': 'RENDA_PC_REAIS_2024_IPCA',
    'Matrículas em tempo integral (%)': 'PCT_MATRICULAS_INTEGRAL_EM',
    'Docentes com formação adequada (%)': 'PCT_DOCENTES_ADEQUADOS_EM',
    'Horas-aula por dia': 'HORAS_AULA_DIA_EM',
    'Abandono no ensino médio (%)': 'PCT_ABANDONO_EM',
    'Escolas com internet para alunos (%)': 'PCT_IN_INTERNET_ALUNOS',
    'Escolas com laboratório de ciências (%)': 'PCT_IN_LABORATORIO_CIENCIAS',
    'Escolas com biblioteca/leitura (%)': 'PCT_IN_BIBLIOTECA_SALA_LEITURA',
    'Escolas com água potável (%)': 'PCT_IN_AGUA_POTAVEL',
    'Escolas rurais (%)': 'PCT_ESCOLAS_RURAIS_EM'}

def correlacao_segura(d, coluna, alvo='MEDIANA'):
    pares = d[[coluna, alvo]].dropna()
    if len(pares) < 4 or pares[coluna].nunique() < 2 or pares[alvo].nunique() < 2:
        return np.nan
    return spearmanr(pares[coluna], pares[alvo]).statistic

def mostrar_associacao(ano, area, atributo, regiao, universo, medida, vista, ver_tabela):
    origem = base_integrada if universo == 'Todas as redes' else base_estadual
    d = origem[origem.AREA.eq(area)]
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    coluna = ATRIBUTOS[atributo]
    atual = d[d.ANO.eq(ano)].dropna(subset=[coluna, medida])
    nome = 'Mediana' if medida == 'MEDIANA' else 'Média'
    if vista == 'Dispersão entre UFs':
        if atual.empty:
            print('Nenhum par válido no recorte selecionado.')
            return
        rho = correlacao_segura(atual, coluna, medida)
        legenda_rho = f'{rho:+.2f}' if pd.notna(rho) else 'indisponível'
        fig = px.scatter(atual, x=coluna, y=medida, color='REGIAO', hover_name='UF_ESCOLA',
                         color_discrete_map=CORES_REGIOES,
                         labels={coluna: atributo, medida: 'Nota do meio (mediana, em pontos)' if medida == 'MEDIANA' else 'Nota média (pontos)',
                                 'REGIAO': 'Região', 'N': 'Notas elegíveis'},
                         hover_data={coluna: ':.2f', medida: ':.2f', 'N': ':,'})
        fig.update_traces(marker=dict(size=10, line=dict(color='white', width=1)))
        if coluna.startswith('PCT_'):
            fixar_percentual(fig)
        titulo = 'Como o indicador e a nota se relacionam entre os estados?'
        detalhe = f'{ano} · {len(atual)} UFs · correlação de Spearman: {legenda_rho}'
    else:
        historico = pd.DataFrame([
            {'ANO': a, 'SPEARMAN': correlacao_segura(g, coluna, medida),
             'N_UFS': len(g[[coluna, medida]].dropna())}
            for a, g in d.groupby('ANO')
        ])
        fig = px.line(historico, x='ANO', y='SPEARMAN', markers=True,
                      labels={'ANO': 'Ano do ENEM', 'SPEARMAN': 'Correlação de Spearman (−1 a +1)', 'N_UFS': 'UFs com par válido'},
                      hover_data={'SPEARMAN': ':+.2f', 'N_UFS': True})
        fig.add_hline(y=0, line_color='#6A7D92', line_dash='dot')
        fig.update_xaxes(tickvals=ANOS_ESTUDO)
        fig.update_yaxes(range=[-1.05, 1.05])
        titulo = 'A relação com a nota muda de um ano para outro?'
        detalhe = atributo
    mostrar_figura(configurar_painel(fig, titulo, f'{area} · {universo} · {regiao} · {detalhe}'))
    if vista == 'Dispersão entre UFs':
        tabela_opcional(atual[['UF_ESCOLA', coluna, medida, 'N']], ver_tabela)
    else:
        tabela_opcional(historico, ver_tabela)

painel(mostrar_associacao, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    atributo=escolher(list(ATRIBUTOS), 'Renda por pessoa (R$ de 2024)', 'Indicador:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'),
    medida=escolher(['MEDIANA', 'MEDIA'], 'MEDIANA', 'Medida:'),
    vista=escolher(['Dispersão entre UFs', 'Série de Spearman'], 'Dispersão entre UFs', 'Vista:'),
    ver_tabela=widgets.Checkbox(value=False, description='Ver tabela'))

### Estabilidade por área e edição

O mapa de calor reúne as cinco áreas sem somar suas notas. Cada célula é uma correlação entre UFs, para o ano e universo selecionados. A cor representa associação, **não importância de variável, porcentagem explicada ou efeito de política pública**. Os resultados completos de todos os anos serão salvos, incluindo quantas UFs participaram de cada cálculo.

Se uma relação aparecer em apenas um ano, merece cautela. Se duas características se movem juntas, um futuro modelo pode ter dificuldade em separar suas contribuições. Por isso a seleção de atributos usará justificativa, disponibilidade e redundância, além destes gráficos.

**Revisão:** Recalcular associações com o alvo estadual completo.

In [ ]:
# @title
associacoes = []
for universo, origem in [('Todas as redes', base_integrada), ('Rede estadual', base_estadual)]:
    for (ano, area), grupo in origem.groupby(['ANO', 'AREA'], observed=True):
        for nome, coluna in ATRIBUTOS.items():
            pares = grupo[[coluna, 'MEDIANA']].dropna()
            associacoes.append({'UNIVERSO': universo, 'ANO': ano, 'AREA': area, 'ATRIBUTO': nome,
                'SPEARMAN': correlacao_segura(grupo, coluna), 'N_UFS': len(pares)})
associacoes = pd.DataFrame(associacoes)
salvar(associacoes, 'associacoes_por_ano_area.csv')

def mostrar_matriz_associacoes(ano, universo):
    d = associacoes[associacoes.ANO.eq(ano) & associacoes.UNIVERSO.eq(universo)]
    m = d.pivot(index='ATRIBUTO', columns='AREA', values='SPEARMAN').reindex(index=ATRIBUTOS, columns=AREAS_ESTUDO)
    n = d.pivot(index='ATRIBUTO', columns='AREA', values='N_UFS').reindex_like(m)
    rotulos = ['Linguagens', 'Humanas', 'Natureza', 'Matemática', 'Redação']
    fig = go.Figure(go.Heatmap(
        x=rotulos, y=m.index, z=m.to_numpy(), customdata=n.to_numpy(),
        text=[[f'{v:+.2f}'.replace('.', ',') if np.isfinite(v) else '—'
               for v in linha] for linha in m.to_numpy()],
        texttemplate='%{text}', colorscale=ESCALA_DIVERGENTE, zmin=-1, zmax=1,
        colorbar=dict(title='Correlação de Spearman'), xgap=1, ygap=1,
        hovertemplate='<b>%{y}</b><br>%{x} · Spearman %{z:+.2f}<br>%{customdata} UFs<extra></extra>'))
    fig.update_xaxes(title='Área da prova')
    fig.update_yaxes(title='Indicador do território', autorange='reversed')
    mostrar_figura(configurar_painel(fig, 'Quais indicadores acompanham as notas em cada área?', f'{ano} · {universo}', 515))

painel(mostrar_matriz_associacoes, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'))

### Diagnóstico conjunto de renda, notas e recursos

**Pergunta:** em quais UFs convivem menor desempenho observado, menor renda e menor disponibilidade de um recurso? O painel usa a rede estadual para aproximar o recorte da atuação das secretarias. Renda permanece como contexto da população da UF. Tamanho do ponto indica a proporção de escolas estaduais sem o recurso, entre respostas válidas.

As linhas tracejadas são medianas entre UFs, usadas apenas para orientar a leitura; não são metas oficiais nem um critério de elegibilidade. As UFs abaixo delas não recebem automaticamente uma classificação de prioridade. Primeiro precisamos conferir cobertura, desigualdades internas, demanda local e evidências de outras avaliações, como o Saeb.

Um território com notas baixas e muitas carências pode motivar uma visita técnica. Um com infraestrutura mais disponível e notas baixas pode motivar investigação de outras dificuldades. Um com resultados altos em contexto desfavorável pode ser selecionado para estudo de práticas. Nenhuma dessas situações, sozinha, demonstra qual intervenção funcionará.

**Revisão:** Corrigir rótulo do diagnóstico territorial.

In [ ]:
# @title

def mostrar_diagnostico(ano, area, recurso, regiao):
    d = base_estadual[base_estadual.ANO.eq(ano) & base_estadual.AREA.eq(area)].copy()
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    coluna = 'PCT_' + next(c for c, nome in RECURSOS.items() if nome == recurso)
    d['PCT_SEM_RECURSO'] = 100 - d[coluna]
    validas = d.dropna(subset=['RENDA_PC_REAIS_2024_IPCA', 'MEDIANA', 'PCT_SEM_RECURSO'])
    if validas.empty:
        print('Sem informações simultâneas de renda, nota e recurso para este recorte.')
        return
    fig = px.scatter(validas, x='RENDA_PC_REAIS_2024_IPCA', y='MEDIANA', color='REGIAO',
                     size='PCT_SEM_RECURSO', size_max=34, hover_name='UF_ESCOLA',
                     color_discrete_map=CORES_REGIOES,
                     labels={'RENDA_PC_REAIS_2024_IPCA': 'Renda por pessoa na UF (R$ de 2024)',
                             'MEDIANA': 'Nota do meio (mediana, em pontos)', 'REGIAO': 'Região',
                             'PCT_SEM_RECURSO': 'Escolas sem o recurso (%)', 'N': 'Notas elegíveis'},
                     hover_data={'RENDA_PC_REAIS_2024_IPCA': ':,.0f', 'MEDIANA': ':.1f',
                                 'PCT_SEM_RECURSO': ':.1f', 'N': ':,'})
    fig.update_traces(marker=dict(sizemin=5, opacity=.86, line=dict(color='white', width=1)))
    fig.add_vline(x=validas.RENDA_PC_REAIS_2024_IPCA.median(), line_dash='dot', line_color='#6A7D92')
    fig.add_hline(y=validas.MEDIANA.median(), line_dash='dot', line_color='#6A7D92')
    mostrar_figura(configurar_painel(fig, 'Como renda, notas e recursos das escolas se combinam?',
                                   f'Rede estadual · {ano} · {area} · {regiao}'))
    print(f'Pontos maiores indicam mais escolas sem {recurso.lower()}, em proporção. '
          'As linhas mostram as medianas entre as UFs exibidas.')
    salvar(d[['ANO', 'UF_ESCOLA', 'AREA', 'N', 'N_AMOSTRA', 'MEDIANA',
              'RENDA_PC_REAIS_2024_IPCA', 'PCT_SEM_RECURSO']], 'diagnostico_territorial.csv')

painel(mostrar_diagnostico, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Ciências da Natureza', 'Área:'),
    recurso=escolher(list(RECURSOS.values()), 'Laboratório de ciências', 'Recurso:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'))

## 7. Quais relações são estáveis?

### Redundância dos atributos candidatos

A escolha começa pela pergunta, não pela quantidade de colunas. Pretendemos avaliar se condições observadas ajudam a estimar o desempenho futuro e a selecionar territórios para investigação. Uma primeira lista reúne renda, oferta de tempo integral, formação docente, abandono, internet e laboratório de ciências. O desempenho anterior será uma referência indispensável.

| Informação | Papel | Decisão inicial |
|---|---|---|
| Mediana por área | Resultado a estimar | Manter o mesmo alvo entre UFs; média como análise de sensibilidade |
| Mediana do ano anterior | Histórico e referência simples | Comparar o futuro modelo com “repetir a nota anterior” |
| Renda real por pessoa | Contexto econômico | Manter uma versão de preços; não incluir simultaneamente renda nominal e deflacionada |
| Matrículas integrais | Oferta de jornada ampliada | Candidata; não identifica participação individual em programa |
| Formação docente adequada | Condição de ensino | Candidata; indicador agregado do ensino médio |
| Abandono | Permanência escolar | Candidata; revisar mudanças administrativas durante a pandemia |
| Internet e laboratório de ciências | Dimensões específicas da infraestrutura | Candidatas; presença não mede qualidade do uso |
| Horas-aula, biblioteca, água e outras condições | Alternativas e diagnóstico | Conservar na base ampla; revisar redundância antes do treinamento |
| N, cobertura e respostas válidas | Qualidade e composição | Guardar para auditoria; não usar como “nota de qualidade” |
| FUNDEB de dezembro | Contexto financeiro parcial | Excluir da lista principal até obter série anual de execução adequada |
| Perfis feitos com as notas do mesmo ano | Análise descritiva individual | Não usar para prever as próprias notas que os definiram |

O gráfico seguinte verifica redundância entre candidatos e jornada, usando uma linha por UF no ano escolhido. Não repetimos cada UF cinco vezes por causa das áreas. Correlação alta é um sinal para avaliar alternativas; não elimina automaticamente uma variável nem determina importância.

Esta lista é uma hipótese de trabalho. Confrontamos a justificativa qualitativa com associação, redundância e influência de UFs. O ganho preditivo fica para a Etapa 3. Correlação de duas variáveis não basta para escolher entradas nem uma família de modelos.

In [ ]:
# @title

CANDIDATOS = ['RENDA_PC_REAIS_2024_IPCA', 'PCT_MATRICULAS_INTEGRAL_EM',
              'PCT_DOCENTES_ADEQUADOS_EM', 'PCT_ABANDONO_EM',
              'PCT_IN_INTERNET_ALUNOS', 'PCT_IN_LABORATORIO_CIENCIAS']
NOMES_ATRIBUTOS = {coluna: nome for nome, coluna in ATRIBUTOS.items()}

def mostrar_redundancia(ano, universo):
    origem = base_integrada if universo == 'Todas as redes' else base_estadual
    d = origem[origem.ANO.eq(ano)].drop_duplicates('UF_ESCOLA')
    colunas = CANDIDATOS + ['HORAS_AULA_DIA_EM']
    m = d[colunas].corr(method='spearman')
    n = d[colunas].notna().astype(int)
    n_pares = n.T @ n
    nomes = ['Renda', 'Tempo integral', 'Formação docente', 'Abandono', 'Internet', 'Lab. ciências', 'Horas-aula']
    fig = go.Figure(go.Heatmap(
        z=m.to_numpy(), x=nomes, y=nomes, customdata=n_pares.to_numpy(),
        text=[[f'{v:+.2f}'.replace('.', ',') if np.isfinite(v) else '—'
               for v in linha] for linha in m.to_numpy()],
        texttemplate='%{text}', zmin=-1, zmax=1, colorscale=ESCALA_DIVERGENTE,
        xgap=1, ygap=1, colorbar=dict(title='Correlação de Spearman'),
        hovertemplate='<b>%{y} × %{x}</b><br>Spearman: %{z:+.2f}<br>UFs: %{customdata}<extra></extra>'))
    fig.update_xaxes(title='Indicador')
    fig.update_yaxes(title='Indicador comparado', autorange='reversed')
    mostrar_figura(configurar_painel(fig, 'Quais indicadores variam de forma parecida entre estados?', f'{ano} · {universo}', 535))
    pares = m.where(np.triu(np.ones(m.shape, dtype=bool), k=1)).stack()
    if not pares.empty:
        par = pares.abs().idxmax()
        rotulos = dict(zip(colunas, nomes))
        print(f'Maior associação absoluta no recorte: {rotulos[par[0]]} e {rotulos[par[1]]} '
              f'(Spearman {pares.loc[par]:+.2f}).')

painel(mostrar_redundancia, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'))

In [ ]:
from scipy.stats import pearsonr, false_discovery_control

CONTEXTO = CANDIDATOS
ROTULOS = ['Renda', 'Tempo integral', 'Formação docente', 'Abandono', 'Internet', 'Lab. ciências']
ROTULO = dict(zip(CONTEXTO, ROTULOS))
SEMENTE_TESTES = 42


### Forma das associações e influência das UFs

Spearman sozinho não responde a toda a pergunta. **Pearson** resume alinhamento em uma reta; **Spearman** compara ordenações, aceitando relações crescentes ou decrescentes curvas; **correlação de distâncias** pode detectar outras dependências, inclusive formas em U. Esta última não tem sinal e seu valor amostral positivo não comprova associação.

Calculamos as três medidas em cada ano e área. Para avaliar sensibilidade a uma UF, repetimos Spearman retirando uma UF por vez: o intervalo mínimo–máximo é influência, não intervalo de confiança. Não excluímos um estado porque altera o resultado.

Para Pearson e distância, usamos 999 permutações do resultado **dentro de cada região**, preservando a composição regional. Essa referência pergunta se há mais associação do que a obtida ao reorganizar as UFs dentro das regiões. Ela pressupõe que essas trocas sejam razoáveis sob a hipótese nula; dependência espacial e diferenças internas limitam essa interpretação. Aplicamos correção Benjamini–Yekutieli sobre os 300 testes (6 atributos × 5 áreas × 5 anos × 2 medidas). Isso reduz falsas descobertas sob dependência entre testes, mas não corrige um teste cuja hipótese de permutação seja inadequada.

Um teste de distância significativo não prova que a relação seja especificamente não linear: relações lineares também são dependentes. A utilidade preditiva de permitir curvas fica para a Etapa 3. Fontes: [Pearson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), [correlação de distâncias — artigo original](https://arxiv.org/abs/0803.4101), [correção de múltiplos testes](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html).

In [ ]:
# @title

def matriz_distancias(x):
    distancias = abs(x[:, None] - x[None, :])
    return distancias - distancias.mean(axis=0) - distancias.mean(axis=1)[:, None] + distancias.mean()

def medir_associacoes(d, coluna, repeticoes=999):
    d = d[['UF_ESCOLA', 'REGIAO', coluna, 'MEDIANA']].dropna().reset_index(drop=True)
    x, y = d[coluna].to_numpy(float), d.MEDIANA.to_numpy(float)
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return dict(N=len(x), PEARSON=np.nan, SPEARMAN=np.nan, DISTANCIA=np.nan,
                    P_PEARSON=np.nan, P_DISTANCIA=np.nan, RHO_MIN=np.nan, RHO_MAX=np.nan)
    pearson = pearsonr(x, y).statistic
    spearman = spearmanr(x, y).statistic
    a, b = matriz_distancias(x), matriz_distancias(y)
    denominador = np.sqrt(np.mean(a*a) * np.mean(b*b))
    produto = np.mean(a*b)
    distancia = np.sqrt(max(0, produto / denominador))
    rng = np.random.default_rng(SEMENTE_TESTES)
    regioes = [g.index.to_numpy() for _, g in d.groupby('REGIAO')]
    excede_p = excede_d = 0
    xc, yc = x - x.mean(), y - y.mean()
    norma = np.linalg.norm(xc) * np.linalg.norm(yc)
    for _ in range(repeticoes):
        ordem = np.arange(len(d))
        for indices in regioes:
            ordem[indices] = rng.permutation(indices)
        excede_p += abs(np.dot(xc, yc[ordem]) / norma) >= abs(pearson) - 1e-12
        excede_d += np.mean(a * b[np.ix_(ordem, ordem)]) >= produto - 1e-12
    influencia = [spearmanr(np.delete(x, i), np.delete(y, i)).statistic for i in range(len(d))]
    return dict(N=len(d), PEARSON=pearson, SPEARMAN=spearman, DISTANCIA=distancia,
        P_PEARSON=(excede_p+1)/(repeticoes+1), P_DISTANCIA=(excede_d+1)/(repeticoes+1),
        RHO_MIN=np.nanmin(influencia), RHO_MAX=np.nanmax(influencia))

associacoes_robustas = []
for (ano, area), d in base_estadual.groupby(['ANO', 'AREA'], observed=True):
    for coluna in CONTEXTO:
        associacoes_robustas.append(dict(ANO=ano, AREA=area, ATRIBUTO=coluna, **medir_associacoes(d, coluna)))
associacoes_robustas = pd.DataFrame(associacoes_robustas)
p = associacoes_robustas[['P_PEARSON', 'P_DISTANCIA']].to_numpy().ravel()
ajuste = np.full(len(p), np.nan)
ajuste[np.isfinite(p)] = false_discovery_control(p[np.isfinite(p)], method='by')
associacoes_robustas[['Q_PEARSON_BY', 'Q_DISTANCIA_BY']] = ajuste.reshape(-1, 2)
salvar(associacoes_robustas, 'associacoes_lineares_nao_lineares.csv')
n_testes = np.isfinite(p).sum()
n_abaixo = (ajuste < .05).sum()
print(f'{n_abaixo} de {n_testes} testes tiveram p ajustado abaixo de 0,05. '
      'A leitura depende da hipótese de permutação dentro das regiões.')

def mostrar_formas_associacao(ano, area, atributo, forma, vista):
    coluna = CONTEXTO[ROTULOS.index(atributo)]
    d = base_estadual.loc[base_estadual.ANO.eq(ano) & base_estadual.AREA.eq(area)].dropna(subset=[coluna, 'MEDIANA'])
    if len(d) < 4 or d[coluna].nunique() < 2 or d.MEDIANA.nunique() < 2:
        print(f'Sem variação ou pares suficientes: {len(d)} UFs.')
        return
    r = associacoes_robustas.query('ANO == @ano and AREA == @area and ATRIBUTO == @coluna').iloc[0]
    if vista == 'Dispersão e ajuste':
        fig = px.scatter(d, x=coluna, y='MEDIANA', hover_name='UF_ESCOLA', color='REGIAO',
                         color_discrete_map=CORES_REGIOES,
                         labels={coluna: NOMES_ATRIBUTOS.get(coluna, atributo), 'MEDIANA': 'Nota do meio (mediana, em pontos)',
                                 'REGIAO': 'Região', 'N': 'Notas elegíveis'},
                         hover_data={coluna: ':.2f', 'MEDIANA': ':.1f', 'N': ':,'})
        fig.update_traces(marker=dict(size=10, line=dict(color='white', width=1)))
        z = (d[coluna] - d[coluna].mean()) / d[coluna].std()
        eixo = np.linspace(z.min(), z.max(), 150)
        grau = min(1 if forma == 'Reta' else 2, d[coluna].nunique() - 1)
        coef = np.polyfit(z, d.MEDIANA, grau)
        fig.add_scatter(x=eixo * d[coluna].std() + d[coluna].mean(), y=np.polyval(coef, eixo),
                        mode='lines', name=f'{forma} descritiva', line=dict(color='#243747', dash='dash'),
                        hovertemplate='Indicador: %{x:.2f}<br>Nota ajustada: %{y:.1f}<extra></extra>')
        if coluna.startswith('PCT_'):
            fixar_percentual(fig)
        titulo = 'Que forma tem a relação entre o indicador e a nota?'
    else:
        medidas = ['Pearson', 'Spearman', 'Distâncias']
        valores = [r.PEARSON, r.SPEARMAN, r.DISTANCIA]
        fig = go.Figure(go.Bar(x=valores, y=medidas, orientation='h',
                               marker_color=['#0056B3', '#007763', '#C34109'],
                               text=valores, texttemplate='%{text:.2f}', textposition='outside', cliponaxis=False,
                               customdata=[r.N] * len(medidas),
                               hovertemplate='<b>%{y}</b><br>Coeficiente: %{x:.3f}<br>UFs: %{customdata}<extra></extra>'))
        fig.update_xaxes(title='Valor da medida de associação', range=[-1.12, 1.12], zeroline=True, zerolinecolor='#6A7D92')
        fig.update_yaxes(title='Medida usada')
        titulo = 'Como as três medidas resumem essa relação?'
    mostrar_figura(configurar_painel(fig, titulo, f'{ano} · {area} · {atributo} · rede estadual'))
    print(f'Spearman retirando uma UF: {r.RHO_MIN:.2f} a {r.RHO_MAX:.2f} (faixa de sensibilidade).')
    print(f'p ajustado BY: Pearson={r.Q_PEARSON_BY:.3f}; distâncias={r.Q_DISTANCIA_BY:.3f}.')

painel(mostrar_formas_associacao, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'), atributo=escolher(ROTULOS, 'Renda', 'Atributo:'),
       vista=escolher(['Dispersão e ajuste', 'Coeficientes'], 'Dispersão e ajuste', 'Vista:'),
       forma=widgets.ToggleButtons(options=['Reta', 'Curva'], value='Reta', description='Ajuste:',
           style={'description_width': '100px'}))


0 de 300 testes tiveram p ajustado abaixo de 0,05. A leitura depende da hipótese de permutação dentro das regiões.


### A associação entre UFs reaparece quando o contexto muda?

Contrastamos duas perguntas usando a rede estadual. A primeira compara UFs em uma edição. Para renda, tempo integral e formação docente, recalculamos a correlação dos postos após retirar diferenças associadas às regiões e aos outros dois indicadores, por projeção linear dos postos. Usamos os mesmos pares completos na correlação bruta e na ajustada. Esse ajuste descritivo depende dos controles escolhidos e pode ser instável com 27 UFs.

A segunda pergunta compara **mudanças anuais**. Calculamos, dentro de cada UF, a variação da mediana e a de cada atributo. Em cada transição, Spearman compara essas variações entre UFs. Mostramos as quatro transições separadas para evitar interpretar a repetição de uma UF como independência. O painel usa a mesma escala de −1 a +1. Um sinal diferente entre níveis e mudanças orienta a investigação da série e da composição, sem identificar efeitos de políticas.

In [ ]:
tres_indicadores = ['RENDA_PC_REAIS_2024_IPCA', 'PCT_MATRICULAS_INTEGRAL_EM',
                    'PCT_DOCENTES_ADEQUADOS_EM']
nomes_curto = dict(zip(tres_indicadores, ['Renda', 'Tempo integral', 'Formação docente']))
base_mudanca = base_estadual.sort_values(['UF_ESCOLA', 'AREA', 'ANO'])
variaveis_mudanca = ['MEDIANA', *tres_indicadores]
variacoes = base_mudanca[['ANO', 'UF_ESCOLA', 'AREA']].join(
    base_mudanca.groupby(['UF_ESCOLA', 'AREA'])[variaveis_mudanca].diff())
linhas_ajuste, linhas_mudanca = [], []
for (ano, area), grupo in base_estadual.groupby(['ANO', 'AREA']):
    d = grupo.dropna(subset=['MEDIANA', *tres_indicadores, 'REGIAO'])
    postos = d[['MEDIANA', *tres_indicadores]].rank()
    regiao = pd.get_dummies(d.REGIAO, drop_first=True, dtype=float)
    for x in tres_indicadores:
        outros = [nome for nome in tres_indicadores if nome != x]
        controles = np.column_stack([np.ones(len(d)), regiao, postos[outros]])
        valores = postos[[x, 'MEDIANA']].to_numpy()
        estimativa, _, posto_matriz, _ = np.linalg.lstsq(controles, valores, rcond=None)
        residuos = valores - controles @ estimativa
        valido = (len(d) - posto_matriz > 2) and np.all(residuos.std(axis=0) > 1e-8)
        linhas_ajuste.append(dict(ANO=ano, AREA=area, ATRIBUTO=x, N_UFS=len(d),
                                  RHO=correlacao_segura(d, x),
                                  RHO_PARCIAL=np.corrcoef(residuos.T)[0, 1] if valido else np.nan))
for (ano, area), d in variacoes[variacoes.ANO.gt(2020)].groupby(['ANO', 'AREA']):
    for x in tres_indicadores:
        linhas_mudanca.append(dict(ANO=ano, AREA=area, ATRIBUTO=x, RHO=correlacao_segura(d, x),
                                   N_UFS=len(d[[x, 'MEDIANA']].dropna())))
ajustes = pd.DataFrame(linhas_ajuste)
mudancas_contexto = pd.DataFrame(linhas_mudanca)
salvar(ajustes, 'associacoes_parciais.csv')
salvar(mudancas_contexto, 'associacoes_variacoes_anuais.csv')


def mostrar_contexto(area, indicador, intervalo):
    x = next(c for c, rotulo in nomes_curto.items() if rotulo == indicador)
    nivel = ajustes.query('AREA == @area and ATRIBUTO == @x')
    mudanca = mudancas_contexto.query('AREA == @area and ATRIBUTO == @x')
    fig = go.Figure()
    for tabela, coluna, nome, cor, traco in [
        (nivel, 'RHO', 'Entre UFs', '#0056B3', 'solid'),
        (nivel, 'RHO_PARCIAL', 'Entre UFs, ajustada', '#007763', 'dash'),
        (mudanca, 'RHO', 'Variações na UF', '#C34109', 'dot'),
    ]:
        d = tabela[tabela.ANO.between(*intervalo)]
        fig.add_scatter(x=d.ANO, y=d[coluna], customdata=d.N_UFS, name=nome, mode='lines+markers',
                        line=dict(color=cor, dash=traco, width=2.5),
                        hovertemplate='Ano: %{x:.0f}<br>Correlação: %{y:+.2f}<br>UFs: %{customdata}')
    fig.add_hline(y=0, line_color='#6A7D92', line_dash='dot')
    fig.update_xaxes(title='Ano do ENEM (ano final na comparação de mudanças)', tickvals=list(range(intervalo[0], intervalo[1] + 1)))
    fig.update_yaxes(title='Correlação de Spearman (−1 a +1)', range=[-1.05, 1.05])
    mostrar_figura(configurar_painel(fig, f'A relação entre estados aparece nas mudanças anuais? · {indicador}',
                                   f'{area} · rede estadual'))

painel(mostrar_contexto, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
       indicador=escolher(list(nomes_curto.values()), 'Renda', 'Indicador:'), intervalo=periodo())


## Preparar a Etapa 3

O problema adotado é **prever a mediana dos elegíveis do ENEM da rede estadual, por UF escolar e área, no ano seguinte**. O alvo vem das notas completas desse recorte. A previsão pode orientar onde aprofundar o diagnóstico territorial. Na validação cronológica, todas as UFs de um mesmo ano-alvo entram juntas na avaliação e o treino usa anos-alvo anteriores. A persistência da mediana anterior será a referência. As áreas são tratadas separadamente. Há 135 linhas por área na série completa e 108 pares ano-origem → ano-alvo, com as mesmas UFs em cada transição.

Fixamos como hipótese operacional o **dia 30 de junho do ano-alvo**. Para cada série, a Etapa 3 deverá confirmar que a versão do indicador referente ao ano anterior estava publicada até essa data. A auditoria atual dispõe de anos de referência, sem as datas de publicação por edição de cada fonte. Assim, as transições são uma tabela de preparação com atributos candidatos e elegibilidade temporal pendente. Para renda no arquivo candidato, usamos a série em preços fixos de 2020, referência anterior a todos os alvos, e mantemos a escala de 2024 nos gráficos descritivos. A mediana anterior também precisa passar pela verificação de disponibilidade do resumo completo na data de corte.

Atributos derivados das notas do ano-alvo, perfis e contagens de controle ficam em tabelas separadas. Importância preditiva e eventual ganho além da persistência serão avaliados na Etapa 3.

**Junção entre edições.** À esquerda entram os alvos de 2021 a 2024 por UF escolar e área. À direita, deslocamos em um ano a chave dos atributos de 2020 a 2023 e conservamos a edição de origem em `ANO_REFERENCIA`. Assim, cada linha liga o alvo em `ANO_ALVO` aos atributos do ano anterior na mesma UF e área. O código exige uma correspondência **um para um** e verifica que a diferença entre os anos é sempre igual a um.


In [ ]:
atributos_previsao = ['RENDA_PC_REAIS_2020_IPCA', *CANDIDATOS[1:], 'HORAS_AULA_DIA_EM']
atributos_origem = base_estadual[chaves_resultado + ['MEDIANA', *atributos_previsao]].rename(
    columns={coluna: f'{coluna}_T_MENOS_1' for coluna in ['MEDIANA', *atributos_previsao]})
atributos_origem = atributos_origem.rename(columns={'ANO': 'ANO_REFERENCIA'})
atributos_origem['ANO'] = atributos_origem.ANO_REFERENCIA + 1
alvos = base_estadual.loc[base_estadual.ANO.gt(2020), chaves_resultado + ['MEDIANA']]
transicoes = alvos.rename(columns={'MEDIANA': 'ALVO_MEDIANA'}).merge(
    atributos_origem, on=chaves_resultado, validate='one_to_one', how='left', indicator=True)
assert transicoes._merge.eq('both').all() and transicoes.ANO.sub(transicoes.ANO_REFERENCIA).eq(1).all()
transicoes = transicoes.drop(columns='_merge').rename(columns={'ANO': 'ANO_ALVO'})
assert transicoes.groupby('AREA').size().eq(108).all()
transicoes['DATA_CORTE_PROPOSTA'] = transicoes.ANO_ALVO.astype(str) + '-06-30'
transicoes['STATUS_TEMPORAL'] = 'Pendente: validar publicação e versão por fonte e edição'
salvar(transicoes, 'candidatos_transicoes_etapa3.csv')
dicionario = pd.DataFrame([
    ('ALVO_MEDIANA', 'ENEM', 'Pontos, UF × ano-alvo × área', 'Alvo'),
    ('MEDIANA_T_MENOS_1', 'ENEM', 'Pontos, UF × ano anterior × área', 'Persistência e candidata'),
    ('RENDA_PC_REAIS_2020_IPCA_T_MENOS_1', 'PNAD e IPCA', 'R$ de 2020 por pessoa na UF', 'Candidata'),
    ('PCT_MATRICULAS_INTEGRAL_EM_T_MENOS_1', 'Censo Escolar', '% das matrículas estaduais do EM', 'Candidata'),
    ('PCT_DOCENTES_ADEQUADOS_EM_T_MENOS_1', 'Indicadores Inep', '% Grupo 1 no EM estadual', 'Candidata'),
    ('PCT_ABANDONO_EM_T_MENOS_1', 'Indicadores Inep', '% de abandono no EM estadual', 'Candidata'),
    ('PCT_IN_INTERNET_ALUNOS_T_MENOS_1', 'Censo Escolar', '% escolas estaduais com resposta', 'Candidata'),
    ('PCT_IN_LABORATORIO_CIENCIAS_T_MENOS_1', 'Censo Escolar', '% escolas estaduais com resposta', 'Candidata'),
    ('HORAS_AULA_DIA_EM_T_MENOS_1', 'Indicadores Inep', 'Horas por dia no EM estadual', 'Alternativa'),
    ('N, N_AMOSTRA, N_INFO_*, cobertura', 'ENEM e Censo Escolar', 'Denominadores dos respectivos recortes', 'Auditoria em tabelas separadas'),
    ('QUADRANTE_50, QUADRANTE_40_60', 'ENEM', 'Posições relativas da mesma inscrição', 'Descrição individual em tabela separada'),
], columns=['VARIAVEL', 'FONTE', 'UNIDADE', 'PAPEL'])
dicionario['DISPONIBILIDADE'] = 'Verificar data e versão de publicação por ano de referência'
dicionario.loc[~dicionario.VARIAVEL.str.endswith('_T_MENOS_1'), 'DISPONIBILIDADE'] = 'Conforme papel'
salvar(dicionario, 'dicionario_etapa3.csv')
display(dicionario)
display(transicoes.groupby(['ANO_ALVO', 'AREA']).size().unstack())

,VARIAVEL,FONTE,UNIDADE,PAPEL,DISPONIBILIDADE
0,ALVO_MEDIANA,ENEM,"Pontos, UF × ano-alvo × área",Alvo,Conforme papel
1,MEDIANA_T_MENOS_1,ENEM,"Pontos, UF × ano anterior × área",Persistência e candidata,Verificar data e versão de publicação por ano ...
2,RENDA_PC_REAIS_2020_IPCA_T_MENOS_1,PNAD e IPCA,R$ de 2020 por pessoa na UF,Candidata,Verificar data e versão de publicação por ano ...
3,PCT_MATRICULAS_INTEGRAL_EM_T_MENOS_1,Censo Escolar,% das matrículas estaduais do EM,Candidata,Verificar data e versão de publicação por ano ...
4,PCT_DOCENTES_ADEQUADOS_EM_T_MENOS_1,Indicadores Inep,% Grupo 1 no EM estadual,Candidata,Verificar data e versão de publicação por ano ...
5,PCT_ABANDONO_EM_T_MENOS_1,Indicadores Inep,% de abandono no EM estadual,Candidata,Verificar data e versão de publicação por ano ...
6,PCT_IN_INTERNET_ALUNOS_T_MENOS_1,Censo Escolar,% escolas estaduais com resposta,Candidata,Verificar data e versão de publicação por ano ...
7,PCT_IN_LABORATORIO_CIENCIAS_T_MENOS_1,Censo Escolar,% escolas estaduais com resposta,Candidata,Verificar data e versão de publicação por ano ...
8,HORAS_AULA_DIA_EM_T_MENOS_1,Indicadores Inep,Horas por dia no EM estadual,Alternativa,Verificar data e versão de publicação por ano ...
9,"N, N_AMOSTRA, N_INFO_*, cobertura",ENEM e Censo Escolar,Denominadores dos respectivos recortes,Auditoria em tabelas separadas,Conforme papel


AREA,Ciências Humanas,Ciências da Natureza,Linguagens,Matemática,Redação
ANO_ALVO,,,,,
2021,27,27,27,27,27
2022,27,27,27,27,27
2023,27,27,27,27,27
2024,27,27,27,27,27


In [ ]:
mt = 'Matemática'
cob_mt = cobertura[cobertura.AREA.eq(mt)].set_index('ANO').COBERTURA_UF_ESCOLA
renda_mt = ajustes.query('ANO == 2024 and AREA == @mt and ATRIBUTO == "RENDA_PC_REAIS_2024_IPCA"').iloc[0]
renda_hist = ajustes.query('AREA == @mt and ATRIBUTO == "RENDA_PC_REAIS_2024_IPCA"').RHO
renda_delta = mudancas_contexto.query('AREA == @mt and ATRIBUTO == "RENDA_PC_REAIS_2024_IPCA"').RHO
integral_hist = ajustes.query('AREA == @mt and ATRIBUTO == "PCT_MATRICULAS_INTEGRAL_EM"').RHO
formacao_hist = ajustes.query('AREA == @mt and ATRIBUTO == "PCT_DOCENTES_ADEQUADOS_EM"').RHO
referencia_mt = notas_situacao[notas_situacao.AREA.eq(mt)].copy()
referencia_mt['UF_CONHECIDA'] = referencia_mt.UF_ESCOLA.ne('Sem UF escolar')
diferenca_uf = referencia_mt.groupby(['ANO', 'UF_CONHECIDA'])[['N','SOMA']].sum()
medias_uf = (diferenca_uf.SOMA / diferenca_uf.N).unstack()
gap_uf = medias_uf[True] - medias_uf[False]
j = 'Concluinte declarado do ano'
insc_pct = 100 * brasil_inscritos[j] / brasil_inscritos.sum(axis=1)
display(Markdown(f'''
## O que a Etapa 2 sustenta?

**Quem conseguimos observar?** A cobertura da UF escolar entre as notas elegíveis de Matemática
foi de **{cob_mt.loc[2020]:.1f}% a {cob_mt.loc[2024]:.1f}%** entre 2020 e 2024. A amostra por área
reproduz o sorteio entre elegíveis com UF. A diferença na média entre quem tem e quem não tem
UF foi **{gap_uf.loc[2020]:+.1f}** pontos em 2020 e **{gap_uf.loc[2024]:+.1f}** em 2024. No universo nacional de inscrições, a parcela que
declarou conclusão no ano foi de {insc_pct.loc[2020]:.1f}% em 2020 e {insc_pct.loc[2024]:.1f}%
em 2024. O cruzamento individual entre situação e notas vai até 2023. Entre os elegíveis de
Matemática em 2023, concluintes declarados eram 100% dos registros com UF e 16,8% dos
registros sem UF. Todos os perfis com UF escolar de 2020–2023 também eram
de concluintes declarados, o que motivou a comparação nacional em universo ampliado.

**Quais padrões encontramos?** De 2023 a 2024, {int(delta_regional.lt(0).sum())} das 25
combinações região × área tiveram queda na mediana. Em Matemática, a média nos grupos
comparáveis mudou **{comp_mt["OBSERVADA"]:+.1f}** pontos; a parcela descritiva que varia com os
pesos de UF × rede × localização foi {comp_mt["PARCELA_COMPOSICAO"]:+.1f}. Na amostra conjunta
estadual de 2023, {perfil_50["Alto nos dois"]:.1f}% ficaram altos nos dois pares de áreas pelo
corte 50 e {perfil_50["Baixo nos dois"]:.1f}% ficaram baixos nos dois.

**O que acompanha essas diferenças?** Para Matemática na rede estadual, a renda por pessoa
teve Spearman transversal de {renda_hist.min():+.2f} a {renda_hist.max():+.2f} entre 2020 e
2024. Em 2024, foi {renda_mt.RHO:+.2f} entre {int(renda_mt.N_UFS)} UFs e
{renda_mt.RHO_PARCIAL:+.2f} após o ajuste descritivo. Nas transições anuais, a associação
entre mudanças na renda e mudanças nas notas variou de {renda_delta.min():+.2f} a
{renda_delta.max():+.2f}. Tempo integral variou de {integral_hist.min():+.2f} a
{integral_hist.max():+.2f} e formação docente, de {formacao_hist.min():+.2f} a
{formacao_hist.max():+.2f} nas comparações transversais. No conjunto de medidas de Pearson e distâncias, {n_abaixo} de {n_testes} testes ficaram abaixo
de 0,05 após o ajuste BY sob a referência de permutação regional.

**O que segue para a Etapa 3?** O alvo é a mediana completa da rede estadual por UF, ano e
área. A tabela de {len(transicoes)} transições carrega mediana anterior e contexto candidato
do ano anterior, com verificação de publicação pendente. A unidade, a data de corte
proposta e os denominadores estão documentados nas tabelas exportadas.

**O que permanece em aberto?** A mudança no perfil dos observados em 2024 limita a
comparação com edições anteriores. O vínculo territorial deixa em aberto o acesso
individual a recursos e os efeitos da renda familiar, da pandemia e de políticas.
A dimensão cultural permanece sem indicador definido. As correlações são evidências
descritivas; o ganho de previsão será avaliado em cortes cronológicos.
'''))


## O que a Etapa 2 sustenta?

**Quem conseguimos observar?** A cobertura da UF escolar entre as notas elegíveis de Matemática
foi de **20.8% a 39.9%** entre 2020 e 2024. A amostra por área
reproduz o sorteio entre elegíveis com UF. A diferença na média entre quem tem e quem não tem
UF foi **+25.1** pontos em 2020 e **-10.8** em 2024. No universo nacional de inscrições, a parcela que
declarou conclusão no ano foi de 24.1% em 2020 e 37.3%
em 2024. O cruzamento individual entre situação e notas vai até 2023. Entre os elegíveis de
Matemática em 2023, concluintes declarados eram 100% dos registros com UF e 16,8% dos
registros sem UF. Todos os perfis com UF escolar de 2020–2023 também eram
de concluintes declarados, o que motivou a comparação nacional em universo ampliado.

**Quais padrões encontramos?** De 2023 a 2024, 19 das 25
combinações região × área tiveram queda na mediana. Em Matemática, a média nos grupos
comparáveis mudou **-26.5** pontos; a parcela descritiva que varia com os
pesos de UF × rede × localização foi -13.9. Na amostra conjunta
estadual de 2023, 24.6% ficaram altos nos dois pares de áreas pelo
corte 50 e 54.6% ficaram baixos nos dois.

**O que acompanha essas diferenças?** Para Matemática na rede estadual, a renda por pessoa
teve Spearman transversal de +0.73 a +0.87 entre 2020 e
2024. Em 2024, foi +0.75 entre 27 UFs e
+0.41 após o ajuste descritivo. Nas transições anuais, a associação
entre mudanças na renda e mudanças nas notas variou de -0.27 a
+0.26. Tempo integral variou de -0.31 a
-0.16 e formação docente, de +0.11 a
+0.39 nas comparações transversais. No conjunto de medidas de Pearson e distâncias, 0 de 300 testes ficaram abaixo
de 0,05 após o ajuste BY sob a referência de permutação regional.

**O que segue para a Etapa 3?** O alvo é a mediana completa da rede estadual por UF, ano e
área. A tabela de 540 transições carrega mediana anterior e contexto candidato
do ano anterior, com verificação de publicação pendente. A unidade, a data de corte
proposta e os denominadores estão documentados nas tabelas exportadas.

**O que permanece em aberto?** A mudança no perfil dos observados em 2024 limita a
comparação com edições anteriores. O vínculo territorial deixa em aberto o acesso
individual a recursos e os efeitos da renda familiar, da pandemia e de políticas.
A dimensão cultural permanece sem indicador definido. As correlações são evidências
descritivas; o ganho de previsão será avaliado em cortes cronológicos.


### Prévia da tabela final de transições

O `head()` mostra as primeiras cinco linhas ordenadas por UF, área e ano-alvo. A prévia seleciona as chaves, o alvo e exemplos de atributos do ano anterior. A tabela completa é gravada em `candidatos_transicoes_etapa3.csv`. A coluna `STATUS_TEMPORAL` registra, em cada linha, a verificação ainda pendente da publicação das fontes na data de corte proposta.

In [ ]:
# @title
print(f'Tabela de transições · {len(transicoes):,} linhas × {transicoes.shape[1]} colunas')
display(transicoes.sort_values(['UF_ESCOLA', 'AREA', 'ANO_ALVO']).head()[
    ['UF_ESCOLA', 'AREA', 'ANO_REFERENCIA', 'ANO_ALVO', 'ALVO_MEDIANA',
     'MEDIANA_T_MENOS_1', 'RENDA_PC_REAIS_2020_IPCA_T_MENOS_1', 'DATA_CORTE_PROPOSTA']
].rename(columns={
    'ALVO_MEDIANA': 'Alvo (pontos)', 'MEDIANA_T_MENOS_1': 'Mediana anterior (pontos)',
    'RENDA_PC_REAIS_2020_IPCA_T_MENOS_1': 'Renda anterior (R$ de 2020)',
    'DATA_CORTE_PROPOSTA': 'Corte proposto'
}).round(2))


Tabela de transições · 540 linhas × 15 colunas


,UF_ESCOLA,AREA,ANO_REFERENCIA,ANO_ALVO,Alvo (pontos),Mediana anterior (pontos),Renda anterior (R$ de 2020),Corte proposto
27,AC,Ciências Humanas,2020,2021,480.3,470.5,879.00,2021-06-30
162,AC,Ciências Humanas,2021,2022,492.0,480.3,809.78,2022-06-30
297,AC,Ciências Humanas,2022,2023,490.0,492.0,860.99,2023-06-30
432,AC,Ciências Humanas,2023,2024,469.4,490.0,860.34,2024-06-30
54,AC,Ciências da Natureza,2020,2021,447.7,449.4,879.00,2021-06-30
